# MobiPredict — Flujo MLOps de extremo a extremo con datos reales

**Velocidad de los corredores de Medellín a 1 hora, alertas de colapso y demanda/densidad con la EOD 2025 (AMVA).**

```
Fuentes oficiales ─► Estandarización ─► EDA ─► Features ─► Entrenamiento + Optuna ─► MLflow ─► Selección ─► API ─► Docker
```

| Fuente | Uso | Enlace |
| --- | --- | --- |
| Velocidad y tiempo de viaje GT (SIMM, MEData) | Serie del modelo, 22 tramos, 2017-2020 | https://www.datos.gov.co/d/7t5n-3b3w |
| EOD 2025 (AMVA), API del visor oficial | Demanda horaria, reparto modal, matriz OD 67 macrozonas | https://eod2025.metropol.gov.co/ |
| Siniestralidad Medellín 2008-2025 (Mendeley, CC BY 4.0) | EDA de siniestralidad | https://doi.org/10.17632/r6g5dfnpgh.1 |
| Open-Meteo (ERA5) | Lluvia horaria | https://open-meteo.com/en/docs/historical-weather-api |
| Aforos vehiculares (MEData) | Composición vehicular | https://www.datos.gov.co/d/b9s9-jw7c |

El notebook contiene **todo el proyecto**:

1. **Parte A — Código y tablas de referencia.** Cada archivo se escribe con `%%writefile`.
2. **Parte B — Datos y pipeline.** Descarga de las fuentes oficiales (~175 MB la primera vez), flujo de Prefect y revisión de cada etapa.
3. **Parte C — Servicio.** API FastAPI, pruebas y despliegue.

Requiere internet en la primera ejecución. *Kernel → Restart & Run All*.

## 0. Entorno

In [ ]:
import os, sys, importlib.util
from pathlib import Path

aqui = Path.cwd()
if aqui.name == "notebooks" and (aqui.parent / "run_pipeline.py").exists():
    PROJECT_DIR = aqui.parent
elif (aqui / "run_pipeline.py").exists():
    PROJECT_DIR = aqui
else:
    PROJECT_DIR = aqui / "mobipredict"
PROJECT_DIR.mkdir(exist_ok=True)
os.chdir(PROJECT_DIR)
for d in ["src/mobipredict", "api", "tests/fixtures", "notebooks", "scripts", "data/raw", "data/interim",
          "data/processed", "data/reference/eod2025", "eda/figures", "eda/reports", "models", "docs", "mlruns"]:
    Path(d).mkdir(parents=True, exist_ok=True)
print("Proyecto en:", Path.cwd())

In [ ]:
paquetes = {"prefect": "prefect>=3.0", "mlflow": "mlflow>=2.16", "optuna": "optuna>=3.6", "lightgbm": "lightgbm>=4.3",
            "fastapi": "fastapi>=0.110", "httpx": "httpx>=0.27", "pyarrow": "pyarrow>=15", "sklearn": "scikit-learn>=1.5",
            "pytest": "pytest>=8", "holidays": "holidays>=0.50", "openpyxl": "openpyxl>=3.1"}
faltan = [spec for mod, spec in paquetes.items() if importlib.util.find_spec(mod) is None]
if faltan:
    print("Instalando:", faltan)
    %pip install -q {" ".join(faltan)}
else:
    print("Todas las dependencias están instaladas.")

---
# Parte A — Código y tablas de referencia

### `requirements.txt`

In [ ]:
%%writefile requirements.txt
# Núcleo de datos y modelado
numpy>=1.26
pandas>=2.2
pyarrow>=15
scipy>=1.12
scikit-learn>=1.5
lightgbm>=4.3
optuna>=3.6
joblib>=1.3
matplotlib>=3.8
# MLOps
mlflow>=2.16
prefect>=3.0
# API
fastapi>=0.110
uvicorn[standard]>=0.29
pydantic>=2.6
httpx>=0.27
# Fuentes
holidays>=0.50
openpyxl>=3.1

### `data/fuentes_datos.csv` — catálogo de fuentes con URLs, licencias y cobertura

In [ ]:
%%writefile data/fuentes_datos.csv
id,fuente,publicador,uso_en_proyecto,cobertura,formato,tamano,licencia,url_descarga,ficha,archivo_local
velocidad_gt,Velocidad y tiempo de viaje GT,Secretaría de Movilidad de Medellín (SIMM) vía MEData,Serie del modelo (velocidad horaria por tramo),"22 tramos, 6:00-20:00, 2017-02-01 a 2020-12-15, 682.502 filas",CSV,89.8 MB,Datos abiertos Alcaldía de Medellín,https://medata.app.medellin.gov.co/sites/default/files/distribution/1-023-25-000287/Velocidad_y_tiempo_de_viaje_GT.csv,https://www.datos.gov.co/d/7t5n-3b3w,data/raw/medata_velocidad_tiempo_viaje_gt.csv
eod2025,Encuesta Origen-Destino 2025 (API del visor oficial),Área Metropolitana del Valle de Aburrá,"Demanda por hora y modo, reparto modal, motorización, matriz OD de 67 macrozonas","Valle de Aburrá, día hábil 2025, 6.490.299 viajes",API JSON,~0.3 MB,Datos públicos AMVA,https://eod2025.metropol.gov.co/api/api/indicadores/1/agregado,https://eod2025.metropol.gov.co/,data/raw/eod2025_indicadores.json (copia en data/reference/eod2025/)
eod2025_macrozonas,Diccionario y centroides de macrozonas EOD 2025,Área Metropolitana del Valle de Aburrá,Relación id de macrozona -> municipio/nombre y ubicación,67 macrozonas,CSV (extraído del visor),4 KB,Datos públicos AMVA,https://eod2025.metropol.gov.co/macrozonas_actualizado.geo.json,https://eod2025.metropol.gov.co/,data/reference/eod2025_macrozonas.csv
siniestros,Road-traffic incidents and fatalities in Medellín 2008-2025,Correa Álvarez et al. (2026) - Mendeley Data (registros Secretaría de Movilidad),Siniestralidad georreferenciada (EDA),"702.540 incidentes, 2008-01-01 a 2025-08-31",ZIP con XLSX,63 MB,CC BY 4.0,https://data.mendeley.com/public-files/datasets/r6g5dfnpgh/files/e992c888-5819-40c6-9538-41980ea1d429/file_downloaded,https://doi.org/10.17632/r6g5dfnpgh.1,data/raw/mendeley_siniestros_medellin_2008_2025.zip
clima,Open-Meteo Historical Weather API (ERA5/ERA5-Land),Open-Meteo / Copernicus,Lluvia y temperatura horaria (features),"Medellín (6.2442, -75.5812), 2017-2020, horario",API JSON,~1 MB,CC BY 4.0,https://archive-api.open-meteo.com/v1/archive?latitude=6.2442&longitude=-75.5812&start_date=2017-01-01&end_date=2020-12-31&hourly=precipitation&timezone=America%2FBogota,https://open-meteo.com/en/docs/historical-weather-api,data/raw/open_meteo_medellin_horario.csv
aforos,Aforos vehiculares,Secretaría de Movilidad de Medellín vía MEData,Composición vehicular (EDA),"74.900 registros de 15 min, 2017-2019, hasta 113 intersecciones",CSV,19.7 MB,Datos abiertos Alcaldía de Medellín,https://medata.app.medellin.gov.co/sites/default/files/distribution/1-023-25-000301/Aforos_Vehiculares.csv,https://www.datos.gov.co/d/b9s9-jw7c,data/raw/medata_aforos_vehiculares.csv
corredores_simm,Ubicación de cámaras SIMM por corredor,Secretaría de Movilidad de Medellín vía MEData (Conteo Vehicular Simmtrafficdata),Coordenadas de los tramos,"49 corredores, enero 2021",CSV (derivado),2 KB,Datos abiertos Alcaldía de Medellín,https://medata.app.medellin.gov.co/sites/default/files/distribution/1-023-25-000300/simmtrafficdata.csv,https://www.datos.gov.co/d/6ymu-tzvs,data/reference/corredores_simm.csv
festivos,Festivos de Colombia,Paquete holidays (Ley 51 de 1983),Tipo de día (features),2017-2026,Calculado,—,MIT,https://pypi.org/project/holidays/,https://pypi.org/project/holidays/,data/interim/festivos_colombia.csv
referencia_muertes_2025,Balance de siniestralidad 2025,Observatorio de Movilidad de Medellín (vía El Colombiano),Cifra de referencia: 276 muertes (162 motociclistas; 100 peatones),2025,Nota de prensa,—,—,https://www.elcolombiano.com/medellin/balance-de-accidentalidad-en-medellin-el-ano-pasado-PF32374139,https://www.elcolombiano.com/medellin/balance-de-accidentalidad-en-medellin-el-ano-pasado-PF32374139,—
referencia_eod_2017_2025,Resultados EOD 2025 frente a 2017,AMVA (vía El Colombiano),Cifras de referencia: +48 % tenencia de motos; -14 % participación TP; +5.9 % viajes,2017-2025,Nota de prensa,—,—,https://www.elcolombiano.com/medellin/movilidad-habitantes-medellin-aburra-disminucion-teletrabajo-DD36063145,https://www.elcolombiano.com/medellin/movilidad-habitantes-medellin-aburra-disminucion-teletrabajo-DD36063145,—

### Tablas de referencia oficiales
`eod2025_macrozonas.csv`: diccionario id → municipio/macrozona de la EOD 2025 y centroides (visor oficial del AMVA).

In [ ]:
%%writefile data/reference/eod2025_macrozonas.csv
id_macrozona,municipio,macrozona,tipo,lat,lon
1,Itagüí,Comuna 06,Urbano,6.16956,-75.63346
2,Itagüí,Comuna 05,Urbano,6.17663,-75.61493
3,Itagüí,Comuna 04,Urbano,6.18693,-75.59373
4,Itagüí,Comuna 03,Urbano,6.18073,-75.60699
5,Bello,Comuna 05,Urbano,6.34126,-75.56992
6,Bello,Comuna 06,Urbano,6.35300,-75.56693
7,Itagüí,Comuna 02,Urbano,6.16217,-75.62282
8,Bello,Comuna 03,Urbano,6.32646,-75.56331
9,Bello,Comuna 07,Urbano,6.34737,-75.55309
10,Bello,Comuna 04,Urbano,6.33590,-75.55838
11,Medellín,Robledo,Urbano,6.27886,-75.59252
12,Medellín,El Poblado,Urbano,6.20169,-75.56458
13,Medellín,Manrique,Urbano,6.27308,-75.54477
14,Medellín,Guayabal,Urbano,6.21315,-75.58639
15,Medellín,Buenos Aires,Urbano,6.23149,-75.55290
16,Medellín,Belén,Urbano,6.22508,-75.59982
17,Medellín,Aranjuez,Urbano,6.27620,-75.56157
18,Medellín,La América,Urbano,6.25420,-75.60612
19,Medellín,La Candelaria,Urbano,6.24772,-75.56808
20,Medellín,Villa Hermosa,Urbano,6.24754,-75.54412
21,Medellín,Laureles-Estadio,Urbano,6.25055,-75.59183
22,Medellín,San Javier,Urbano,6.25931,-75.61557
23,Itagüí,Comuna 01,Urbano,6.17002,-75.60852
24,Caldas,Urbana Caldas,Urbano,6.07658,-75.63321
25,La Estrella,Urbana La Estrella,Urbano,6.14300,-75.63781
26,Barbosa,Urbana Barbosa,Urbano,6.43690,-75.33290
27,Girardota,Urbana Girardota,Urbano,6.38089,-75.44330
28,Bello,Comuna 02,Urbano,6.31737,-75.56136
29,Bello,Comuna 10,Urbano,6.32892,-75.54260
30,Medellín,Castilla,Urbano,6.29231,-75.56868
31,Medellín,Popular,Urbano,6.29364,-75.54420
32,Medellín,Doce de Octubre,Urbano,6.29932,-75.57920
33,Medellín,Santa Cruz,Urbano,6.29842,-75.55453
34,Bello,Comuna 11,Urbano,6.31478,-75.55032
35,Copacabana,Urbana Copacabana,Urbano,6.34666,-75.50869
36,Bello,Comuna 08,Urbano,6.34218,-75.54239
37,Bello,Comuna 09,Urbano,6.34457,-75.52599
38,Bello,Comuna 01,Urbano,6.31515,-75.58052
39,Bello,Comuna 12,Urbano,6.30040,-75.53030
40,Bello,Rural Bello,Rural,6.31460,-75.53806
41,Barbosa,Rural Barbosa,Rural,6.44324,-75.32944
42,Copacabana,Rural Copacabana,Rural,6.35223,-75.50019
43,Girardota,Rural Girardota,Rural,6.38078,-75.44836
44,Medellín,Corregimiento San Sebastián de Palmitas,Rural,6.32759,-75.68934
45,Medellín,Corregimiento San Cristóbal,Rural,6.29073,-75.64056
46,Medellín,Corregimiento Altavista,Rural,6.22731,-75.63531
47,Medellín,Corregimiento Santa Elena,Rural,6.23482,-75.51665
48,Medellín,Corregimiento San Antonio de Prado,Rural,6.21609,-75.67336
49,Itagüí,Rural Itagüí,Rural,6.18766,-75.61716
50,La Estrella,Rural La Estrella,Rural,6.12075,-75.62244
51,Envigado,Zona 01,Urbano,6.17492,-75.59422
52,Envigado,Zona 08,Urbano,6.16691,-75.59567
53,Envigado,Zona 07,Urbano,6.15937,-75.59320
54,Envigado,Zona 06,Urbano,6.15383,-75.58208
55,Envigado,Zona 05,Urbano,6.16114,-75.57290
56,Envigado,Zona 03,Urbano,6.17430,-75.57799
57,Envigado,Zona 09,Urbano,6.17008,-75.58423
58,Envigado,Zona 02,Urbano,6.17894,-75.58531
59,Envigado,Zona 04,Urbano,6.17409,-75.56835
60,Envigado,Rural 03,Rural,6.14641,-75.53229
61,Envigado,Rural 01,Rural,6.14137,-75.55718
62,Envigado,Rural 02,Rural,6.12489,-75.57265
63,Envigado,Rural 04,Rural,6.17196,-75.51541
64,Sabaneta,Urbana Sabaneta,Urbano,6.15197,-75.61651
65,Sabaneta,Rural Sabaneta,Rural,6.13442,-75.60763
66,Caldas,Rural Caldas,Rural,6.05397,-75.62707
67,Sin especificar,Sin especificar,Ninguno,,

`corredores_simm.csv`: ubicación media de las cámaras del SIMM por corredor (MEData).

In [ ]:
%%writefile data/reference/corredores_simm.csv
corredor,n_camaras,lat,lon,lat_min,lat_max
Avenida Regional,8,6.22661,-75.57752,6.1888,6.2599
Autopista Sur,6,6.22761,-75.57797,6.2010,6.2587
Autopista Norte,8,6.28897,-75.56805,6.2627,6.3095
Avenida 80,7,6.23750,-75.59731,6.2007,6.2733
Avenida Las Palmas,4,6.22337,-75.56249,6.2153,6.2330
Avenida El Poblado,6,6.21256,-75.57140,6.1927,6.2312
Avenida Guayabal,4,6.21201,-75.58376,6.2020,6.2217
Avenida 33,5,6.23885,-75.59005,6.2381,6.2397
Calle 30,2,6.23133,-75.59243,6.2309,6.2318
Avenida San Juan,4,6.24919,-75.58931,6.2470,6.2520
Avenida Ferrocarril,2,6.25168,-75.57504,6.2498,6.2536
Avenida Oriental,3,6.25245,-75.56321,6.2491,6.2548
Avenida Las Vegas,5,6.21220,-75.57667,6.1907,6.2255
Avenida Bolivariana,2,6.24626,-75.58292,6.2454,6.2471
Carrera 65,4,6.24544,-75.58275,6.2318,6.2561
Calle 10,4,6.21147,-75.57269,6.2084,6.2136
Calle 80,3,6.28394,-75.57583,6.2763,6.2913
Carrera 70,2,6.23594,-75.59038,6.2227,6.2492

`eod2025/indicadores_api.json`: los 42 indicadores agregados oficiales de la EOD 2025 (respaldo si la API no responde).

In [ ]:
%%writefile data/reference/eod2025/indicadores_api.json
{"fuente":"https://eod2025.metropol.gov.co/api/api/indicadores/{id}/agregado","descargado":"2026-10-23","nota":"Indicadores agregados oficiales de la EOD 2025 (AMVA), sin filtros. El indicador 15 (matriz OD) está en matriz_od_macrozonas.csv.","indicadores":{"1":{"id_indicador":1,"nombre":"Total de viajes en un día hábil","valor":6490298.966998089},"2":{"id_indicador":2,"nombre":"% de personas que no viajan","valor":0.3158524088010615},"3":{"id_indicador":3,"nombre":"Tiempo promedio de viaje","valor":39.27870070992075},"4":{"id_indicador":4,"nombre":"Viajes diarios en modos privados","valor":1879453.9713011228},"5":{"id_indicador":5,"nombre":"Viajes diarios en modos no motorizados","valor":2179414.7241534367},"6":{"id_indicador":6,"nombre":"Viajes diarios en modos públicos","valor":2196960.64438758},"7":{"id_indicador":7,"nombre":"Viajes diarios promedio por persona","valor":1.6002034842581587},"8":{"id_indicador":8,"nombre":"Viajes diarios promedio por personas que realizan viajes","valor":2.338974082264724},"9":{"id_indicador":9,"nombre":"Cantidad total de vehículos propios (toda tipología)","valor":815414.1689828252},"10":{"id_indicador":10,"nombre":"Cantidad promedio de vehículos por hogar","valor":0.49524241921998535},"11":{"id_indicador":11,"nombre":"% de vehículos que operan con tecnologías limpias","valor":0.050379833517386015},"12":{"id_indicador":12,"nombre":"Autos por 1000 habitantes","valor":67.13468059512508},"13":{"id_indicador":13,"nombre":"Motocicletas por 1000 habitantes","valor":120.49587941972563},"14":{"id_indicador":14,"nombre":"Bicicletas por 1000 habitantes","valor":10.459201907116244},"16":{"id_indicador":16,"nombre":"Distribución de viajes INFORMALES en un día (según hora de inicio)","grupos":[{"criterio":"0","valor":578.035690235962},{"criterio":"1","valor":1037.3069253288738},{"criterio":"2","valor":1760.7710750105157},{"criterio":"3","valor":976.1150325750498},{"criterio":"4","valor":1995.4369813710855},{"criterio":"5","valor":6721.494300771577},{"criterio":"6","valor":16545.99832581314},{"criterio":"7","valor":19531.684336329847},{"criterio":"8","valor":14244.13998762791},{"criterio":"9","valor":17450.073429551536},{"criterio":"10","valor":20024.893247835684},{"criterio":"11","valor":14477.753188719704},{"criterio":"12","valor":12338.865530631272},{"criterio":"13","valor":11715.246614681224},{"criterio":"14","valor":14095.984049977324},{"criterio":"15","valor":13932.957149727663},{"criterio":"16","valor":15673.828298456896},{"criterio":"17","valor":17909.10224323727},{"criterio":"18","valor":10813.525442054895},{"criterio":"19","valor":8178.340326988511},{"criterio":"20","valor":3953.4254517691666},{"criterio":"21","valor":4581.188330494268},{"criterio":"22","valor":4130.108894613837},{"criterio":"23","valor":1803.352302146688}]},"17":{"id_indicador":17,"nombre":"Distribución de viajes PRIVADOS en un día (según hora de inicio)","grupos":[{"criterio":"0","valor":1774.7469483317132},{"criterio":"1","valor":3685.0561991777913},{"criterio":"2","valor":6873.077611480348},{"criterio":"3","valor":7441.9389507124115},{"criterio":"4","valor":26722.660755887347},{"criterio":"5","valor":114379.55223617573},{"criterio":"6","valor":230844.39939990384},{"criterio":"7","valor":200941.4500576473},{"criterio":"8","valor":100780.1781694766},{"criterio":"9","valor":65838.90440871674},{"criterio":"10","valor":72141.59133239134},{"criterio":"11","valor":69099.06830546347},{"criterio":"12","valor":123540.3994122522},{"criterio":"13","valor":85685.78678599474},{"criterio":"14","valor":98802.99682666906},{"criterio":"15","valor":92947.28962411202},{"criterio":"16","valor":130123.38587103249},{"criterio":"17","valor":181713.69287311845},{"criterio":"18","valor":97205.39055286715},{"criterio":"19","valor":54752.863195717335},{"criterio":"20","valor":50848.44218584645},{"criterio":"21","valor":27629.96683715258},{"criterio":"22","valor":22792.78267543439},{"criterio":"23","valor":12888.35008556135}]},"18":{"id_indicador":18,"nombre":"Distribución de viajes PÚBLICOS en un día (según hora de inicio)","grupos":[{"criterio":"0","valor":2927.319656217451},{"criterio":"1","valor":1346.81844639251},{"criterio":"2","valor":3112.8481965453184},{"criterio":"3","valor":5859.297134218854},{"criterio":"4","valor":64219.92343283563},{"criterio":"5","valor":148582.4473178992},{"criterio":"6","valor":239033.82777169722},{"criterio":"7","valor":175366.5762563718},{"criterio":"8","valor":118391.92307937666},{"criterio":"9","valor":98624.04557403381},{"criterio":"10","valor":113663.1310523966},{"criterio":"11","valor":100226.735791438},{"criterio":"12","valor":119081.95222054442},{"criterio":"13","valor":105321.89261197843},{"criterio":"14","valor":131900.0151001582},{"criterio":"15","valor":116221.47584944085},{"criterio":"16","valor":157919.4309809503},{"criterio":"17","valor":212837.43173775356},{"criterio":"18","valor":130452.63896955352},{"criterio":"19","valor":60774.29456102083},{"criterio":"20","valor":41997.68242871865},{"criterio":"21","valor":27154.7899345171},{"criterio":"22","valor":19844.25739059742},{"criterio":"23","valor":2099.888892923741}]},"19":{"id_indicador":19,"nombre":"Distribución de viajes NO MOTORIZADOS en un día (según hora de inicio)","grupos":[{"criterio":"0","valor":1777.9142353626562},{"criterio":"1","valor":1530.3299639274028},{"criterio":"2","valor":1883.043385729484},{"criterio":"3","valor":2343.192553185073},{"criterio":"4","valor":9844.874540203098},{"criterio":"5","valor":74269.58458655693},{"criterio":"6","valor":230544.48418767488},{"criterio":"7","valor":123770.6747408812},{"criterio":"8","valor":113921.654827102},{"criterio":"9","valor":110015.44009968324},{"criterio":"10","valor":135480.95746848156},{"criterio":"11","valor":151332.7111681349},{"criterio":"12","valor":341052.36205545},{"criterio":"13","valor":110942.50639313029},{"criterio":"14","valor":102425.07517371999},{"criterio":"15","valor":95025.6115326948},{"criterio":"16","valor":107067.2429872778},{"criterio":"17","valor":225682.97609597948},{"criterio":"18","valor":111277.10558261658},{"criterio":"19","valor":58161.94718220544},{"criterio":"20","valor":31118.225403923494},{"criterio":"21","valor":17632.495055980427},{"criterio":"22","valor":16027.558798468448},{"criterio":"23","valor":6286.756135067512}]},"20":{"id_indicador":20,"nombre":"Distribución de viajes según su duración (% de viajes)","grupos":[{"criterio":"0-15","valor":0.27369425181075224},{"criterio":"16-30","valor":0.2877777135949483},{"criterio":"31-45","valor":0.1351136554047951},{"criterio":"46-60","valor":0.1479037112353575},{"criterio":"61-90","valor":0.10448587545318512},{"criterio":"91-120","valor":0.031525046529495344},{"criterio":"120 o más","valor":0.019499745971466448}]},"21":{"id_indicador":21,"nombre":"Frecuencia del viaje (% de viajes)","grupos":[{"criterio":"Todos los días","valor":0.0919942337904701},{"criterio":"Entre 2 y 6 días a la semana","valor":0.6692473948774752},{"criterio":"Solo un día a la semana","valor":0.052094549656699},{"criterio":"Es un viaje eventual","valor":0.18666382167535578}]},"22":{"id_indicador":22,"nombre":"Viajes diarios según estrato (% de viajes)","grupos":[{"criterio":1,"valor":0.13375582154303625},{"criterio":2,"valor":0.39894913393942105},{"criterio":3,"valor":0.2944229047661828},{"criterio":4,"valor":0.05870129079164888},{"criterio":5,"valor":0.0701921093024816},{"criterio":6,"valor":0.04397873965722944}]},"23":{"id_indicador":23,"nombre":"Tiempo promedio de viaje por tipo de transporte PÚBLICO (min)","valor":56.57551106011754},"24":{"id_indicador":24,"nombre":"Tiempo promedio de viaje por tipo de transporte PRIVADO (min)","valor":39.02584885434897},"25":{"id_indicador":25,"nombre":"Tiempo promedio de viaje por tipo de transporte INFORMAL (min)","valor":38.87273036775368},"26":{"id_indicador":26,"nombre":"Tiempo promedio de viaje por tipo de transporte NO MOTORIZADO (min)","valor":22.104364839317785},"27":{"id_indicador":27,"nombre":"Modo Principal (% de viajes)","grupos":[{"criterio":"A pie","valor":0.32675755626936775},{"criterio":"Masivo","valor":0.16435523598385357},{"criterio":"TPC","valor":0.14943573067562463},{"criterio":"Motocicleta","valor":0.14574688048404896},{"criterio":"Automóvil","valor":0.11702439103460727},{"criterio":"Taxi","valor":0.024708177302718126},{"criterio":"Bicicleta","valor":0.008647864943511776},{"criterio":"Otros","valor":0.06332416330626801}]},"28":{"id_indicador":28,"nombre":"Motivo de viaje (% de viajes)","grupos":[{"criterio":"Trabajo","valor":0.21975511743963635},{"criterio":"Estudio","valor":0.10228977852701061},{"criterio":"Regreso al hogar","valor":0.47161744726084065},{"criterio":"Ocio","valor":0.07094278754166919},{"criterio":"Otro","valor":0.13539486923084318}]},"29":{"id_indicador":29,"nombre":"Etapas (% de viajes)","grupos":[{"criterio":"Una etapa","valor":0.6966071124209036},{"criterio":"Dos etapas","valor":0.18053684784672142},{"criterio":"Tres etapas","valor":0.087906520819147},{"criterio":"4 o más etapas","valor":0.034949518913227985}]},"31":{"id_indicador":31,"nombre":"% de personas que sí viajan en grupos poblacionales de interés - Cuidador","valor":0.5706453559286481},"32":{"id_indicador":32,"nombre":"% de personas que sí viajan en grupos poblacionales de interés - Extranjero","valor":0.6314159657398445},"33":{"id_indicador":33,"nombre":"% de personas que sí viajan en grupos poblacionales de interés - Madre cabeza de familia","valor":0.6829112865784962},"34":{"id_indicador":34,"nombre":"% de personas que sí viajan en grupos poblacionales de interés - Discapacidad","valor":0.32545288873220246},"35":{"id_indicador":35,"nombre":"% de personas que sí viajan en grupos poblacionales de interés - Ninguna","valor":0.7006578194399208},"36":{"id_indicador":36,"nombre":"Tipología (% de vehículos)","grupos":[{"criterio":"Automóvil","valor":0.3127851971700597},{"criterio":"Camioneta","valor":0.028970485476608177},{"criterio":"Motocicleta","valor":0.5993540340316188},{"criterio":"Bicicleta","valor":0.05202472388242598},{"criterio":"Otro","valor":0.006865559439287213}]},"37":{"id_indicador":37,"nombre":"Cantidad (% de vehículos)","grupos":[{"criterio":"Sin vehículo","valor":0.6011120077858062},{"criterio":"Un vehículo","valor":0.31969357018123445},{"criterio":"Dos o más vehículos","valor":0.07919442203295927}]},"38":{"id_indicador":38,"nombre":"Modelo (% de vehículos)","grupos":[{"criterio":"Menor de 5 años","valor":0.22053413875134698},{"criterio":"5 a 10 años","valor":0.1898198438816594},{"criterio":"11 a 15 años","valor":0.07429671041796755},{"criterio":"16 años o más","valor":0.05255557632945929},{"criterio":"NS/NR","valor":0.4627937306195667}]},"39":{"id_indicador":39,"nombre":"Vehículos por estrato (% de vehículos)","grupos":[{"criterio":1,"valor":0.08874127711187801},{"criterio":2,"valor":0.37330219048845614},{"criterio":3,"valor":0.32368121907369807},{"criterio":4,"valor":0.06846601196353129},{"criterio":5,"valor":0.0932951009868513},{"criterio":6,"valor":0.05251420037558509}]},"42":{"id_indicador":42,"nombre":"Población por Edad","grupos":[{"criterio":"5-11","valor":0.07118342262863528},{"criterio":"12-17","valor":0.07486260343594443},{"criterio":"18-25","valor":0.11779392794026415},{"criterio":"26-35","valor":0.16889801302368077},{"criterio":"36-45","valor":0.14579764969219253},{"criterio":"46-60","valor":0.18691338885479333},{"criterio":"60+","valor":0.2345509944244896}]},"43":{"id_indicador":43,"nombre":"Población por Género","grupos":[{"criterio":"Hombre","valor":0.46984089434937076},{"criterio":"Mujer","valor":0.5301591056506292}]},"44":{"id_indicador":44,"nombre":"Población por Ocupación","grupos":[{"criterio":"Trabajador","valor":0.44155615510265417},{"criterio":"Estudiante","valor":0.18172579504246125},{"criterio":"Ama de casa","valor":0.1705696324707044},{"criterio":"Desempleado","valor":0.04912213634347508},{"criterio":"Jubilado","valor":0.09648532921265045},{"criterio":"Ninguna","valor":0.06054095182805472}]},"45":{"id_indicador":45,"nombre":"Población por Escolaridad","grupos":[{"criterio":"Primaria","valor":0.23868897887066823},{"criterio":"Noveno Grado","valor":0.04942328782743882},{"criterio":"Bachillerato","valor":0.3562752173141694},{"criterio":"Educación No formal","valor":0.0010696129792500764},{"criterio":"Técnico","valor":0.07575860257220127},{"criterio":"Tecnológico","valor":0.04864718293337842},{"criterio":"Universitario","valor":0.11521566629840327},{"criterio":"Posgrado","valor":0.015011990105457989},{"criterio":"Ninguno","valor":0.09990946109903269}]},"46":{"id_indicador":46,"nombre":"Población por Estrato","grupos":[{"criterio":1,"valor":0.1353604762072917},{"criterio":2,"valor":0.4265571183434062},{"criterio":3,"valor":0.29406643835055696},{"criterio":4,"valor":0.051110899706894655},{"criterio":5,"valor":0.06445103329089812},{"criterio":6,"valor":0.028454034100952468}]}},"viajes_por_municipio_residencia":{"Barbosa":92256.06835339018,"Bello":767915.6242265768,"Caldas":131842.6455997969,"Copacabana":100477.08446501521,"Envigado":398478.82300539943,"Girardota":71761.5205619482,"Itagüí":492718.1637687397,"La Estrella":99332.46737139043,"Medellín":4183261.457004232,"Sabaneta":152255.11264160048}}

`eod2025/matriz_od_macrozonas.csv`: matriz origen-destino oficial, 3.126 pares, 6.490.299 viajes.

In [ ]:
%%writefile data/reference/eod2025/matriz_od_macrozonas.csv
origen,destino,viajes
1,1,8180.2
1,2,5581.4
1,3,1800.1
1,4,706.1
1,7,1006.6
1,11,218.3
1,12,1219.0
1,14,496.9
1,17,103.1
1,18,112.0
1,19,1978.6
1,21,328.6
1,22,232.3
1,23,6325.2
1,24,99.6
1,25,336.3
1,30,80.7
1,47,14.7
1,48,1.0
1,49,472.8
1,50,11.6
1,51,85.8
1,52,157.1
1,54,114.6
1,57,450.6
1,58,131.2
1,60,77.4
1,61,124.2
1,62,3.0
1,64,685.6
1,65,4.3
1,67,457.9
2,1,5613.9
2,2,11659.4
2,3,3270.8
2,4,519.6
2,7,615.0
2,8,31.9
2,10,158.6
2,11,406.4
2,12,2043.4
2,13,110.6
2,14,728.0
2,15,35.0
2,16,527.6
2,17,243.4
2,18,35.1
2,19,2660.1
2,21,408.2
2,22,67.4
2,23,7142.0
2,24,133.4
2,25,504.6
2,27,31.1
2,30,95.3
2,32,140.1
2,36,58.1
2,41,205.2
2,43,20.1
2,46,37.9
2,48,111.6
2,49,505.8
2,50,11.3
2,51,302.5
2,52,126.7
2,53,64.7
2,54,110.5
2,55,54.7
2,56,109.7
2,57,856.7
2,58,380.5
2,61,35.5
2,64,724.6
2,65,11.4
2,67,66.2
3,1,1696.6
3,2,3201.4
3,3,62073.4
3,4,1439.5
3,6,200.0
3,7,3276.3
3,10,353.4
3,11,1173.4
3,12,4406.7
3,13,1010.7
3,14,2844.4
3,15,819.2
3,16,1994.3
3,17,1107.4
3,18,221.9
3,19,6990.2
3,20,572.8
3,21,865.1
3,22,586.6
3,23,7928.9
3,24,796.9
3,25,1126.5
3,27,96.9
3,30,290.5
3,32,646.1
3,33,369.8
3,36,259.3
3,37,6.5
3,40,32.9
3,42,349.0
3,43,55.3
3,44,3.3
3,45,514.0
3,46,2.0
3,47,14.9
3,48,1828.8
3,49,4714.2
3,50,137.3
3,53,820.1
3,54,427.2
3,55,914.1
3,56,439.8
3,57,876.3
3,58,558.3
3,59,7.9
3,60,2.9
3,61,2.5
3,62,13.6
3,63,5.3
3,64,1247.8
3,65,86.4
3,66,130.1
3,67,575.7
4,1,601.0
4,2,519.6
4,3,1439.5
4,4,19498.6
4,6,93.4
4,7,1975.6
4,10,154.4
4,11,388.3
4,12,2974.3
4,14,703.8
4,15,288.5
4,18,45.8
4,19,5567.5
4,20,204.2
4,21,923.7
4,23,11325.7
4,24,1239.0
4,25,2520.6
4,27,103.1
4,30,1138.4
4,31,2095.3
4,32,1037.0
4,37,3.1
4,41,175.4
4,42,47.4
4,46,48.6
4,47,1.0
4,48,1492.6
4,49,1766.3
4,50,58.6
4,51,583.4
4,52,366.3
4,55,271.4
4,56,80.2
4,57,59.0
4,58,58.1
4,59,5.6
4,60,3.3
4,62,11.9
4,63,1.4
4,64,1585.2
4,65,23.5
4,66,175.6
5,5,25885.4
5,6,2755.6
5,8,1891.0
5,10,15125.9
5,11,654.2
5,12,1233.6
5,13,76.3
5,14,226.3
5,15,165.6
5,16,370.7
5,17,194.2
5,19,6056.6
5,20,421.1
5,21,689.2
5,23,529.8
5,24,67.7
5,25,85.0
5,26,224.6
5,27,106.3
5,28,1075.4
5,30,345.6
5,31,233.4
5,34,78.8
5,35,194.5
5,36,1891.3
5,37,313.4
5,38,720.2
5,39,97.3
5,40,1057.2
5,41,81.1
5,42,593.3
5,43,26.9
5,45,65.6
5,50,15.6
5,57,307.4
5,61,170.1
5,64,514.9
5,67,197.9
6,3,200.0
6,4,93.4
6,5,2776.3
6,6,22504.4
6,8,726.1
6,9,87.2
6,10,11446.1
6,11,759.2
6,12,1761.3
6,14,124.5
6,15,180.8
6,16,338.4
6,18,130.1
6,19,4365.7
6,20,100.3
6,21,1399.4
6,23,693.2
6,25,302.0
6,26,114.4
6,27,225.7
6,28,779.1
6,30,485.5
6,32,286.1
6,33,110.7
6,34,169.6
6,35,233.4
6,36,1711.9
6,37,2.2
6,38,216.1
6,40,206.2
6,42,322.7
6,45,182.1
6,52,95.4
6,57,1571.1
6,64,301.7
6,67,1550.2
7,1,912.9
7,2,580.9
7,3,3351.3
7,4,1975.6
7,7,13720.4
7,8,62.5
7,11,1481.5
7,12,2068.4
7,14,1890.2
7,15,425.4
7,16,2249.7
7,17,522.1
7,18,142.9
7,19,3232.6
7,20,75.0
7,21,860.9
7,22,759.4
7,23,9850.0
7,24,1582.5
7,25,2798.1
7,27,67.9
7,30,770.2
7,33,154.6
7,36,276.5
7,42,73.9
7,44,1.0
7,45,75.5
7,47,26.3
7,48,549.9
7,49,441.5
7,50,70.0
7,51,201.8
7,52,231.0
7,53,68.8
7,54,122.4
7,55,151.5
7,56,114.7
7,57,977.2
7,58,182.7
7,59,6.5
7,62,156.6
7,63,80.6
7,64,3117.4
7,65,127.1
7,66,90.6
7,67,338.6
8,2,31.9
8,3,35.6
8,4,45.8
8,5,1598.1
8,6,671.8
8,7,62.5
8,8,5987.8
8,9,237.5
8,10,9346.1
8,11,264.7
8,12,922.2
8,13,628.3
8,14,431.6
8,15,67.4
8,16,1422.5
8,17,226.0
8,18,52.5
8,19,3243.7
8,21,532.6
8,22,1051.3
8,23,49.9
8,24,249.9
8,25,153.0
8,26,236.8
8,27,470.6
8,28,1978.0
8,29,523.2
8,30,1040.2
8,31,527.3
8,32,2313.9
8,33,1004.0
8,34,598.9
8,35,628.6
8,36,1843.7
8,37,388.2
8,38,907.5
8,39,79.1
8,40,136.7
8,41,51.5
8,42,194.0
8,43,91.5
8,44,5.0
8,45,42.2
8,48,57.9
8,49,32.7
8,51,126.1
8,52,149.6
8,55,84.0
8,56,139.0
8,57,102.4
8,62,3.6
8,64,92.5
8,66,20.6
8,67,321.8
9,6,87.2
9,8,237.5
9,9,1952.4
9,10,3715.6
9,11,933.6
9,12,1194.8
9,15,343.2
9,16,23308.4
9,17,246.0
9,19,2010.2
9,21,414.7
9,23,171.2
9,25,339.6
9,30,120.7
9,33,262.1
9,36,4789.8
9,67,935.3
10,2,123.3
10,3,353.4
10,4,154.4
10,5,14196.9
10,6,11709.3
10,8,9605.9
10,9,3606.1
10,10,39952.9
10,11,1728.5
10,12,1944.7
10,13,400.0
10,15,438.2
10,16,1574.3
10,17,1115.7
10,18,654.3
10,19,4940.1
10,20,576.0
10,21,2606.5
10,22,874.1
10,23,628.8
10,24,291.5
10,25,474.5
10,26,687.6
10,27,912.9
10,28,17270.7
10,29,1800.5
10,30,3698.6
10,31,926.4
10,32,1782.5
10,33,2277.7
10,34,2883.4
10,35,3536.0
10,36,11482.3
10,37,171.0
10,38,4682.3
10,39,452.2
10,40,2478.2
10,41,866.5
10,42,454.6
10,43,106.9
10,44,10.4
10,45,630.6
10,46,51.6
10,47,67.5
10,48,241.7
10,49,35.5
10,50,24.3
10,52,37.0
10,53,193.8
10,54,214.2
10,55,98.9
10,56,59.7
10,57,343.3
10,58,128.8
10,59,256.3
10,60,3.2
10,62,8.1
10,64,271.5
10,65,25.1
10,67,1051.3
11,1,218.3
11,2,406.4
11,3,1173.4
11,4,388.3
11,5,521.2
11,6,857.0
11,7,1481.5
11,8,218.2
11,9,933.6
11,10,1316.0
11,11,149962.3
11,12,15170.7
11,13,3993.3
11,14,4188.3
11,15,3410.8
11,16,6027.1
11,17,5005.1
11,18,7691.0
11,19,27021.7
11,20,2186.6
11,21,17815.9
11,22,7470.5
11,23,1855.5
11,24,488.7
11,25,818.9
11,26,4234.6
11,27,167.4
11,28,1203.3
11,29,242.0
11,30,9774.8
11,31,20390.8
11,32,9803.8
11,33,1517.5
11,34,328.3
11,35,2017.3
11,36,1086.5
11,37,8.6
11,38,1252.0
11,39,82.1
11,40,69.3
11,41,1658.2
11,42,81.0
11,43,38.0
11,44,18.3
11,45,3580.2
11,46,52.1
11,47,362.3
11,48,254.3
11,50,321.1
11,51,857.9
11,52,482.3
11,53,262.7
11,55,523.6
11,57,2010.2
11,58,521.1
11,59,3.5
11,61,245.4
11,62,24.4
11,64,1429.5
11,65,429.2
11,67,6016.2
12,1,1219.0
12,2,1936.0
12,3,3969.6
12,4,2861.9
12,5,1546.1
12,6,1333.7
12,7,2131.1
12,8,1011.0
12,9,1194.8
12,10,2329.8
12,11,13519.2
12,12,144424.6
12,13,3741.3
12,14,18522.1
12,15,13714.4
12,16,11237.4
12,17,6748.2
12,18,14594.2
12,19,32322.5
12,20,6163.5
12,21,22532.1
12,22,20751.3
12,23,3546.3
12,24,3238.7
12,25,2938.6
12,26,330.1
12,27,1159.3
12,28,2096.5
12,29,1046.3
12,30,4510.2
12,31,9437.6
12,32,6981.5
12,33,4516.5
12,34,1579.6
12,35,2646.8
12,36,2998.3
12,37,84.9
12,38,1245.6
12,39,256.7
12,40,433.4
12,41,256.2
12,42,101.0
12,43,309.1
12,44,29.9
12,45,1437.2
12,46,614.7
12,47,1252.9
12,48,2517.5
12,49,854.2
12,50,143.2
12,51,2511.7
12,52,1805.7
12,53,4736.1
12,54,3706.4
12,55,6551.7
12,56,2322.0
12,57,9427.4
12,58,5510.0
12,59,1235.8
12,60,374.8
12,61,551.8
12,62,924.5
12,63,31.2
12,64,13093.1
12,65,709.3
12,66,303.9
12,67,8093.0
13,2,110.6
13,3,1010.7
13,5,76.3
13,8,628.3
13,10,400.0
13,11,3772.7
13,12,5240.1
13,13,149527.4
13,14,2035.2
13,15,1035.3
13,16,2537.3
13,17,7231.3
13,18,1807.6
13,19,31750.0
13,20,5593.1
13,21,1513.7
13,22,1534.2
13,23,418.8
13,25,2606.7
13,27,372.9
13,31,7449.1
13,32,930.4
13,33,603.9
13,34,819.6
13,36,767.2
13,37,3.1
13,38,208.0
13,39,24.8
13,40,31.0
13,45,229.1
13,46,57.3
13,47,133.5
13,48,48.0
13,49,40.8
13,50,8.4
13,51,1100.6
13,56,395.8
13,57,540.4
13,59,23.1
13,60,403.0
13,62,483.7
13,64,1122.1
13,65,379.9
13,66,25.4
13,67,8341.8
14,1,496.9
14,2,691.6
14,3,2434.3
14,4,703.8
14,5,305.0
14,6,124.5
14,7,2546.2
14,8,2485.3
14,10,192.1
14,11,4012.4
14,12,18099.2
14,13,1722.0
14,14,46880.6
14,15,2835.4
14,16,8108.8
14,17,2414.4
14,18,1713.8
14,19,11984.1
14,20,1018.2
14,21,6961.8
14,22,3308.3
14,23,2006.2
14,24,1104.1
14,25,936.3
14,29,226.2
14,30,1413.9
14,31,4144.2
14,32,4321.4
14,33,275.0
14,34,465.5
14,35,198.7
14,36,916.1
14,37,7.6
14,38,407.4
14,39,50.1
14,40,62.3
14,41,39.5
14,42,95.3
14,43,324.7
14,44,8.8
14,45,680.6
14,46,749.0
14,47,325.6
14,48,889.6
14,49,528.6
14,50,79.1
14,51,192.1
14,52,136.6
14,53,773.6
14,54,388.4
14,56,350.7
14,57,848.1
14,58,147.5
14,59,29.3
14,60,15.1
14,61,184.0
14,62,25.0
14,63,4.3
14,64,1038.9
14,65,122.4
14,66,101.1
14,67,1135.5
15,2,35.0
15,3,831.7
15,4,288.5
15,5,165.6
15,6,93.4
15,7,425.4
15,8,67.4
15,9,343.2
15,10,926.0
15,11,3065.9
15,12,13910.9
15,13,1035.3
15,14,2973.8
15,15,149088.8
15,16,727.1
15,17,822.6
15,18,1847.1
15,19,27026.4
15,20,8425.1
15,21,2309.4
15,22,2929.8
15,23,509.9
15,24,3562.2
15,25,1319.3
15,26,273.1
15,27,219.7
15,29,98.4
15,30,1026.9
15,31,845.5
15,32,854.6
15,33,285.1
15,34,1029.9
15,35,861.6
15,36,167.1
15,37,5.0
15,39,21.7
15,40,67.8
15,41,59.3
15,43,187.6
15,44,24.0
15,45,113.6
15,46,97.9
15,47,2132.7
15,48,181.9
15,49,4611.2
15,51,365.9
15,52,68.2
15,53,1038.4
15,55,255.0
15,56,102.2
15,57,1821.5
15,59,28.4
15,62,33.6
15,63,1.6
15,64,212.7
15,65,95.2
15,67,20284.2
16,2,1148.6
16,3,2235.6
16,5,370.7
16,6,338.4
16,7,2249.7
16,8,1422.5
16,9,23308.4
16,10,1574.3
16,11,5144.7
16,12,14305.2
16,13,2316.9
16,14,7388.1
16,15,834.0
16,16,128724.1
16,17,26675.2
16,18,6022.1
16,19,28693.9
16,20,1250.8
16,21,36958.5
16,22,3032.4
16,23,7580.9
16,24,711.8
16,25,644.7
16,26,90.0
16,27,173.9
16,28,568.2
16,29,786.7
16,30,1841.1
16,31,1654.8
16,32,2730.0
16,33,664.0
16,34,307.2
16,35,1197.9
16,36,769.0
16,37,116.2
16,39,30.2
16,40,633.1
16,41,57.6
16,42,33.4
16,43,56.4
16,44,138.4
16,45,911.5
16,46,2006.4
16,47,239.8
16,48,638.6
16,49,42.0
16,50,769.0
16,51,594.3
16,52,1127.0
16,53,1494.0
16,54,353.4
16,55,466.6
16,56,224.1
16,57,2820.1
16,58,901.9
16,59,23.3
16,60,10.4
16,61,8.2
16,62,58.2
16,63,7.4
16,64,2284.4
16,65,22.5
16,66,41.3
16,67,603.7
17,1,103.1
17,2,243.4
17,3,1107.4
17,5,462.1
17,6,98.0
17,7,522.1
17,8,226.0
17,9,246.0
17,10,1151.9
17,11,5845.1
17,12,6759.5
17,13,8351.7
17,14,2206.4
17,15,1143.8
17,16,25919.6
17,17,111694.4
17,18,1933.7
17,19,19477.9
17,20,3941.6
17,21,7037.3
17,22,2113.9
17,23,645.7
17,24,493.2
17,25,1079.5
17,26,564.0
17,27,454.1
17,28,760.8
17,29,183.4
17,30,4854.5
17,31,4124.0
17,32,1690.4
17,33,6074.2
17,34,455.4
17,35,796.4
17,36,1336.9
17,37,24.1
17,38,979.4
17,39,190.8
17,40,379.9
17,41,126.2
17,42,72.6
17,43,154.2
17,44,7.5
17,45,575.3
17,46,138.5
17,47,200.6
17,48,302.5
17,51,392.1
17,52,137.3
17,53,803.1
17,54,177.1
17,55,338.0
17,57,578.7
17,58,356.1
17,59,7.0
17,60,3.9
17,61,9.8
17,62,16.1
17,63,6.1
17,64,3224.4
17,65,39.8
17,66,211.3
17,67,527.9
18,1,112.0
18,2,35.1
18,3,221.9
18,6,266.8
18,7,142.9
18,8,98.2
18,10,841.9
18,11,7771.2
18,12,14866.0
18,13,1717.3
18,14,1713.8
18,15,1727.9
18,16,6637.2
18,17,2155.4
18,18,57534.6
18,19,18013.8
18,20,715.8
18,21,25901.2
18,22,25899.4
18,23,361.1
18,24,227.2
18,25,250.6
18,26,109.9
18,27,69.0
18,28,739.5
18,29,136.3
18,30,362.0
18,31,368.3
18,32,1479.7
18,33,282.8
18,34,331.7
18,35,337.7
18,36,708.1
18,37,10.7
18,38,196.2
18,40,3.4
18,41,142.5
18,42,51.3
18,43,20.3
18,44,6.6
18,45,1099.0
18,46,153.0
18,47,250.4
18,48,58.0
18,49,30.2
18,50,11.1
18,51,1004.2
18,52,28.5
18,53,476.1
18,57,390.8
18,58,275.1
18,59,126.7
18,62,5.8
18,63,1.4
18,64,1283.7
18,67,708.4
19,1,1847.9
19,2,2660.1
19,3,7671.4
19,4,5567.5
19,5,5771.1
19,6,4427.6
19,7,3103.8
19,8,3265.0
19,9,2010.2
19,10,3936.6
19,11,26131.7
19,12,30285.9
19,13,32555.4
19,14,12554.9
19,15,27541.6
19,16,26511.2
19,17,18997.9
19,18,17731.4
19,19,114641.1
19,20,22256.7
19,21,23502.9
19,22,21280.1
19,23,4141.7
19,24,1221.5
19,25,1736.6
19,26,627.4
19,27,1573.6
19,28,3738.3
19,29,2307.1
19,30,14262.4
19,31,13039.1
19,32,21782.5
19,33,8272.2
19,34,3320.5
19,35,8522.2
19,36,7277.7
19,37,124.2
19,38,5348.0
19,39,3569.0
19,40,528.0
19,41,1451.1
19,42,646.3
19,43,254.0
19,44,90.8
19,45,4070.1
19,46,2631.9
19,47,1548.9
19,48,6946.7
19,49,1661.7
19,50,427.5
19,51,410.8
19,52,1400.6
19,53,1891.8
19,54,2405.4
19,55,1669.8
19,56,1759.9
19,57,3118.7
19,58,2104.2
19,59,101.0
19,60,55.7
19,61,48.6
19,62,473.0
19,63,32.5
19,64,7887.2
19,65,407.9
19,66,432.9
19,67,1148.0
20,3,603.7
20,4,204.2
20,5,421.1
20,6,100.3
20,7,341.2
20,10,576.0
20,11,2186.6
20,12,6864.5
20,13,6668.5
20,14,1251.8
20,15,7902.1
20,16,1250.8
20,17,4270.5
20,18,715.8
20,19,24612.4
20,20,65851.8
20,21,4709.7
20,22,758.7
20,23,574.3
20,25,790.4
20,26,2662.9
20,27,444.3
20,28,1160.1
20,30,644.0
20,31,771.8
20,32,1230.9
20,33,239.1
20,34,107.4
20,36,381.0
20,38,450.4
20,39,200.8
20,41,1673.8
20,43,264.7
20,45,204.2
20,46,2.0
20,47,3012.9
20,48,55.1
20,49,41.9
20,50,347.0
20,51,262.5
20,52,200.8
20,55,457.6
20,56,199.2
20,57,711.5
20,60,267.8
20,61,155.4
20,62,217.4
20,63,1.3
20,64,1001.4
20,65,6.4
20,67,484.1
21,1,328.6
21,2,408.2
21,3,1028.2
21,4,923.7
21,5,689.2
21,6,1114.0
21,7,860.9
21,8,558.5
21,9,240.2
21,10,2298.3
21,11,17660.4
21,12,24041.5
21,13,162.3
21,14,6612.8
21,15,2027.7
21,16,37502.6
21,17,7407.2
21,18,25197.1
21,19,23355.2
21,20,5262.3
21,21,73570.8
21,22,16384.7
21,23,1319.4
21,24,618.5
21,25,791.6
21,26,353.7
21,27,725.0
21,28,3867.5
21,29,273.6
21,30,4712.6
21,31,2102.1
21,32,6679.0
21,33,2027.7
21,34,806.3
21,35,2695.3
21,36,1147.5
21,37,11.4
21,38,1098.2
21,39,79.7
21,40,230.6
21,41,404.4
21,42,851.6
21,43,239.8
21,44,40.1
21,45,1609.9
21,46,694.6
21,47,197.8
21,48,1123.5
21,49,342.4
21,50,246.0
21,51,652.9
21,52,819.5
21,53,650.9
21,54,466.5
21,55,1416.3
21,56,817.0
21,57,1542.3
21,58,609.3
21,59,51.1
21,60,205.7
21,61,46.9
21,62,740.1
21,63,7.8
21,64,4316.7
21,65,231.6
21,66,182.8
21,67,910.6
22,1,232.3
22,2,67.4
22,3,586.6
22,7,759.4
22,8,816.9
22,10,1232.0
22,11,8073.8
22,12,21160.9
22,13,1080.5
22,14,3594.9
22,15,2636.8
22,16,3032.4
22,17,3014.8
22,18,26542.7
22,19,20261.0
22,20,1051.7
22,21,15912.5
22,22,101198.6
22,23,889.3
22,24,949.1
22,25,491.1
22,27,296.9
22,28,262.7
22,30,1938.3
22,32,232.3
22,33,268.2
22,35,737.4
22,37,5.7
22,38,262.9
22,39,19.6
22,44,9.8
22,45,1598.7
22,46,1114.3
22,47,286.9
22,48,137.1
22,49,32.5
22,51,1244.7
22,52,31.3
22,53,2048.8
22,55,1263.7
22,56,1377.5
22,57,3761.9
22,58,271.7
22,59,400.4
22,61,541.2
22,62,343.3
22,64,2371.9
22,65,321.3
22,67,2323.3
23,1,5978.5
23,2,6865.1
23,3,7919.6
23,4,11284.3
23,5,529.8
23,6,595.4
23,7,9732.2
23,8,49.9
23,9,171.2
23,10,628.8
23,11,1605.4
23,12,3504.5
23,13,418.8
23,14,2068.7
23,15,454.1
23,16,6981.4
23,17,645.7
23,18,361.1
23,19,4134.2
23,20,809.8
23,21,1254.6
23,22,889.3
23,23,36815.5
23,24,1894.3
23,25,3735.0
23,28,452.7
23,29,602.0
23,30,611.6
23,31,613.6
23,32,2367.3
23,33,324.8
23,34,296.7
23,35,1225.5
23,36,446.4
23,37,4.0
23,38,657.9
23,39,124.7
23,40,35.8
23,41,167.6
23,42,89.0
23,43,117.3
23,44,15.8
23,45,522.8
23,46,271.0
23,47,17.2
23,48,4058.8
23,49,3795.8
23,50,457.9
23,51,690.3
23,52,446.5
23,53,849.0
23,54,675.5
23,55,1063.3
23,56,794.0
23,57,2235.1
23,58,487.8
23,59,41.3
23,60,57.7
23,62,95.5
23,63,16.2
23,64,2298.7
23,65,202.8
23,66,200.4
23,67,603.7
24,1,208.2
24,2,133.4
24,3,796.9
24,4,1239.0
24,5,67.7
24,7,1582.5
24,8,5053.5
24,10,291.5
24,11,488.7
24,12,3203.6
24,13,279.6
24,14,1104.1
24,16,711.8
24,17,328.5
24,18,227.2
24,19,1221.5
24,20,166.0
24,21,828.3
24,22,949.1
24,23,2412.3
24,24,62467.6
24,25,5948.8
24,28,251.7
24,30,151.2
24,33,359.5
24,34,96.5
24,37,4.9
24,39,25.8
24,41,71.8
24,45,2.0
24,47,1.0
24,48,179.4
24,49,39.0
24,50,919.7
24,51,211.3
24,52,577.4
24,53,644.9
24,54,394.3
24,56,169.2
24,57,919.3
24,58,355.8
24,59,8.9
24,60,2.5
24,61,2.2
24,62,35.9
24,63,1.4
24,64,2016.9
24,65,46.8
24,66,6692.8
24,67,223.6
25,1,336.3
25,2,504.6
25,3,1126.5
25,4,2520.6
25,5,85.0
25,6,302.0
25,7,3204.2
25,8,153.0
25,9,339.6
25,10,474.5
25,11,1505.0
25,12,2839.6
25,13,2606.7
25,14,936.3
25,15,1319.3
25,16,932.9
25,17,951.7
25,18,250.6
25,19,2265.8
25,20,790.4
25,21,791.6
25,22,491.1
25,23,3371.7
25,24,6076.0
25,25,33692.8
25,28,190.6
25,29,206.9
25,30,1017.9
25,32,230.6
25,33,322.2
25,34,350.6
25,35,485.8
25,36,211.4
25,37,2.1
25,38,291.4
25,39,43.2
25,41,68.1
25,42,308.3
25,43,19.6
25,44,11.2
25,45,130.3
25,46,193.7
25,47,87.4
25,48,1558.4
25,49,559.8
25,50,3636.0
25,51,227.0
25,52,176.5
25,53,537.2
25,54,645.0
25,55,685.0
25,56,37.6
25,57,682.3
25,58,401.0
25,59,157.3
25,60,3.2
25,62,5.2
25,63,6.5
25,64,3722.8
25,65,373.7
25,66,567.0
25,67,414.4
26,5,224.6
26,6,114.4
26,8,236.8
26,10,739.2
26,11,4556.4
26,12,657.4
26,14,864.7
26,15,273.1
26,16,184.2
26,17,675.7
26,18,109.9
26,19,717.7
26,20,228.6
26,21,453.0
26,26,21341.5
26,27,504.3
26,28,1567.0
26,30,655.8
26,35,364.6
26,36,862.5
26,37,5.9
26,41,8974.7
26,42,253.2
26,43,125.5
26,45,128.9
26,46,2.0
26,52,77.5
26,53,321.1
26,57,96.0
26,58,96.3
26,67,202.8
27,2,31.1
27,3,96.9
27,4,103.1
27,5,106.3
27,6,225.7
27,7,67.9
27,8,269.9
27,10,1120.2
27,11,167.4
27,12,1310.2
27,13,372.9
27,15,219.7
27,16,173.9
27,17,454.1
27,18,69.0
27,19,2013.1
27,20,444.3
27,21,618.7
27,22,296.9
27,26,468.0
27,27,25271.9
27,28,635.3
27,29,267.5
27,30,447.0
27,31,983.3
27,33,191.4
27,34,128.6
27,35,707.5
27,36,829.9
27,37,220.6
27,38,187.4
27,40,36.7
27,41,2035.7
27,42,583.3
27,43,6339.6
27,45,1.0
27,47,1.0
27,48,174.1
27,49,89.6
27,51,86.8
27,52,76.3
27,54,89.9
27,57,100.8
27,64,151.9
27,65,5.3
27,67,307.9
28,5,1007.9
28,6,681.1
28,8,2048.9
28,10,17990.1
28,11,1203.3
28,12,2283.4
28,16,568.2
28,17,760.8
28,18,552.5
28,19,3544.3
28,20,1160.1
28,21,3625.7
28,22,262.7
28,23,452.7
28,24,251.7
28,25,190.6
28,26,1567.0
28,27,635.3
28,28,13089.9
28,29,215.8
28,30,2397.0
28,31,333.0
28,33,708.1
28,34,337.9
28,35,420.9
28,36,2268.5
28,37,4.6
28,38,955.4
28,40,66.4
28,42,980.5
28,43,15.1
28,45,483.3
28,46,243.8
28,47,1.0
28,48,556.1
28,54,462.0
28,57,371.3
28,64,695.8
28,65,6.0
28,67,810.3
29,8,523.2
29,10,1613.8
29,11,144.9
29,12,1046.3
29,14,226.2
29,15,98.4
29,16,786.7
29,17,259.2
29,18,136.3
29,19,2665.0
29,21,346.6
29,23,602.0
29,25,206.9
29,27,267.5
29,28,215.8
29,29,6519.8
29,30,194.3
29,31,239.2
29,32,109.3
29,34,406.2
29,35,365.8
29,36,1035.4
29,37,122.9
29,38,189.5
29,42,431.6
29,47,60.5
29,48,64.0
29,49,68.5
29,57,535.8
29,64,232.8
29,65,70.1
29,67,182.5
30,1,80.7
30,2,95.3
30,3,290.5
30,4,1138.4
30,5,345.6
30,6,389.9
30,7,599.9
30,8,2949.0
30,9,120.7
30,10,3311.2
30,11,10169.7
30,12,4740.6
30,13,1088.7
30,14,1346.3
30,15,1542.6
30,16,1559.4
30,17,4705.0
30,18,362.0
30,19,13935.2
30,20,644.0
30,21,4451.4
30,22,1966.2
30,23,611.6
30,24,377.0
30,25,879.5
30,26,655.8
30,27,574.6
30,28,2443.2
30,29,194.3
30,30,118078.1
30,31,1752.1
30,32,10716.3
30,33,3208.7
30,34,1175.7
30,35,533.3
30,36,2420.4
30,37,36.2
30,38,948.9
30,39,89.7
30,40,83.9
30,41,729.3
30,42,80.6
30,43,435.0
30,44,5.6
30,45,1306.1
30,46,216.9
30,47,30.2
30,48,406.6
30,49,69.5
30,50,30.6
30,51,370.0
30,52,622.4
30,53,201.0
30,55,276.7
30,56,150.9
30,57,775.0
30,58,186.8
30,59,13.8
30,60,3.1
30,61,2.7
30,62,6.4
30,63,4.8
30,64,586.2
30,65,43.7
30,66,117.1
30,67,1166.7
31,3,165.7
31,4,2095.3
31,5,233.4
31,8,494.3
31,10,1428.5
31,11,21348.1
31,12,9071.8
31,13,6698.2
31,14,3762.9
31,15,183.9
31,16,1713.2
31,17,5176.9
31,18,378.2
31,19,14823.3
31,20,1399.5
31,21,1636.1
31,23,613.6
31,27,941.8
31,28,674.3
31,29,239.2
31,30,2893.6
31,31,87881.4
31,32,634.3
31,33,8771.6
31,34,1045.8
31,35,175.7
31,36,444.0
31,37,8.9
31,39,1954.2
31,40,107.6
31,42,176.2
31,43,975.5
31,44,72.1
31,45,338.0
31,46,93.4
31,47,70.4
31,48,1127.3
31,51,706.4
31,52,337.6
31,55,751.3
31,63,255.5
31,64,535.4
31,65,192.7
31,67,13455.9
32,2,140.1
32,3,646.1
32,4,1037.0
32,6,78.1
32,8,2313.9
32,10,1799.0
32,11,9311.0
32,12,6205.2
32,13,634.2
32,14,4677.5
32,15,473.8
32,16,2805.8
32,17,1365.8
32,18,1657.3
32,19,24482.9
32,20,1808.7
32,21,8167.6
32,22,232.3
32,23,2367.3
32,25,230.6
32,29,109.3
32,30,12427.9
32,31,1017.0
32,32,110036.0
32,33,177.8
32,35,193.6
32,36,350.2
32,37,3.5
32,38,2213.4
32,40,3.3
32,41,75.2
32,43,246.9
32,44,2.4
32,45,1327.7
32,46,326.3
32,47,2.0
32,48,154.4
32,49,120.4
32,55,221.0
32,56,31.7
32,57,289.8
32,59,750.4
32,62,6.8
32,64,1154.2
32,65,317.5
32,67,842.4
33,3,510.7
33,6,110.7
33,7,149.6
33,8,4981.5
33,9,262.1
33,10,2261.1
33,11,764.7
33,12,4574.9
33,13,1863.9
33,14,362.5
33,15,364.9
33,16,664.0
33,17,6145.9
33,18,403.4
33,19,14140.5
33,20,159.0
33,21,2026.7
33,22,394.7
33,23,621.2
33,24,359.5
33,25,532.0
33,27,191.4
33,28,708.1
33,30,3574.2
33,31,10677.4
33,32,469.0
33,33,41469.8
33,34,2880.4
33,35,626.8
33,36,77.9
33,37,160.3
33,39,24.0
33,40,252.5
33,42,1199.8
33,45,194.3
33,46,1.0
33,47,16.3
33,48,62.1
33,50,121.4
33,51,402.1
33,52,178.9
33,53,101.4
33,54,91.2
33,55,83.3
33,57,401.0
33,58,179.0
33,62,2.4
33,64,746.4
33,65,167.4
33,67,4259.4
34,5,287.3
34,6,169.6
34,8,1138.4
34,10,2877.9
34,11,328.3
34,12,2035.6
34,13,819.6
34,14,465.5
34,15,1029.9
34,16,400.2
34,17,455.4
34,18,331.7
34,19,3978.4
34,20,107.4
34,21,806.3
34,23,398.5
34,24,96.5
34,25,350.6
34,27,128.6
34,28,337.9
34,29,525.3
34,30,1201.2
34,31,735.5
34,33,3104.9
34,34,21237.3
34,35,468.7
34,36,4173.3
34,37,103.2
34,38,520.9
34,40,35.1
34,41,125.0
34,42,249.5
34,46,1.0
34,47,1.0
34,48,1.0
34,49,52.1
34,51,427.4
34,57,115.0
34,58,205.9
34,64,502.6
34,65,6.0
34,67,655.5
35,5,391.1
35,6,233.4
35,8,628.6
35,10,3711.6
35,11,1576.9
35,12,2838.9
35,14,198.7
35,15,1296.8
35,16,1197.9
35,17,796.4
35,18,337.7
35,19,8065.3
35,21,2695.3
35,22,737.4
35,23,1225.5
35,25,485.8
35,26,226.6
35,27,928.0
35,28,420.9
35,29,365.8
35,30,684.4
35,31,175.7
35,32,193.6
35,33,264.3
35,34,345.5
35,35,7252.8
35,36,634.8
35,37,76.2
35,38,201.9
35,40,190.6
35,41,203.1
35,42,12023.3
35,43,99.4
35,45,1.8
35,48,122.7
35,49,146.4
35,57,428.9
35,64,441.6
35,67,1141.5
36,2,58.1
36,3,259.3
36,5,1931.9
36,6,1485.3
36,7,276.5
36,8,1900.5
36,9,4789.8
36,10,11404.4
36,11,1086.5
36,12,2998.3
36,13,767.2
36,14,916.1
36,16,769.0
36,17,1340.0
36,18,597.9
36,19,7057.4
36,20,383.2
36,21,1147.5
36,23,446.4
36,25,211.4
36,26,771.0
36,27,811.0
36,28,2418.0
36,29,1102.2
36,30,2420.4
36,31,873.4
36,32,350.2
36,33,161.0
36,34,1892.3
36,35,746.7
36,36,18431.2
36,37,277.0
36,39,186.1
36,40,352.9
36,41,1324.2
36,42,912.4
36,43,193.6
36,45,292.1
36,47,608.9
36,48,2.0
36,49,95.3
36,52,58.3
36,53,188.4
36,56,68.5
36,57,792.3
36,58,43.1
36,59,3.9
36,63,4.5
36,64,1226.9
36,67,824.2
37,3,6.5
37,4,3.1
37,5,313.4
37,6,4.3
37,8,388.2
37,10,173.6
37,11,10.3
37,12,89.7
37,13,3.1
37,14,9.0
37,15,5.0
37,16,118.1
37,17,24.1
37,18,10.7
37,19,121.6
37,21,11.4
37,22,5.7
37,23,4.0
37,24,2.9
37,25,2.1
37,26,5.9
37,27,222.7
37,28,4.6
37,29,122.9
37,30,44.5
37,31,8.9
37,32,3.5
37,33,160.3
37,34,7.3
37,35,74.4
37,36,278.8
37,37,234.5
37,40,192.0
37,41,2.9
37,42,97.3
37,43,22.0
37,51,7.7
37,52,2.2
37,57,14.1
37,61,2.1
37,64,19.8
37,66,3.2
37,67,15.3
38,5,687.4
38,6,424.1
38,8,1090.9
38,10,4591.3
38,11,1657.9
38,12,1930.8
38,13,208.0
38,14,407.4
38,15,296.3
38,17,1012.5
38,18,196.2
38,19,5778.0
38,20,450.4
38,21,585.5
38,22,262.9
38,23,657.9
38,25,291.4
38,28,955.4
38,29,189.5
38,30,948.9
38,32,2005.5
38,34,520.9
38,35,201.9
38,38,29474.9
38,40,549.9
38,42,184.8
38,45,495.7
38,64,900.4
38,67,152.6
39,5,97.3
39,8,79.1
39,10,452.2
39,11,82.1
39,12,256.7
39,13,24.8
39,14,50.1
39,15,21.7
39,16,30.2
39,17,190.8
39,19,3602.6
39,20,200.8
39,21,79.7
39,22,19.6
39,23,124.7
39,24,25.8
39,25,43.2
39,30,89.7
39,31,1986.8
39,33,24.0
39,36,186.1
39,39,4317.1
39,40,703.0
39,42,179.4
39,43,30.2
39,47,20.5
39,53,26.6
39,55,23.9
39,57,77.1
39,61,34.5
39,64,77.4
40,3,32.9
40,5,1149.6
40,6,208.8
40,8,136.7
40,10,2631.3
40,11,69.3
40,12,433.4
40,13,31.0
40,14,62.3
40,15,67.8
40,16,633.1
40,17,379.9
40,18,3.4
40,19,528.0
40,21,230.6
40,23,35.8
40,27,36.7
40,28,101.4
40,30,83.9
40,31,107.6
40,32,3.3
40,33,261.9
40,34,35.1
40,35,190.6
40,36,352.9
40,37,192.0
40,38,549.9
40,39,737.8
40,40,2515.4
40,42,103.8
40,43,77.2
40,45,75.7
40,48,175.3
40,57,2.9
40,67,646.4
41,2,205.2
41,4,175.4
41,5,81.1
41,8,51.5
41,10,788.9
41,11,1701.6
41,12,411.0
41,14,39.5
41,15,59.3
41,16,57.6
41,17,186.5
41,18,142.5
41,19,1290.9
41,20,454.8
41,21,381.0
41,23,173.0
41,24,365.8
41,25,193.1
41,26,8748.8
41,27,1966.4
41,30,1106.2
41,32,618.2
41,35,203.1
41,36,1232.6
41,37,2.9
41,41,11647.1
41,42,770.7
41,43,1022.9
41,44,1.0
41,45,108.0
41,46,1.0
41,47,13.0
41,55,90.2
41,61,142.8
41,64,72.9
41,67,78.1
42,3,349.0
42,4,47.4
42,5,593.3
42,6,322.7
42,7,73.9
42,8,194.0
42,10,474.3
42,11,79.3
42,12,101.0
42,14,95.3
42,16,33.4
42,17,72.6
42,18,51.3
42,19,646.3
42,21,851.6
42,23,89.0
42,25,308.3
42,26,158.1
42,27,583.3
42,28,980.5
42,29,431.6
42,30,80.6
42,31,176.2
42,33,1054.6
42,34,133.3
42,35,12415.8
42,36,912.4
42,37,95.8
42,38,184.8
42,39,179.4
42,40,103.8
42,41,770.7
42,42,7444.1
42,43,204.0
42,44,5.1
42,45,33.9
42,47,61.1
42,48,2.0
42,49,27.7
42,54,21.1
42,57,140.9
42,59,2.5
42,64,42.1
42,65,6.2
42,66,17.6
42,67,265.2
43,2,20.1
43,3,55.3
43,5,26.9
43,8,254.9
43,10,106.9
43,11,38.0
43,12,325.7
43,13,1038.6
43,14,324.7
43,15,187.6
43,16,56.4
43,17,154.2
43,18,174.0
43,19,269.3
43,20,264.7
43,21,236.9
43,23,117.3
43,25,19.6
43,26,217.7
43,27,7435.6
43,28,15.1
43,30,435.0
43,31,2211.2
43,32,268.5
43,35,116.4
43,36,174.7
43,37,22.0
43,39,30.2
43,40,77.2
43,41,871.4
43,42,204.0
43,43,8000.0
43,45,22.6
43,46,16.7
43,47,1.0
43,48,55.5
43,57,56.1
43,64,61.8
43,67,54.7
44,3,3.3
44,7,1.0
44,8,5.0
44,10,10.4
44,11,21.7
44,12,39.7
44,14,8.8
44,15,24.0
44,16,147.4
44,17,7.5
44,18,6.6
44,19,89.0
44,21,37.9
44,22,9.8
44,23,72.1
44,25,8.9
44,30,6.6
44,31,72.1
44,32,255.1
44,41,1.0
44,42,5.1
44,44,122.4
44,45,160.9
44,46,53.4
44,48,58.7
44,49,2.0
44,51,5.3
44,53,1.0
44,55,6.8
44,57,2.4
44,64,4.8
44,66,1.0
44,67,243.0
45,3,514.0
45,5,65.6
45,6,182.1
45,7,75.5
45,8,42.2
45,10,629.6
45,11,3721.7
45,12,1555.6
45,13,230.1
45,14,680.6
45,15,113.6
45,16,637.9
45,17,739.8
45,18,766.7
45,19,4459.3
45,20,204.2
45,21,1754.9
45,22,1931.0
45,23,405.5
45,24,2.0
45,25,130.3
45,26,128.9
45,28,483.3
45,30,1157.2
45,31,338.0
45,32,1327.7
45,33,194.3
45,35,1.8
45,36,292.1
45,38,495.7
45,40,75.7
45,41,107.0
45,42,33.9
45,43,22.6
45,44,157.5
45,45,27079.5
45,46,6.3
45,47,152.9
45,48,48.9
45,49,282.0
45,51,2.0
45,54,1.0
45,55,1.0
45,57,842.5
45,58,54.8
45,59,7.0
45,61,1.0
45,64,133.1
45,67,5081.4
46,2,37.9
46,3,2.0
46,4,48.6
46,10,51.6
46,11,53.1
46,12,614.7
46,13,57.3
46,14,688.9
46,15,97.9
46,16,2064.7
46,17,138.5
46,18,153.0
46,19,2751.6
46,20,2.0
46,21,693.6
46,22,1114.3
46,23,272.0
46,25,193.7
46,26,2.0
46,28,243.8
46,30,216.9
46,31,92.4
46,32,326.3
46,33,1.0
46,34,1.0
46,43,16.7
46,44,53.4
46,45,6.3
46,46,1590.8
46,47,13.5
46,48,1.0
46,51,1.0
46,52,140.7
46,53,1.0
46,57,110.2
46,59,54.6
46,61,1.0
46,64,122.6
46,67,666.0
47,1,14.7
47,3,14.9
47,4,1.0
47,7,25.3
47,10,79.7
47,11,362.3
47,12,1254.7
47,13,136.4
47,14,84.0
47,15,2107.9
47,16,237.2
47,17,199.6
47,18,234.6
47,19,1624.0
47,20,2147.1
47,21,213.1
47,22,286.9
47,23,18.2
47,24,1.0
47,25,87.4
47,27,1.0
47,28,16.5
47,29,60.5
47,30,44.3
47,31,57.9
47,32,2.0
47,33,16.3
47,34,1.0
47,36,442.8
47,39,20.5
47,41,1.0
47,42,60.1
47,43,1.0
47,45,152.9
47,46,13.5
47,47,2121.9
47,48,311.9
47,49,1.0
47,50,8.5
47,54,14.7
47,55,217.9
47,57,130.6
47,58,18.7
47,59,7.7
47,60,3.2
47,61,100.8
47,62,15.8
47,64,35.7
47,65,7.7
47,66,1.0
47,67,540.4
48,1,2.0
48,2,68.5
48,3,1837.9
48,4,1429.2
48,7,655.2
48,8,114.0
48,10,179.6
48,11,254.3
48,12,2922.4
48,13,48.0
48,14,945.1
48,15,181.9
48,16,638.6
48,17,302.5
48,18,59.0
48,19,6815.6
48,20,55.1
48,21,1004.1
48,22,197.5
48,23,4135.4
48,24,179.4
48,25,1557.5
48,27,174.1
48,28,556.1
48,29,64.0
48,30,287.1
48,31,892.5
48,32,154.4
48,33,62.1
48,34,1.0
48,35,184.8
48,36,2.0
48,40,175.3
48,42,2.0
48,43,55.5
48,44,58.7
48,45,48.9
48,46,1.0
48,47,311.9
48,48,16751.9
48,49,310.3
48,50,156.7
48,51,451.4
48,52,520.4
48,53,383.5
48,55,1.0
48,56,1.0
48,57,281.5
48,58,1856.4
48,60,54.1
48,62,56.5
48,63,2.8
48,64,980.1
48,65,56.7
48,66,176.5
48,67,2472.6
49,1,398.9
49,2,505.8
49,3,4671.7
49,4,1807.8
49,7,488.1
49,8,32.7
49,10,35.5
49,12,892.6
49,13,40.8
49,14,605.3
49,15,4611.2
49,16,42.0
49,18,30.2
49,19,1797.0
49,20,41.9
49,21,342.4
49,22,32.5
49,23,3710.7
49,24,39.0
49,25,521.4
49,27,89.6
49,29,68.5
49,30,69.5
49,32,120.4
49,34,52.1
49,35,146.4
49,36,95.3
49,42,27.7
49,44,2.0
49,45,282.0
49,47,1.0
49,48,348.7
49,49,8847.4
49,51,398.1
49,53,82.7
49,55,115.7
49,57,204.2
49,58,121.9
49,63,5.5
49,64,1127.5
49,65,47.9
49,66,37.2
49,67,186.1
50,1,11.6
50,2,11.3
50,3,137.3
50,4,58.6
50,5,15.6
50,7,113.3
50,10,24.3
50,11,321.1
50,12,235.1
50,13,8.4
50,14,161.9
50,16,812.7
50,18,39.7
50,19,689.8
50,21,71.4
50,23,465.9
50,24,1471.4
50,25,4206.9
50,26,14.8
50,30,30.6
50,33,121.4
50,40,31.8
50,41,14.8
50,47,8.5
50,48,156.7
50,50,2544.4
50,51,45.6
50,52,25.6
50,53,17.0
50,55,10.1
50,56,28.8
50,57,199.7
50,58,30.0
50,59,14.8
50,61,11.1
50,62,10.6
50,64,1177.8
50,65,1294.4
50,66,235.2
50,67,193.7
51,1,85.8
51,2,298.3
51,3,153.7
51,4,583.4
51,7,201.8
51,8,82.7
51,11,599.0
51,12,1579.6
51,13,1547.0
51,14,192.1
51,15,365.9
51,16,742.7
51,17,392.1
51,18,1344.1
51,19,868.1
51,20,262.5
51,21,503.5
51,22,1168.1
51,23,686.3
51,24,211.3
51,25,227.0
51,27,86.8
51,30,240.4
51,31,706.4
51,33,660.0
51,34,427.4
51,37,7.7
51,44,5.3
51,45,2.0
51,46,1.0
51,48,526.8
51,49,436.9
51,50,45.6
51,51,262.9
51,52,456.1
51,54,1172.1
51,55,1075.6
51,56,349.0
51,57,506.8
51,58,1289.8
51,59,89.5
51,60,21.8
51,61,5.1
51,62,81.5
51,63,6.2
51,64,2327.9
51,65,128.6
51,66,35.8
52,1,157.1
52,2,126.7
52,4,366.3
52,6,95.4
52,7,264.4
52,8,196.0
52,10,37.0
52,11,446.8
52,12,1737.9
52,14,103.8
52,15,68.2
52,16,1895.6
52,17,137.3
52,18,24.1
52,19,1433.3
52,20,200.8
52,21,819.5
52,22,31.3
52,23,406.3
52,24,577.4
52,25,176.5
52,27,76.3
52,30,745.9
52,31,337.6
52,33,178.9
52,36,58.3
52,37,2.2
52,46,140.7
52,47,1.0
52,48,520.4
52,50,25.6
52,51,613.8
52,52,2709.3
52,53,1800.1
52,54,799.1
52,55,406.8
52,56,1172.2
52,57,4357.9
52,58,1623.0
52,59,156.9
52,60,6.4
52,62,39.9
52,63,7.0
52,64,1189.4
52,65,19.9
53,1,103.5
53,2,104.7
53,3,942.6
53,4,167.4
53,7,136.0
53,10,193.8
53,11,262.7
53,12,4305.3
53,14,951.4
53,15,587.3
53,16,760.1
53,17,993.5
53,18,247.6
53,19,1913.0
53,20,188.2
53,21,650.9
53,22,2048.8
53,23,769.2
53,24,644.9
53,25,537.2
53,26,321.1
53,30,201.0
53,33,101.4
53,36,188.4
53,39,26.6
53,46,1.0
53,48,383.5
53,49,82.7
53,50,17.0
53,51,339.8
53,52,1943.2
53,53,3815.0
53,54,2777.5
53,55,485.9
53,56,631.6
53,57,16650.0
53,58,2169.1
53,59,118.9
53,60,27.2
53,61,254.7
53,62,54.1
53,63,149.7
53,64,3348.0
53,65,95.7
53,67,382.1
54,1,114.6
54,2,110.5
54,3,427.2
54,7,122.4
54,10,214.2
54,12,3439.9
54,14,388.4
54,16,208.4
54,19,2415.6
54,20,115.9
54,21,765.6
54,23,675.5
54,24,394.3
54,25,645.0
54,27,89.9
54,28,462.0
54,33,91.2
54,41,96.9
54,42,21.1
54,44,1.0
54,45,1.0
54,47,13.7
54,51,1334.0
54,52,1043.3
54,53,3283.1
54,54,11541.3
54,55,1750.3
54,56,1746.2
54,57,10268.8
54,58,568.7
54,59,411.2
54,60,330.8
54,61,5.1
54,62,66.5
54,63,310.6
54,64,789.4
54,65,31.9
54,67,812.7
55,2,54.7
55,3,914.1
55,4,75.1
55,7,151.5
55,8,185.6
55,10,98.9
55,11,523.6
55,12,6912.3
55,13,345.0
55,15,255.0
55,16,466.6
55,17,338.0
55,19,1322.7
55,20,112.6
55,21,1416.3
55,22,1263.7
55,23,928.4
55,25,557.1
55,30,206.8
55,31,751.3
55,32,416.4
55,33,83.3
55,39,23.9
55,41,90.2
55,44,8.7
55,45,1.0
55,47,217.9
55,48,1.0
55,49,115.7
55,50,10.1
55,51,1646.4
55,52,406.8
55,53,2086.8
55,54,1599.1
55,55,4006.6
55,56,1988.4
55,57,5342.3
55,58,976.8
55,59,936.4
55,60,323.9
55,62,822.3
55,63,83.6
55,64,2134.7
55,65,259.6
55,66,132.0
55,67,1158.1
56,2,109.7
56,3,494.0
56,4,80.2
56,7,114.7
56,8,211.2
56,10,59.7
56,12,2266.2
56,13,395.8
56,14,407.4
56,15,65.1
56,16,261.3
56,17,50.9
56,19,1580.6
56,21,855.0
56,22,1042.5
56,23,794.0
56,24,169.2
56,25,37.6
56,27,336.8
56,28,42.5
56,30,138.3
56,32,31.7
56,36,68.5
56,48,1.0
56,50,28.8
56,51,412.5
56,52,1257.8
56,53,631.6
56,54,1744.5
56,55,1951.6
56,56,5790.6
56,57,8006.1
56,58,1047.3
56,59,311.6
56,60,212.1
56,61,12.2
56,62,134.7
56,63,41.5
56,64,609.1
56,65,16.8
56,67,99.2
57,1,450.6
57,2,855.1
57,3,657.1
57,4,213.2
57,5,307.4
57,6,1571.1
57,7,977.2
57,8,102.4
57,10,343.3
57,11,1888.7
57,12,6390.1
57,13,540.4
57,14,1014.4
57,15,182.3
57,16,4132.0
57,17,578.7
57,18,252.0
57,19,3825.8
57,20,595.6
57,21,1585.2
57,22,3761.9
57,23,2106.1
57,24,973.5
57,25,957.3
57,26,96.0
57,27,100.8
57,28,371.3
57,29,535.8
57,30,749.9
57,32,289.8
57,33,155.7
57,35,428.9
57,36,795.5
57,37,14.1
57,39,77.1
57,40,2.9
57,41,31.8
57,42,140.9
57,43,56.1
57,44,4.6
57,45,843.5
57,46,109.2
57,47,113.4
57,48,253.5
57,49,204.2
57,50,65.3
57,51,523.0
57,52,4265.2
57,53,16395.4
57,54,10745.1
57,55,4561.5
57,56,6835.3
57,57,24719.9
57,58,4817.9
57,59,364.6
57,60,294.5
57,61,128.6
57,62,693.3
57,63,139.1
57,64,2278.2
57,65,341.1
57,66,226.0
57,67,373.5
58,1,131.2
58,2,380.5
58,3,558.3
58,4,120.7
58,7,107.1
58,10,128.8
58,11,579.0
58,12,6790.1
58,14,182.6
58,16,849.6
58,17,356.1
58,18,275.1
58,19,2293.8
58,21,609.3
58,22,271.7
58,23,563.8
58,24,266.5
58,25,346.0
58,26,96.3
58,33,142.2
58,36,43.1
58,45,54.8
58,47,18.7
58,48,1848.2
58,49,121.9
58,50,30.0
58,51,1411.0
58,52,1638.0
58,53,1864.5
58,54,577.4
58,55,931.1
58,56,1047.3
58,57,6343.4
58,58,2591.1
58,59,1278.8
58,60,177.2
58,61,11.2
58,62,621.7
58,63,99.1
58,64,635.1
58,65,92.4
58,67,93.7
59,3,7.9
59,4,5.6
59,7,6.5
59,10,253.1
59,11,3.5
59,12,1260.0
59,13,23.1
59,14,23.3
59,15,28.4
59,16,23.3
59,17,10.1
59,18,131.1
59,19,115.2
59,21,51.1
59,22,400.4
59,23,41.3
59,24,8.9
59,25,157.3
59,30,83.8
59,32,750.4
59,36,3.9
59,42,2.5
59,45,7.0
59,46,54.6
59,47,7.7
59,50,14.8
59,51,73.3
59,52,77.2
59,53,122.2
59,54,422.9
59,55,936.8
59,56,314.4
59,57,421.5
59,58,920.0
59,59,130.8
59,60,32.4
59,61,7.5
59,62,57.3
59,63,6.4
59,64,49.5
59,66,6.9
59,67,31.2
60,1,77.4
60,3,2.9
60,4,3.3
60,12,383.8
60,13,403.0
60,14,15.1
60,16,10.4
60,17,3.9
60,19,56.4
60,20,14.3
60,21,205.7
60,23,60.2
60,24,2.5
60,25,3.2
60,30,3.1
60,47,3.2
60,48,111.2
60,51,21.8
60,52,6.4
60,53,27.2
60,54,327.7
60,55,323.9
60,56,215.8
60,57,297.6
60,58,177.2
60,59,32.4
60,60,97.6
60,61,8.5
60,62,81.5
60,63,29.7
60,64,19.6
60,66,2.6
60,67,3.1
61,1,124.2
61,2,35.5
61,3,2.5
61,5,170.1
61,6,95.7
61,11,245.4
61,12,551.8
61,14,184.0
61,15,2.1
61,16,8.2
61,17,9.8
61,19,52.6
61,21,35.1
61,22,541.2
61,23,2.8
61,24,2.2
61,26,41.4
61,30,5.0
61,33,68.4
61,36,2.1
61,37,2.1
61,39,34.5
61,41,242.7
61,45,1.0
61,47,102.6
61,50,11.1
61,51,2.3
61,53,254.7
61,54,5.1
61,56,19.9
61,57,123.2
61,58,108.7
61,59,11.0
61,60,8.5
61,61,108.7
61,62,246.4
61,63,1.7
61,64,9.4
61,67,80.2
62,1,3.0
62,3,16.4
62,4,11.9
62,7,158.6
62,8,3.6
62,10,8.1
62,11,15.8
62,12,945.0
62,13,483.7
62,14,32.5
62,15,27.9
62,16,346.8
62,17,16.1
62,18,5.8
62,19,481.9
62,20,6.2
62,21,469.7
62,22,343.3
62,23,95.5
62,24,35.9
62,25,5.2
62,30,8.4
62,32,6.8
62,33,2.4
62,47,18.1
62,48,56.5
62,50,10.6
62,51,155.9
62,52,39.9
62,53,54.1
62,54,63.7
62,55,822.3
62,56,137.9
62,57,708.2
62,58,624.0
62,59,57.3
62,60,81.5
62,61,249.5
62,62,1818.1
62,63,18.1
62,64,99.1
62,65,24.1
62,67,140.4
63,3,5.3
63,4,1.4
63,7,80.6
63,12,32.7
63,14,5.5
63,15,1.6
63,16,9.3
63,17,2.9
63,18,1.4
63,19,32.5
63,20,1.3
63,21,9.7
63,23,16.2
63,24,1.4
63,25,6.5
63,30,6.0
63,36,4.5
63,48,2.8
63,49,5.5
63,51,6.2
63,52,7.0
63,53,147.9
63,54,310.8
63,55,83.6
63,56,39.8
63,57,234.5
63,58,6.8
63,59,6.4
63,60,29.7
63,61,1.7
63,62,18.1
63,63,26.9
63,64,188.3
63,67,4.0
64,1,685.6
64,2,790.9
64,3,1065.3
64,4,1585.2
64,5,514.9
64,6,301.7
64,7,3076.8
64,8,165.1
64,10,1300.8
64,11,1918.5
64,12,10253.1
64,13,550.3
64,14,674.2
64,15,212.7
64,16,2008.1
64,17,2879.9
64,18,1282.0
64,19,6988.3
64,20,625.5
64,21,5859.3
64,22,1256.0
64,23,2438.1
64,24,2016.9
64,25,3668.7
64,27,151.9
64,28,695.8
64,29,232.8
64,30,713.8
64,31,535.4
64,32,1407.3
64,33,630.2
64,34,364.8
64,35,441.6
64,36,1385.5
64,37,19.8
64,38,1091.9
64,39,77.4
64,41,134.7
64,42,43.1
64,43,61.8
64,44,3.0
64,45,132.1
64,46,122.6
64,47,63.1
64,48,923.1
64,49,1010.2
64,50,764.0
64,51,2120.5
64,52,1219.8
64,53,3273.5
64,54,789.4
64,55,2058.0
64,56,609.1
64,57,2546.4
64,58,518.5
64,59,49.5
64,60,19.6
64,61,9.4
64,62,99.1
64,63,186.6
64,64,74727.3
64,65,5578.8
64,66,322.2
64,67,1036.9
65,1,4.3
65,2,11.4
65,3,73.4
65,4,23.5
65,7,133.2
65,10,25.1
65,11,434.3
65,12,908.4
65,13,379.9
65,14,121.3
65,15,95.2
65,16,47.6
65,17,39.8
65,19,577.5
65,20,6.4
65,21,89.8
65,22,321.3
65,23,239.1
65,24,41.9
65,25,378.7
65,27,11.3
65,28,6.0
65,29,70.1
65,30,47.4
65,31,192.7
65,32,517.2
65,33,87.4
65,34,6.0
65,47,7.7
65,48,56.7
65,49,47.9
65,50,965.9
65,51,128.0
65,52,19.9
65,53,102.5
65,54,31.9
65,55,259.6
65,56,16.8
65,57,480.6
65,58,178.4
65,62,24.1
65,64,4972.0
65,65,4563.7
65,66,33.1
65,67,52.8
66,3,130.1
66,4,349.5
66,7,146.1
66,8,397.1
66,12,303.9
66,13,25.4
66,14,101.1
66,16,41.3
66,17,211.3
66,19,414.1
66,21,182.8
66,23,200.4
66,24,9690.5
66,25,611.0
66,30,117.1
66,37,3.2
66,42,17.6
66,44,1.0
66,47,1.0
66,48,176.5
66,49,37.2
66,50,235.2
66,51,35.8
66,55,132.0
66,56,45.9
66,57,226.0
66,58,57.0
66,59,6.9
66,60,2.6
66,64,265.2
66,65,33.1
66,66,1978.0
66,67,531.1
67,1,457.9
67,2,66.2
67,3,576.7
67,5,197.9
67,6,1550.2
67,7,338.6
67,8,281.9
67,9,712.4
67,10,880.9
67,11,5883.1
67,12,2736.4
67,13,6484.3
67,14,180.4
67,15,1966.0
67,16,1274.0
67,17,350.8
67,18,358.7
67,19,390.8
67,21,1020.3
67,22,1613.2
67,23,632.0
67,24,223.6
67,25,286.3
67,26,110.0
67,27,307.9
67,28,262.7
67,29,182.5
67,30,690.4
67,31,1856.5
67,32,306.4
67,33,238.6
67,34,534.0
67,35,1141.5
67,36,824.2
67,37,15.3
67,38,152.6
67,40,404.9
67,41,78.1
67,42,265.2
67,43,35.2
67,44,233.0
67,45,5080.4
67,46,665.0
67,47,508.7
67,48,2465.6
67,49,186.1
67,50,148.0
67,53,382.1
67,54,812.7
67,55,570.1
67,56,95.6
67,57,436.5
67,58,93.7
67,59,25.7
67,60,6.7
67,61,82.4
67,62,129.7
67,63,1.4
67,64,277.7
67,65,25.0
67,66,489.6
67,67,1312.3

### Paquete `mobipredict`

In [ ]:
%%writefile src/mobipredict/__init__.py
"""MobiPredict: modelo predictivo de movilidad urbana del Valle de Aburrá."""
__version__ = "1.0.0"

### 1.1 `config.py` — URLs oficiales, rutas, ventanas temporales y parámetros

In [ ]:
%%writefile src/mobipredict/config.py
"""Configuración central de MobiPredict (datos reales).

Todas las rutas, URLs de las fuentes oficiales, ventanas temporales y parámetros
viven aquí para que cada módulo (ingesta, EDA, features, entrenamiento, API) use
exactamente los mismos valores. Se pueden sobrescribir con variables de entorno.
"""
from __future__ import annotations

import os
from pathlib import Path

# --------------------------------------------------------------------------- #
# Rutas
# --------------------------------------------------------------------------- #
ROOT = Path(os.getenv("MOBIPREDICT_HOME", Path(__file__).resolve().parents[2]))

DATA_RAW = ROOT / "data" / "raw"
DATA_REFERENCE = ROOT / "data" / "reference"
DATA_INTERIM = ROOT / "data" / "interim"
DATA_PROCESSED = ROOT / "data" / "processed"
EDA_FIGURES = ROOT / "eda" / "figures"
EDA_REPORTS = ROOT / "eda" / "reports"
MODELS_DIR = ROOT / "models"
DOCS_DIR = ROOT / "docs"
MLRUNS_DIR = ROOT / "mlruns"

ALL_DIRS = [DATA_RAW, DATA_INTERIM, DATA_PROCESSED, EDA_FIGURES, EDA_REPORTS, MODELS_DIR, DOCS_DIR, MLRUNS_DIR]

# --------------------------------------------------------------------------- #
# Fuentes oficiales (todas abiertas). Ver docs/fuentes_de_datos.md
# --------------------------------------------------------------------------- #
MEDATA = "https://medata.app.medellin.gov.co/sites/default/files/distribution"
EOD_API = os.getenv("MOBIPREDICT_EOD_API", "https://eod2025.metropol.gov.co/api/api")

FUENTES = {
    "velocidad_gt": {
        "url": f"{MEDATA}/1-023-25-000287/Velocidad_y_tiempo_de_viaje_GT.csv",
        "archivo": "medata_velocidad_tiempo_viaje_gt.csv",
        "titulo": "Velocidad y tiempo de viaje GT (SIMM, Secretaría de Movilidad de Medellín)",
        "ficha": "https://www.datos.gov.co/d/7t5n-3b3w",
    },
    "aforos": {
        "url": f"{MEDATA}/1-023-25-000301/Aforos_Vehiculares.csv",
        "archivo": "medata_aforos_vehiculares.csv",
        "titulo": "Aforos vehiculares (Secretaría de Movilidad de Medellín)",
        "ficha": "https://www.datos.gov.co/d/b9s9-jw7c",
    },
    "siniestros": {
        "url": "https://data.mendeley.com/public-files/datasets/r6g5dfnpgh/files/"
               "e992c888-5819-40c6-9538-41980ea1d429/file_downloaded",
        "archivo": "mendeley_siniestros_medellin_2008_2025.zip",
        "titulo": "Road-traffic incidents and fatalities in Medellín 2008–2025 (Mendeley Data, CC BY 4.0)",
        "ficha": "https://doi.org/10.17632/r6g5dfnpgh.1",
    },
    "eod2025": {
        "url": EOD_API,
        "archivo": "eod2025_indicadores.json",
        "titulo": "Encuesta Origen-Destino 2025 — API del visor oficial (AMVA)",
        "ficha": "https://eod2025.metropol.gov.co/",
    },
    "clima": {
        "url": "https://archive-api.open-meteo.com/v1/archive",
        "archivo": "open_meteo_medellin_horario.csv",
        "titulo": "Open-Meteo Historical Weather API (reanálisis ERA5 / ERA5-Land)",
        "ficha": "https://open-meteo.com/en/docs/historical-weather-api",
    },
}
CLIMA_LAT, CLIMA_LON = 6.2442, -75.5812          # centro de Medellín
HTTP_TIMEOUT_S = float(os.getenv("MOBIPREDICT_HTTP_TIMEOUT", "120"))

MACROZONAS_FILE = DATA_REFERENCE / "eod2025_macrozonas.csv"
CORREDORES_FILE = DATA_REFERENCE / "corredores_simm.csv"
MANIFEST_FILE = DATA_RAW / "manifest.json"

# Tablas estandarizadas (salida de fuentes.py)
SENSORES_STD = DATA_INTERIM / "sensores_velocidad_horaria.parquet"
TRAMOS_STD = DATA_INTERIM / "tramos.csv"
CLIMA_STD = DATA_INTERIM / "clima_horario.parquet"
FESTIVOS_STD = DATA_INTERIM / "festivos_colombia.csv"
EOD_HORA_STD = DATA_INTERIM / "eod2025_viajes_por_hora.csv"
EOD_OD_STD = DATA_INTERIM / "eod2025_matriz_od_macrozonas.csv"
SINIESTROS_STD = DATA_INTERIM / "siniestros_2008_2025.parquet"
AFOROS_STD = DATA_INTERIM / "aforos_composicion.parquet"

# --------------------------------------------------------------------------- #
# Cifras publicadas usadas como referencia (no se recalculan con los datos)
# --------------------------------------------------------------------------- #
REFERENCIAS = {
    "muertes_viales_medellin_2025": {
        "valor": 276, "detalle": "162 motociclistas y 100 peatones; -11,8 % frente a 2024",
        "fuente": "Observatorio de Movilidad de Medellín, vía El Colombiano (8 ene 2026)",
        "url": "https://www.elcolombiano.com/medellin/balance-de-accidentalidad-en-medellin-el-ano-pasado-PF32374139"},
    "variacion_tenencia_motos_2017_2025_pct": {
        "valor": 48.0, "fuente": "Resultados EOD 2025 (AMVA), vía El Colombiano (2 may 2026)",
        "url": "https://www.elcolombiano.com/medellin/movilidad-habitantes-medellin-aburra-disminucion-teletrabajo-DD36063145"},
    "variacion_participacion_tp_2017_2025_pct": {
        "valor": -14.0, "fuente": "Resultados EOD 2025 (AMVA), vía El Colombiano (2 may 2026)",
        "url": "https://www.elcolombiano.com/medellin/movilidad-habitantes-medellin-aburra-disminucion-teletrabajo-DD36063145"},
}

# --------------------------------------------------------------------------- #
# Ventanas temporales del modelado (serie de velocidad: feb-2017 a dic-2020)
# --------------------------------------------------------------------------- #
FECHA_INICIO = "2017-02-01"
CORTE_VALIDACION = os.getenv("MOBIPREDICT_CORTE_VALID", "2019-09-01")   # train < corte
CORTE_TEST = os.getenv("MOBIPREDICT_CORTE_TEST", "2019-12-01")          # valid < corte <= test
FIN_TEST = "2020-03-20"            # inicio de las restricciones por COVID-19 en Medellín
FIN_DRIFT = "2021-01-01"           # 2020-03-20..2020-12-31 = periodo de drift (pandemia)

# --------------------------------------------------------------------------- #
# Modelado
# --------------------------------------------------------------------------- #
SEED = int(os.getenv("MOBIPREDICT_SEED", "42"))
FAST = os.getenv("MOBIPREDICT_FAST", "0") == "1"

FREQ_MIN = 60                    # resolución de la serie (horaria, 6:00-20:00)
HORIZON_STEPS = 1                # predicción a 1 hora
LAGS = [1, 2, 3]                 # rezagos en horas
MIN_HISTORY = max(LAGS) + 1      # lecturas horarias consecutivas para inferir

PCT_VEL_LIBRE = 0.95             # velocidad de referencia = percentil 95 del tramo (solo entrenamiento)
COLLAPSE_SPEED_RATIO = float(os.getenv("MOBIPREDICT_UMBRAL_COLAPSO", "0.50"))  # colapso: v < 50 % de la referencia

N_TRIALS_OPTUNA = int(os.getenv("MOBIPREDICT_TRIALS", "8" if FAST else "25"))
N_ITER_RANDOM = int(os.getenv("MOBIPREDICT_RANDOM_ITER", "4" if FAST else "10"))
SEARCH_SAMPLE = 25_000 if FAST else 50_000     # filas para la búsqueda de hiperparámetros
FIT_MAX_ROWS = 80_000 if FAST else 200_000     # tope de filas para RF y MLP (los más recientes)

# --------------------------------------------------------------------------- #
# MLflow
# --------------------------------------------------------------------------- #
# Backend de archivos en mlruns/ (requisito del proyecto). MLflow 3.x lo exige habilitar.
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", MLRUNS_DIR.resolve().as_uri())
MLFLOW_EXPERIMENT = os.getenv("MLFLOW_EXPERIMENT", "MobiPredict")

# --------------------------------------------------------------------------- #
# Artefactos
# --------------------------------------------------------------------------- #
MODEL_SPEED_FILE = MODELS_DIR / "modelo_velocidad.joblib"
MODEL_COLLAPSE_FILE = MODELS_DIR / "modelo_colapso.joblib"
FEATURE_STORE_FILE = MODELS_DIR / "feature_store.joblib"
METADATA_FILE = MODELS_DIR / "metadata.json"
COMPARISON_FILE = MODELS_DIR / "model_comparison.csv"
DECISION_FILE = DOCS_DIR / "model_decision.md"


def ensure_dirs() -> None:
    for d in ALL_DIRS:
        d.mkdir(parents=True, exist_ok=True)

### 1.2 `data_acquisition.py` — descarga automática con reintentos y manifiesto (sin datos simulados)

In [ ]:
%%writefile src/mobipredict/data_acquisition.py
"""Adquisición automática de las fuentes oficiales.

Para cada fuente:
  1. Si el archivo ya está en data/raw/ (descargado antes o puesto a mano), se reutiliza.
  2. Si no, se descarga desde la URL oficial (con reintentos).
  3. Si la descarga falla y no hay archivo local, el pipeline se detiene con un mensaje
     claro. NO hay datos simulados: el proyecto trabaja solo con datos reales.

Cada ejecución deja data/raw/manifest.json con origen, URL, tamaño y hash de cada archivo.
"""
from __future__ import annotations

import hashlib
import json
import logging
import time
from datetime import datetime, timezone
from pathlib import Path

import httpx
import pandas as pd

from . import config

log = logging.getLogger(__name__)
UA = {"User-Agent": "MobiPredict/1.0 (proyecto academico de movilidad; contacto en README)"}


class FuenteNoDisponible(RuntimeError):
    pass


def _sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:16]


def _descargar(url: str, destino: Path, intentos: int = 3, **kw) -> None:
    tmp = destino.with_suffix(destino.suffix + ".part")
    ultimo = None
    for i in range(1, intentos + 1):
        try:
            with httpx.stream("GET", url, timeout=config.HTTP_TIMEOUT_S, follow_redirects=True,
                              headers=UA, **kw) as r:
                r.raise_for_status()
                with open(tmp, "wb") as f:
                    for chunk in r.iter_bytes(1 << 20):
                        f.write(chunk)
            tmp.replace(destino)
            return
        except Exception as exc:  # red, 4xx/5xx, corte de conexión
            ultimo = exc
            log.warning("Intento %d/%d fallido para %s: %s", i, intentos, url, exc)
            time.sleep(2 * i)
    tmp.unlink(missing_ok=True)
    raise FuenteNoDisponible(f"No se pudo descargar {url}: {ultimo}")


def _get_json(url: str, **params):
    r = httpx.get(url, params=params or None, timeout=config.HTTP_TIMEOUT_S, follow_redirects=True, headers=UA)
    r.raise_for_status()
    return r.json()


# --------------------------------------------------------------------------- #
# Fuentes
# --------------------------------------------------------------------------- #
def _archivo_simple(clave: str, forzar: bool) -> tuple[Path, str]:
    f = config.FUENTES[clave]
    destino = config.DATA_RAW / f["archivo"]
    if destino.exists() and destino.stat().st_size > 0 and not forzar:
        return destino, "local:data/raw"
    _descargar(f["url"], destino)
    return destino, f"descargado:{f['url']}"


def _eod_desde_referencia(destino: Path) -> None:
    """Respaldo: arma el JSON de la EOD 2025 con los agregados oficiales versionados en data/reference/eod2025/."""
    ref = config.DATA_REFERENCE / "eod2025"
    base = json.loads((ref / "indicadores_api.json").read_text(encoding="utf-8"))
    od = pd.read_csv(ref / "matriz_od_macrozonas.csv")
    base["indicadores"]["15"] = {
        "id_indicador": 15, "nombre": "Mapas",
        "matriz": [{"agrupa_mz_origen": int(o), "agrupa_mz_destino": int(d), "valor": float(v)}
                   for o, d, v in od[["origen", "destino", "viajes"]].itertuples(index=False)]}
    base["api"] = config.EOD_API
    destino.write_text(json.dumps(base, ensure_ascii=False), encoding="utf-8")


def _eod2025(forzar: bool) -> tuple[Path, str]:
    """Descarga todos los indicadores agregados de la EOD 2025; si la API no responde, usa la copia
    oficial versionada en data/reference/eod2025/ (extraída de la misma API el 2026-10-23)."""
    destino = config.DATA_RAW / config.FUENTES["eod2025"]["archivo"]
    if destino.exists() and not forzar:
        return destino, "local:data/raw"
    base = config.EOD_API
    try:
        ids = _get_json(f"{base}/metadata/indicadores")["ids_indicadores"]
        indicadores = {i: _get_json(f"{base}/indicadores/{i}/agregado") for i in ids}
        por_municipio = {m: _get_json(f"{base}/indicadores/1/agregado", municipio=m)["valor"]
                         for m in _get_json(f"{base}/metadata/municipios")["municipios"]}
    except Exception as exc:
        log.warning("API EOD 2025 no disponible (%s); se usa la copia de data/reference/eod2025/", exc)
        _eod_desde_referencia(destino)
        return destino, "referencia:data/reference/eod2025 (API oficial, extraído 2026-10-23)"
    payload = {"descargado_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "api": base, "indicadores": indicadores, "viajes_por_municipio_residencia": por_municipio}
    destino.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
    try:   # polígonos de macrozonas (opcional; el EDA usa los centroides de data/reference)
        _descargar(config.FUENTES["eod2025"]["ficha"].rstrip("/") + "/macrozonas_actualizado.geo.json",
                   config.DATA_RAW / "eod2025_macrozonas.geo.json", intentos=1)
    except FuenteNoDisponible as exc:
        log.warning("GeoJSON de macrozonas no descargado: %s", exc)
    return destino, f"descargado:{base}"


def _clima(forzar: bool, inicio: str, fin: str) -> tuple[Path, str]:
    destino = config.DATA_RAW / config.FUENTES["clima"]["archivo"]
    if destino.exists() and not forzar:
        return destino, "local:data/raw"
    manual = destino.with_suffix(".json")      # descargado con scripts/descargar_datos.ps1
    if manual.exists() and not forzar:
        data = json.loads(manual.read_text(encoding="utf-8"))
        h = data["hourly"]
        pd.DataFrame({"timestamp": h["time"], "lluvia_mm": h["precipitation"],
                      "temperatura_c": h["temperature_2m"]}).to_csv(destino, index=False)
        return destino, "local:data/raw (json Open-Meteo)"
    data = _get_json(config.FUENTES["clima"]["url"], latitude=config.CLIMA_LAT, longitude=config.CLIMA_LON,
                     start_date=inicio, end_date=fin, hourly="precipitation,temperature_2m",
                     timezone="America/Bogota")
    h = data["hourly"]
    pd.DataFrame({"timestamp": h["time"], "lluvia_mm": h["precipitation"],
                  "temperatura_c": h["temperature_2m"]}).to_csv(destino, index=False)
    return destino, "descargado:open-meteo"


def adquirir_datos(forzar: bool = False) -> dict:
    """Obtiene todas las fuentes reales y escribe el manifiesto. Lanza FuenteNoDisponible si falta alguna."""
    config.ensure_dirs()
    resultados, errores = {}, {}
    tareas = {
        "velocidad_gt": lambda: _archivo_simple("velocidad_gt", forzar),
        "aforos": lambda: _archivo_simple("aforos", forzar),
        "siniestros": lambda: _archivo_simple("siniestros", forzar),
        "eod2025": lambda: _eod2025(forzar),
        "clima": lambda: _clima(forzar, "2017-01-01", "2020-12-31"),
    }
    for clave, fn in tareas.items():
        try:
            resultados[clave] = fn()
            log.info("Fuente %s: %s", clave, resultados[clave][1])
        except Exception as exc:
            errores[clave] = str(exc)
            log.error("Fuente %s no disponible: %s", clave, exc)

    manifest = {
        "generado_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "fuentes": {
            k: {"titulo": config.FUENTES[k]["titulo"], "url": config.FUENTES[k]["url"],
                "ficha": config.FUENTES[k]["ficha"], "archivo": str(p.relative_to(config.ROOT)),
                "origen": o, "bytes": p.stat().st_size, "sha256_16": _sha256(p)}
            for k, (p, o) in resultados.items()
        },
        "errores": errores,
    }
    config.MANIFEST_FILE.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    obligatorias = {"velocidad_gt", "eod2025", "clima"}
    if obligatorias & set(errores):
        faltan = sorted(obligatorias & set(errores))
        raise FuenteNoDisponible(
            f"Faltan fuentes obligatorias: {faltan}. Descárguelas a data/raw/ con los nombres de "
            "config.FUENTES o revise la conexión (ver README, sección Datos).")
    return manifest

### 1.3 `fuentes.py` — validación y estandarización
Incluye las correcciones de calidad: fecha desde `FECHA_HORA` (`DÍA`/`MES` corridos en 2020), agregación horaria de lecturas cada 30 min y promedio de duplicados.

In [ ]:
%%writefile src/mobipredict/fuentes.py
"""Estandarización de las fuentes reales a tablas limpias en data/interim/.

Cada función lee el archivo crudo descargado en data/raw/, valida su estructura y
escribe una tabla con nombres de columna uniformes. El resto del proyecto (EDA,
features, API) solo lee estas tablas estandarizadas.
"""
from __future__ import annotations

import io
import json
import logging
import re
import unicodedata
import zipfile

import numpy as np
import pandas as pd

from . import config

log = logging.getLogger(__name__)


def _norm(s: str) -> str:
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")


def _slug_tramo(corredor: str, sentido: str) -> str:
    return f"{_norm(corredor).upper()}_{sentido.strip().upper()}"


# --------------------------------------------------------------------------- #
# Sensores: velocidad y tiempo de viaje por corredor (MEData / SIMM, Google Traffic)
# --------------------------------------------------------------------------- #
COLS_GT = ["id", "fecha_hora", "tv_s", "corredor_id", "dia", "mes", "anio", "hora", "fecha", "corredor",
           "flujo_tipo", "sentido", "inicio", "fin", "longitud_km", "tv_min", "velocidad_kmh", "tv_min_km",
           "diasem", "nombre_dia"]


def leer_velocidad_gt(ruta) -> pd.DataFrame:
    """Lee el CSV crudo de MEData y devuelve lecturas con timestamp confiable.

    Hallazgos de calidad (verificados sobre las 682.502 filas):
      * Las columnas DÍA y MES no son confiables en parte de 2020; FECHA_HORA (d/mm/aaaa H:MM)
        coincide con DIASEM en el 100 % de las filas, así que la fecha se toma de FECHA_HORA.
      * Desde 2020 hay lecturas cada 30 min; antes, cada hora. Se agregan al promedio horario.
      * ~9.700 claves (tramo, FECHA_HORA) están duplicadas; se promedian.
    """
    raw = pd.read_csv(ruta, encoding="utf-8", encoding_errors="replace", low_memory=False)
    if raw.shape[1] != len(COLS_GT):
        raise ValueError(f"Velocidad GT: se esperaban {len(COLS_GT)} columnas y llegaron {raw.shape[1]}")
    raw.columns = COLS_GT
    for c in ["velocidad_kmh", "tv_min", "longitud_km", "diasem"]:
        raw[c] = pd.to_numeric(raw[c], errors="coerce")
    raw["timestamp_raw"] = pd.to_datetime(raw["fecha_hora"].astype(str).str.strip(), format="%d/%m/%Y %H:%M",
                                          errors="coerce")
    raw = raw.dropna(subset=["timestamp_raw", "velocidad_kmh"])
    incoherentes = raw["timestamp_raw"].dt.dayofweek + 1 != raw["diasem"]
    if incoherentes.any():
        log.warning("Sensores: %d filas con FECHA_HORA incoherente con DIASEM; se descartan", incoherentes.sum())
        raw = raw[~incoherentes]
    raw["timestamp"] = raw["timestamp_raw"].dt.floor("h")
    return raw


def estandarizar_sensores(ruta=None, guardar: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Serie horaria de velocidad por tramo (corredor + sentido) y catálogo de tramos."""
    raw = leer_velocidad_gt(ruta or config.DATA_RAW / config.FUENTES["velocidad_gt"]["archivo"])
    raw["corredor"] = raw["corredor"].str.strip()
    raw["sentido"] = raw["sentido"].str.strip().str.upper()
    raw["tramo_id"] = [_slug_tramo(c, s) for c, s in zip(raw["corredor"], raw["sentido"])]

    # Valores físicamente imposibles se descartan (velocidad urbana entre 2 y 110 km/h)
    n0 = len(raw)
    raw = raw[(raw["velocidad_kmh"] >= 2) & (raw["velocidad_kmh"] <= 110)]
    log.info("Sensores: %d filas descartadas por velocidad fuera de rango", n0 - len(raw))

    sens = (raw.groupby(["tramo_id", "timestamp"], as_index=False)
            .agg(velocidad_kmh=("velocidad_kmh", "mean"), tiempo_viaje_min=("tv_min", "mean"))
            .sort_values(["tramo_id", "timestamp"]))

    moda = lambda s: s.mode().iat[0] if not s.mode().empty else s.iloc[0]  # noqa: E731
    tramos = (raw.groupby("tramo_id")
              .agg(corredor=("corredor", moda), sentido=("sentido", moda), inicio=("inicio", moda),
                   fin=("fin", moda), longitud_km=("longitud_km", "median"), flujo_tipo=("flujo_tipo", moda),
                   registros=("velocidad_kmh", "size"))
              .reset_index())
    tramos["flujo_continuo"] = (tramos["flujo_tipo"].str.lower().str.startswith("contin")).astype(int)

    # Ubicación (cámaras SIMM) y macrozona EOD 2025 más cercana
    cor = pd.read_csv(config.CORREDORES_FILE)
    tramos = tramos.merge(cor[["corredor", "lat", "lon"]], on="corredor", how="left")
    mz = pd.read_csv(config.MACROZONAS_FILE).dropna(subset=["lat", "lon"])
    d = ((tramos["lat"].to_numpy()[:, None] - mz["lat"].to_numpy()[None, :]) ** 2
         + (tramos["lon"].to_numpy()[:, None] - mz["lon"].to_numpy()[None, :]) ** 2)
    idx = np.nanargmin(np.where(np.isnan(d), np.inf, d), axis=1)
    tramos["id_macrozona"] = mz["id_macrozona"].to_numpy()[idx]
    tramos["macrozona"] = (mz["municipio"] + " - " + mz["macrozona"]).to_numpy()[idx]
    tramos.loc[tramos["lat"].isna(), ["id_macrozona", "macrozona"]] = [np.nan, None]

    if guardar:
        config.DATA_INTERIM.mkdir(parents=True, exist_ok=True)
        sens.to_parquet(config.SENSORES_STD, index=False)
        tramos.to_csv(config.TRAMOS_STD, index=False)
    return sens, tramos


# --------------------------------------------------------------------------- #
# Clima y festivos
# --------------------------------------------------------------------------- #
def estandarizar_clima() -> pd.DataFrame:
    c = pd.read_csv(config.DATA_RAW / config.FUENTES["clima"]["archivo"], parse_dates=["timestamp"])
    c["lluvia_mm"] = c["lluvia_mm"].fillna(0).clip(lower=0)
    c.to_parquet(config.CLIMA_STD, index=False)
    return c


def estandarizar_festivos(anios=range(2017, 2027)) -> pd.DataFrame:
    import holidays
    f = pd.DataFrame([(pd.Timestamp(d), n) for d, n in holidays.Colombia(years=list(anios)).items()],
                     columns=["fecha", "festivo"]).sort_values("fecha")
    f.to_csv(config.FESTIVOS_STD, index=False)
    return f


# --------------------------------------------------------------------------- #
# EOD 2025 (agregados oficiales del visor del AMVA)
# --------------------------------------------------------------------------- #
GRUPOS_HORA = {"16": "informal", "17": "privado", "18": "publico", "19": "no_motorizado"}


def cargar_eod_json() -> dict:
    return json.loads((config.DATA_RAW / config.FUENTES["eod2025"]["archivo"]).read_text(encoding="utf-8"))


def estandarizar_eod() -> tuple[pd.DataFrame, pd.DataFrame]:
    eod = cargar_eod_json()["indicadores"]
    horas = pd.DataFrame({"hora": range(24)})
    for i, nombre in GRUPOS_HORA.items():
        g = pd.DataFrame(eod[i]["grupos"]).rename(columns={"criterio": "hora", "valor": f"viajes_{nombre}"})
        g["hora"] = g["hora"].astype(int)
        horas = horas.merge(g, on="hora", how="left")
    horas = horas.fillna(0)
    cols = [f"viajes_{n}" for n in GRUPOS_HORA.values()]
    horas["viajes_total"] = horas[cols].sum(axis=1)
    for c in cols + ["viajes_total"]:
        horas[f"share_{c}"] = horas[c] / horas[c].sum()
    horas.to_csv(config.EOD_HORA_STD, index=False)

    od = pd.DataFrame(eod["15"]["matriz"]).rename(columns={"agrupa_mz_origen": "origen", "agrupa_mz_destino": "destino",
                                                          "valor": "viajes"})
    od[["origen", "destino"]] = od[["origen", "destino"]].astype(int)
    od.to_csv(config.EOD_OD_STD, index=False)
    return horas, od


def priors_macrozona(od: pd.DataFrame) -> pd.DataFrame:
    gen = od.groupby("origen")["viajes"].sum().rename("eod_viajes_generados_mz")
    atr = od.groupby("destino")["viajes"].sum().rename("eod_viajes_atraidos_mz")
    p = pd.concat([gen, atr], axis=1).fillna(0)
    p.index.name = "id_macrozona"
    return p.reset_index()


# --------------------------------------------------------------------------- #
# Siniestralidad (Mendeley Data, 2008-2025)
# --------------------------------------------------------------------------- #
def _hora(x) -> float:
    if pd.isna(x):
        return np.nan
    if hasattr(x, "hour"):
        return x.hour
    if isinstance(x, (int, float)):
        return int((x % 1) * 24) if x < 1 else (int(x) if x < 24 else np.nan)
    m = re.match(r"\s*(\d{1,2})", str(x))
    return int(m.group(1)) if m and int(m.group(1)) < 24 else np.nan


def estandarizar_siniestros() -> pd.DataFrame | None:
    z = config.DATA_RAW / config.FUENTES["siniestros"]["archivo"]
    if not z.exists():
        log.warning("Siniestros: archivo no disponible; se omite.")
        return None
    if config.SINIESTROS_STD.exists() and config.SINIESTROS_STD.stat().st_mtime > z.stat().st_mtime:
        return pd.read_parquet(config.SINIESTROS_STD)
    with zipfile.ZipFile(z) as zf:
        nombre = next(n for n in zf.namelist() if n.endswith("Fatal_Road_Traffic.xlsx"))
        log.info("Siniestros: leyendo %s (puede tardar unos minutos)", nombre)
        df = pd.read_excel(io.BytesIO(zf.read(nombre)), sheet_name="BD", engine="openpyxl")
    df.columns = [_norm(c) for c in df.columns]
    col = lambda *ops: next((c for c in df.columns if any(o == c or o in c for o in ops)), None)  # noqa: E731
    out = pd.DataFrame({
        "anio": pd.to_numeric(df[col("ano", "anio")], errors="coerce"),
        "fecha": pd.to_datetime(df[col("fecha_incidente")], errors="coerce"),
        "hora": df[col("hora_incidente")].map(_hora),
        "clase": df[col("clase_incidente")].astype(str).str.strip().str.upper(),
        "gravedad": df[col("gravedad_incidente")].astype(str).str.strip().str.upper(),
        "lat": pd.to_numeric(df[col("latitud")], errors="coerce"),
        "lon": pd.to_numeric(df[col("longitud")], errors="coerce"),
        "comuna": df[col("comuna_planeacion")].astype(str) if col("comuna_planeacion") else None,
    })
    out["fatal"] = out["gravedad"].str.startswith("MUERT").astype(int)
    out.to_parquet(config.SINIESTROS_STD, index=False)
    return out


# --------------------------------------------------------------------------- #
# Aforos: composición vehicular por hora (2017-2019)
# --------------------------------------------------------------------------- #
def estandarizar_aforos() -> pd.DataFrame | None:
    p = config.DATA_RAW / config.FUENTES["aforos"]["archivo"]
    if not p.exists():
        return None
    a = pd.read_csv(p, encoding="latin-1", low_memory=False)
    if a.shape[1] < 24:
        log.warning("Aforos: estructura inesperada; se omite.")
        return None
    a = a.iloc[:, [0, 7, 8, 9, 10, 11, 20, 23]]
    a.columns = ["nodo", "autos", "buses", "camiones", "motos", "bicicletas", "anio", "hora"]
    for c in a.columns[1:]:
        a[c] = pd.to_numeric(a[c], errors="coerce")
    a = a.dropna(subset=["hora", "anio"])
    comp = a.groupby(["anio", "hora"], as_index=False)[["autos", "buses", "camiones", "motos", "bicicletas"]].sum()
    comp["nodos"] = a.groupby(["anio", "hora"])["nodo"].nunique().to_numpy()
    comp.to_parquet(config.AFOROS_STD, index=False)
    return comp


def estandarizar_todo() -> dict:
    config.ensure_dirs()
    sens, tramos = estandarizar_sensores()
    clima = estandarizar_clima()
    fest = estandarizar_festivos()
    horas, od = estandarizar_eod()
    sin = estandarizar_siniestros()
    afo = estandarizar_aforos()
    resumen = {
        "sensores_filas": len(sens), "tramos": len(tramos),
        "periodo_sensores": [str(sens["timestamp"].min()), str(sens["timestamp"].max())],
        "clima_horas": len(clima), "festivos": len(fest),
        "eod_viajes_total": float(od["viajes"].sum()), "eod_pares_od": len(od),
        "siniestros_filas": None if sin is None else len(sin),
        "aforos_filas_agregadas": None if afo is None else len(afo),
    }
    log.info("Fuentes estandarizadas: %s", resumen)
    return resumen

### 2. `features.py` — features espaciotemporales y de la EOD, split temporal
Las mismas funciones construyen las features en entrenamiento y en la API.

In [ ]:
%%writefile src/mobipredict/features.py
"""Feature engineering espaciotemporal + demográfico y split temporal (datos reales).

Serie base: velocidad horaria (6:00-20:00) por tramo vial, MEData/SIMM 2017-2020.

Las MISMAS funciones construyen las features para entrenar (por lotes) y para la API
(las últimas lecturas de un tramo), así no hay training-serving skew.

"Feature store" local (models/feature_store.joblib):
  * tramos:  catálogo (corredor, sentido, longitud, tipo de flujo, ubicación, macrozona EOD)
             + velocidad de referencia (p95 del entrenamiento).
  * perfil:  velocidad media por tramo x tipo de día x hora (solo entrenamiento).
  * eod_hora: distribución horaria oficial de viajes EOD 2025 por grupo modal.
  * eod_mz:  viajes generados/atraídos por macrozona (matriz OD EOD 2025).
  * clima:   lluvia horaria (Open-Meteo); en la API se recibe con cada lectura.
  * festivos: calendario oficial de Colombia.
"""
from __future__ import annotations

import json
import logging

import joblib
import numpy as np
import pandas as pd

from . import config, fuentes

log = logging.getLogger(__name__)

TARGET_REG = "y_velocidad_1h"
TARGET_CLF = "y_colapso_1h"
PASO = pd.Timedelta(minutes=config.FREQ_MIN)


# --------------------------------------------------------------------------- #
# Tablas de referencia
# --------------------------------------------------------------------------- #
def _tipo_dia(ts: pd.Series, festivo: pd.Series) -> np.ndarray:
    dow = ts.dt.dayofweek
    return np.select([(festivo == 1) | (dow == 6), dow == 5], ["domingo_festivo", "sabado"], "laboral")


def perfil_historico(d: pd.DataFrame) -> pd.DataFrame:
    return (d.groupby(["tramo_id", "tipo_dia", "hora"])["velocidad_kmh"].mean()
            .rename("perfil_velocidad").reset_index())


def construir_store(sens: pd.DataFrame, tramos: pd.DataFrame, festivos: pd.DataFrame,
                    eod_hora: pd.DataFrame, od: pd.DataFrame, clima: pd.DataFrame, corte_train) -> dict:
    """Calcula las tablas de referencia usando SOLO datos anteriores al corte de entrenamiento."""
    train = sens[sens["timestamp"] < corte_train]
    vref = (train.groupby("tramo_id")["velocidad_kmh"].quantile(config.PCT_VEL_LIBRE)
            .rename("vel_referencia_kmh").reset_index())
    tr = tramos.merge(vref, on="tramo_id", how="left")
    base = _base(train, {"festivos": festivos})
    return {
        "tramos": tr,
        "perfil": perfil_historico(base),
        "eod_hora": eod_hora,
        "eod_mz": fuentes.priors_macrozona(od),
        "clima": clima[["timestamp", "lluvia_mm"]],
        "festivos": set(pd.to_datetime(festivos["fecha"]).dt.normalize()),
    }


# --------------------------------------------------------------------------- #
# Construcción de features (compartida entrenamiento / inferencia)
# --------------------------------------------------------------------------- #
def _base(d: pd.DataFrame, store: dict) -> pd.DataFrame:
    d = d.copy()
    fest = store["festivos"]
    if not isinstance(fest, set):
        fest = set(pd.to_datetime(fest["fecha"]).dt.normalize())
    d["es_festivo"] = d["timestamp"].dt.normalize().isin(fest).astype(int)
    d["tipo_dia"] = _tipo_dia(d["timestamp"], d["es_festivo"])
    d["hora"] = d["timestamp"].dt.hour
    return d


def _lag_seguro(d: pd.DataFrame, col: str, k: int) -> pd.Series:
    """Rezago de k horas solo si la lectura previa está exactamente k horas antes (sin cruzar noches)."""
    g = d.groupby("tramo_id", sort=False)
    val, ts = g[col].shift(k), g["timestamp"].shift(k)
    return val.where(d["timestamp"] - ts == k * PASO)


def construir_features(d: pd.DataFrame, store: dict) -> pd.DataFrame:
    """`d` limpio y ordenado por tramo y tiempo: columnas timestamp, tramo_id, velocidad_kmh y,
    si existe, lluvia_mm (si no, se toma de la tabla de clima)."""
    d = _base(d.sort_values(["tramo_id", "timestamp"]), store)
    if "lluvia_mm" not in d.columns:
        d = d.merge(store["clima"], on="timestamp", how="left")
    d["lluvia_mm"] = d["lluvia_mm"].fillna(0)

    tr = store["tramos"][["tramo_id", "longitud_km", "flujo_continuo", "lat", "lon", "id_macrozona",
                          "vel_referencia_kmh"]]
    d = d.merge(tr, on="tramo_id", how="left")
    d["ratio_vel"] = d["velocidad_kmh"] / d["vel_referencia_kmh"]

    h = d["hora"]
    d["hora_sin"], d["hora_cos"] = np.sin(2 * np.pi * h / 24), np.cos(2 * np.pi * h / 24)
    dow = d["timestamp"].dt.dayofweek
    d["dow_sin"], d["dow_cos"] = np.sin(2 * np.pi * dow / 7), np.cos(2 * np.pi * dow / 7)
    d["es_fin_semana"] = (dow >= 5).astype(int)
    d["es_pico"] = h.isin([6, 7, 8, 17, 18, 19]).astype(int)
    d["mes_sin"] = np.sin(2 * np.pi * d["timestamp"].dt.month / 12)
    d["mes_cos"] = np.cos(2 * np.pi * d["timestamp"].dt.month / 12)

    for col in ["velocidad_kmh", "ratio_vel", "lluvia_mm"]:
        for k in config.LAGS:
            d[f"{col}_lag{k}"] = _lag_seguro(d, col, k)
    d["velocidad_media_3h"] = d[["velocidad_kmh", "velocidad_kmh_lag1", "velocidad_kmh_lag2"]].mean(axis=1)
    d["velocidad_delta1"] = d["velocidad_kmh"] - d["velocidad_kmh_lag1"]
    d["ratio_vel_delta1"] = d["ratio_vel"] - d["ratio_vel_lag1"]
    d["lluvia_3h"] = d[["lluvia_mm", "lluvia_mm_lag1", "lluvia_mm_lag2"]].sum(axis=1, min_count=1)
    d["n_lecturas_previas"] = d[[f"velocidad_kmh_lag{k}" for k in config.LAGS]].notna().sum(axis=1)

    # Perfil histórico y priors EOD en la hora OBJETIVO (t + 1 h)
    t_obj = d["timestamp"] + config.HORIZON_STEPS * PASO
    d["hora_obj"] = t_obj.dt.hour
    perfil = store["perfil"].rename(columns={"hora": "hora_obj"})
    d = d.merge(perfil, on=["tramo_id", "tipo_dia", "hora_obj"], how="left")
    d["desvio_perfil"] = d["velocidad_kmh"] - d["perfil_velocidad"]

    eh = store["eod_hora"][["hora", "share_viajes_privado", "share_viajes_publico", "share_viajes_total"]]
    d = d.merge(eh.rename(columns={"hora": "hora_obj", "share_viajes_privado": "eod_share_privado_hora",
                                   "share_viajes_publico": "eod_share_publico_hora",
                                   "share_viajes_total": "eod_share_total_hora"}), on="hora_obj", how="left")
    d = d.merge(store["eod_mz"], on="id_macrozona", how="left")
    return d


FEATURES_NUM = (
    ["hora_sin", "hora_cos", "dow_sin", "dow_cos", "mes_sin", "mes_cos", "es_fin_semana", "es_pico", "es_festivo",
     "velocidad_kmh", "ratio_vel", "velocidad_media_3h", "velocidad_delta1", "ratio_vel_delta1",
     "lluvia_mm", "lluvia_3h", "n_lecturas_previas",
     "longitud_km", "flujo_continuo", "lat", "lon", "vel_referencia_kmh",
     "perfil_velocidad", "desvio_perfil",
     "eod_share_privado_hora", "eod_share_publico_hora", "eod_share_total_hora",
     "eod_viajes_generados_mz", "eod_viajes_atraidos_mz"]
    + [f"{c}_lag{k}" for c in ["velocidad_kmh", "ratio_vel"] for k in config.LAGS]
)


def agregar_targets(d: pd.DataFrame) -> pd.DataFrame:
    g = d.groupby("tramo_id", sort=False)
    v_fut, ts_fut = g["velocidad_kmh"].shift(-config.HORIZON_STEPS), g["timestamp"].shift(-config.HORIZON_STEPS)
    ok = (ts_fut - d["timestamp"]) == config.HORIZON_STEPS * PASO
    d[TARGET_REG] = v_fut.where(ok)
    d[TARGET_CLF] = np.where(d[TARGET_REG].isna(), np.nan,
                             (d[TARGET_REG] / d["vel_referencia_kmh"] < config.COLLAPSE_SPEED_RATIO).astype(float))
    return d


# --------------------------------------------------------------------------- #
# Pipeline de features + split temporal
# --------------------------------------------------------------------------- #
def preparar_dataset() -> dict:
    config.ensure_dirs()
    sens = pd.read_parquet(config.SENSORES_STD)
    tramos = pd.read_csv(config.TRAMOS_STD)
    festivos = pd.read_csv(config.FESTIVOS_STD, parse_dates=["fecha"])
    eod_hora = pd.read_csv(config.EOD_HORA_STD)
    od = pd.read_csv(config.EOD_OD_STD)
    clima = pd.read_parquet(config.CLIMA_STD)

    c_valid, c_test = pd.Timestamp(config.CORTE_VALIDACION), pd.Timestamp(config.CORTE_TEST)
    fin_test, fin_drift = pd.Timestamp(config.FIN_TEST), pd.Timestamp(config.FIN_DRIFT)
    sens = sens[sens["timestamp"] >= pd.Timestamp(config.FECHA_INICIO)]

    store = construir_store(sens, tramos, festivos, eod_hora, od, clima, corte_train=c_valid)
    d = agregar_targets(construir_features(sens, store))
    d = d.dropna(subset=[TARGET_REG, TARGET_CLF])
    d[TARGET_CLF] = d[TARGET_CLF].astype(int)

    # Train < corte_valid <= valid < corte_test <= test < 2020-03-20 <= drift (pandemia)
    d["particion"] = np.select(
        [d["timestamp"] < c_valid, d["timestamp"] < c_test, d["timestamp"] < fin_test, d["timestamp"] < fin_drift],
        ["train", "valid", "test", "drift"], "fuera")
    # El objetivo de la última hora de cada partición cae en la siguiente: se descarta (sin fuga)
    t_obj = d["timestamp"] + config.HORIZON_STEPS * PASO
    frontera = (((d["timestamp"] < c_valid) & (t_obj >= c_valid)) | ((d["timestamp"] < c_test) & (t_obj >= c_test))
                | ((d["timestamp"] < fin_test) & (t_obj >= fin_test)))
    d = d[~frontera & (d["particion"] != "fuera")].reset_index(drop=True)

    cols = ["timestamp", "tramo_id", "particion"] + FEATURES_NUM + [TARGET_REG, TARGET_CLF]
    d = d[cols]
    for part in ["train", "valid", "test", "drift"]:
        d[d["particion"] == part].to_parquet(config.DATA_PROCESSED / f"{part}.parquet", index=False)

    joblib.dump(store, config.FEATURE_STORE_FILE)
    spec = {
        "fuente_sensores": config.FUENTES["velocidad_gt"]["titulo"],
        "features": FEATURES_NUM, "target_regresion": TARGET_REG, "target_clasificacion": TARGET_CLF,
        "horizonte_min": config.FREQ_MIN * config.HORIZON_STEPS, "lags_h": config.LAGS,
        "umbral_colapso_ratio_vel": config.COLLAPSE_SPEED_RATIO, "percentil_vel_referencia": config.PCT_VEL_LIBRE,
        "corte_validacion": str(c_valid.date()), "corte_test": str(c_test.date()), "fin_test": str(fin_test.date()),
        "ventana_drift": [str(fin_test.date()), "2020-12-31"],
        "filas": d["particion"].value_counts().to_dict(),
        "tasa_colapso": d.groupby("particion")[TARGET_CLF].mean().round(4).to_dict(),
        "tramos": int(d["tramo_id"].nunique()),
    }
    (config.DATA_PROCESSED / "feature_spec.json").write_text(json.dumps(spec, indent=2, ensure_ascii=False),
                                                             encoding="utf-8")
    log.info("Dataset listo: %s | tasa de colapso: %s", spec["filas"], spec["tasa_colapso"])
    return spec


def cargar_particiones() -> dict[str, pd.DataFrame]:
    return {p: pd.read_parquet(config.DATA_PROCESSED / f"{p}.parquet") for p in ["train", "valid", "test", "drift"]}


# --------------------------------------------------------------------------- #
# Inferencia
# --------------------------------------------------------------------------- #
def features_inferencia(tramo_id: str, observaciones: list[dict], store: dict) -> pd.DataFrame:
    """Fila de features para la última de las lecturas horarias recibidas de un tramo."""
    if tramo_id not in set(store["tramos"]["tramo_id"]):
        raise ValueError(f"tramo_id desconocido: {tramo_id}")
    if len(observaciones) < config.MIN_HISTORY:
        raise ValueError(f"Se requieren al menos {config.MIN_HISTORY} lecturas horarias; llegaron {len(observaciones)}")
    h = pd.DataFrame(observaciones)
    h["timestamp"] = pd.to_datetime(h["timestamp"]).dt.tz_localize(None).dt.floor("h")
    h["tramo_id"] = tramo_id
    if "lluvia_mm" not in h.columns:
        h["lluvia_mm"] = 0.0
    h = h.drop_duplicates("timestamp", keep="last").sort_values("timestamp")
    return construir_features(h, store).iloc[[-1]][FEATURES_NUM]

### 3. `eda.py` — EDA reproducible con datos reales (9 figuras + reportes)

In [ ]:
%%writefile src/mobipredict/eda.py
"""EDA reproducible sobre las fuentes reales: figuras en eda/figures/, reportes en eda/reports/.

Cubre: congestión crónica por corredor y hora (SIMM 2017-2020), los 6,49 M de viajes
diarios de la EOD 2025 y su reparto modal y horario, la densidad origen-destino por
macrozona, la composición vehicular (aforos con ~40 % de motos), la siniestralidad
2008-2025 (276 muertes en 2025 según el Observatorio), el efecto de la lluvia y el
cambio de régimen de la pandemia.
"""
from __future__ import annotations

import json
import logging

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from matplotlib.colors import LinearSegmentedColormap  # noqa: E402

from . import config  # noqa: E402
from .fuentes import cargar_eod_json  # noqa: E402

log = logging.getLogger(__name__)

AZUL, NARANJA, AQUA, AMARILLO = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
CRITICO = "#e34948"
TINTA, TINTA_2, GRILLA, FONDO, NEUTRO = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb", "#b9b8b2"
SECUENCIAL = LinearSegmentedColormap.from_list("mobi_seq", ["#eef4fc", "#9cc3ee", AZUL, "#0d3f7a"])
FUENTE_SENSORES = "Fuente: MEData — Velocidad y tiempo de viaje GT (SIMM, Alcaldía de Medellín)"
FUENTE_EOD = "Fuente: EOD 2025, visor oficial del AMVA (eod2025.metropol.gov.co)"


def _estilo() -> None:
    plt.rcParams.update({
        "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
        "axes.edgecolor": GRILLA, "axes.labelcolor": TINTA_2, "axes.titlecolor": TINTA,
        "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
        "axes.labelsize": 10, "xtick.color": TINTA_2, "ytick.color": TINTA_2,
        "xtick.labelsize": 9, "ytick.labelsize": 9, "axes.grid": True, "grid.color": GRILLA,
        "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False, "legend.fontsize": 9, "font.size": 10, "figure.dpi": 110,
    })


def _fuente(fig, texto: str) -> None:
    fig.text(0.01, 0.005, texto, fontsize=7.5, color=TINTA_2, ha="left", va="bottom")


def _guardar(fig, nombre: str) -> str:
    ruta = config.EDA_FIGURES / nombre
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    fig.savefig(ruta, dpi=130, bbox_inches="tight")
    plt.close(fig)
    return str(ruta.relative_to(config.ROOT))


def _miles(x: float) -> str:
    return f"{x:,.0f}".replace(",", ".")


# --------------------------------------------------------------------------- #
# Carga
# --------------------------------------------------------------------------- #
def _datos() -> dict:
    sens = pd.read_parquet(config.SENSORES_STD)
    tramos = pd.read_csv(config.TRAMOS_STD)
    pre = sens[sens["timestamp"] < pd.Timestamp(config.FIN_TEST)]
    vref = pre.groupby("tramo_id")["velocidad_kmh"].quantile(config.PCT_VEL_LIBRE).rename("vref")
    sens = sens.join(vref, on="tramo_id")
    sens["indice_congestion"] = (1 - sens["velocidad_kmh"] / sens["vref"]).clip(lower=0)
    sens["colapso"] = sens["velocidad_kmh"] / sens["vref"] < config.COLLAPSE_SPEED_RATIO
    nombre = tramos.set_index("tramo_id").apply(lambda r: f"{r['corredor']} ({r['sentido']})", axis=1)
    sens["tramo"] = sens["tramo_id"].map(nombre)
    return {
        "sens": sens, "tramos": tramos,
        "clima": pd.read_parquet(config.CLIMA_STD),
        "eod": cargar_eod_json(),
        "eod_hora": pd.read_csv(config.EOD_HORA_STD),
        "od": pd.read_csv(config.EOD_OD_STD),
        "mz": pd.read_csv(config.MACROZONAS_FILE),
        "siniestros": pd.read_parquet(config.SINIESTROS_STD) if config.SINIESTROS_STD.exists() else None,
        "aforos": pd.read_parquet(config.AFOROS_STD) if config.AFOROS_STD.exists() else None,
        "festivos": set(pd.read_csv(config.FESTIVOS_STD, parse_dates=["fecha"])["fecha"].dt.normalize()),
    }


def _laborable_pre(s: pd.DataFrame, festivos: set) -> pd.DataFrame:
    return s[(s["timestamp"] < pd.Timestamp(config.FIN_TEST)) & (s["timestamp"].dt.dayofweek < 5)
             & ~s["timestamp"].dt.normalize().isin(festivos)]


# --------------------------------------------------------------------------- #
# Figuras
# --------------------------------------------------------------------------- #
def fig_congestion(D) -> tuple[str, pd.DataFrame]:
    s = _laborable_pre(D["sens"], D["festivos"])
    piv = s.pivot_table(index="tramo", columns=s["timestamp"].dt.hour, values="indice_congestion", aggfunc="mean")
    piv = piv.loc[piv.mean(axis=1).sort_values(ascending=False).index]
    fig, ax = plt.subplots(figsize=(10.5, 7.2))
    im = ax.imshow(piv.values, aspect="auto", cmap=SECUENCIAL, vmin=0, vmax=0.6)
    ax.set_xticks(range(piv.shape[1]), [f"{h:02d}h" for h in piv.columns])
    ax.set_yticks(range(len(piv)), piv.index, fontsize=8.5)
    ax.grid(False)
    ax.set_title("Congestión crónica: picos de 7 h y 18 h, peores en vías interrumpidas")
    ax.set_xlabel("Hora (días laborables, feb-2017 a mar-2020)")
    cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.01)
    cb.set_label("Índice de congestión = 1 − vel / vel de referencia (p95)", color=TINTA_2)
    cb.outline.set_visible(False)
    _fuente(fig, FUENTE_SENSORES)
    return _guardar(fig, "01_congestion_cronica_heatmap.png"), piv


def fig_eod_horas(D) -> str:
    h = D["eod_hora"]
    fig, ax = plt.subplots(figsize=(10, 4.4))
    series = [("viajes_publico", "Transporte público", AZUL), ("viajes_no_motorizado", "No motorizado", NARANJA),
              ("viajes_privado", "Privado (auto, moto)", AQUA), ("viajes_informal", "Informal", AMARILLO)]
    for col, nom, c in series:
        ax.plot(h["hora"], h[col] / 1e3, color=c, lw=2, marker="o", ms=4, label=nom)
    ax.set_xticks(range(0, 24, 2))
    total = D["eod"]["indicadores"]["1"]["valor"]
    ax.set_title(f"{_miles(total)} viajes en un día hábil (EOD 2025): pico de salida a las 6 h; a pie, también a las 12 h")
    ax.set_xlabel("Hora de inicio del viaje")
    ax.set_ylabel("Miles de viajes")
    ax.legend(loc="upper right", ncols=2)
    _fuente(fig, FUENTE_EOD)
    return _guardar(fig, "02_eod2025_viajes_por_hora.png")


def fig_eod_modos(D) -> tuple[str, dict]:
    ind = D["eod"]["indicadores"]
    m = pd.DataFrame(ind["27"]["grupos"]).set_index("criterio")["valor"].sort_values() * 100
    col = {"Motocicleta": NARANJA, "Masivo": AZUL, "TPC": AZUL}
    fig, ax = plt.subplots(figsize=(9.5, 4.6))
    ax.barh(m.index, m.values, color=[col.get(k, NEUTRO) for k in m.index], height=0.68)
    for y, v in enumerate(m.values):
        ax.text(v + 0.4, y, f"{v:.1f} %", va="center", fontsize=9, color=TINTA)
    moto, auto = ind["13"]["valor"], ind["12"]["valor"]
    ax.set_title("Reparto modal EOD 2025: a pie lidera y la moto ya supera al automóvil")
    ax.set_xlabel("% de viajes (modo principal)")
    ax.grid(axis="y", visible=False)
    ax.text(0.98, 0.06, f"Motorización: {moto:.1f} motos y {auto:.1f} autos por 1.000 habitantes",
            transform=ax.transAxes, ha="right", fontsize=9, color=TINTA_2)
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=NARANJA), plt.Rectangle((0, 0), 1, 1, color=AZUL),
                       plt.Rectangle((0, 0), 1, 1, color=NEUTRO)],
              labels=["Moto", "Transporte público (masivo y TPC)", "Otros"], loc="lower right",
              bbox_to_anchor=(1, 0.12))
    _fuente(fig, FUENTE_EOD + ". TPC = transporte público colectivo")
    return _guardar(fig, "03_eod2025_reparto_modal.png"), m.round(2).to_dict()


def fig_densidad_od(D) -> tuple[str, pd.DataFrame]:
    od, mz, tr = D["od"], D["mz"].dropna(subset=["lat"]), D["tramos"]
    atr = od.groupby("destino")["viajes"].sum().rename("atraidos")
    gen = od.groupby("origen")["viajes"].sum().rename("generados")
    z = mz.join(atr, on="id_macrozona").join(gen, on="id_macrozona").fillna(0)
    z = z[z["lon"] > -75.70]   # Barbosa y Girardota quedan fuera del recorte del valle central
    fig, ax = plt.subplots(figsize=(8.2, 8.6))
    sc = ax.scatter(z["lon"], z["lat"], s=z["atraidos"] / 900, c=z["atraidos"] / 1e3, cmap=SECUENCIAL,
                    edgecolors=FONDO, linewidths=1.5, alpha=0.95, vmin=0)
    top = z.nlargest(6, "atraidos").sort_values("lat")
    for k, (_, r) in enumerate(top.iterrows()):
        dx = 14 if k % 2 == 0 else -14
        ax.annotate(f"{r['macrozona']}\n{_miles(r['atraidos'])}", (r["lon"], r["lat"]), xytext=(dx, 0),
                    textcoords="offset points", fontsize=8, color=TINTA, va="center",
                    ha="left" if dx > 0 else "right")
    ax.scatter(tr["lon"], tr["lat"], marker="s", s=28, color=NARANJA, label="Corredores con sensor (SIMM)", zorder=3)
    ax.set_xlim(-75.70, -75.48)
    ax.set_ylim(6.03, 6.38)
    ax.set_aspect("equal")
    ax.set_title("Densidad de destinos: el centro y El Poblado atraen más viajes")
    ax.set_xlabel("Longitud")
    ax.set_ylabel("Latitud")
    cb = fig.colorbar(sc, ax=ax, fraction=0.035, pad=0.02)
    cb.set_label("Miles de viajes atraídos por día hábil", color=TINTA_2)
    cb.outline.set_visible(False)
    ax.legend(loc="lower right")
    _fuente(fig, FUENTE_EOD + " (matriz OD por macrozona); ubicación de cámaras: MEData/SIMM")
    return _guardar(fig, "04_eod2025_densidad_destinos.png"), z.sort_values("atraidos", ascending=False)


def fig_siniestralidad(D) -> tuple[str | None, dict]:
    s = D["siniestros"]
    if s is None:
        return None, {}
    s = s[(s["anio"] >= 2008) & (s["anio"] <= 2025)]
    anual = s.groupby("anio").agg(incidentes=("fatal", "size"), fatales=("fatal", "sum"))
    hora = s.dropna(subset=["hora"]).groupby("hora")["fatal"].agg(["size", "sum"])
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4), gridspec_kw={"width_ratios": [1.3, 1]})
    ax = axes[0]
    colores = [NEUTRO if a == 2025 else AZUL for a in anual.index]
    ax.bar(anual.index, anual["fatales"], color=colores, width=0.75)
    ax.set_title("Incidentes con al menos un muerto, por año")
    ref = config.REFERENCIAS["muertes_viales_medellin_2025"]
    ax.text(0.02, 0.97, f"2025 (gris): solo ene-ago en la base.\nCierre oficial 2025: {ref['valor']} personas "
            f"fallecidas\n({ref['detalle']}).", transform=ax.transAxes, va="top", fontsize=8.5, color=TINTA_2)
    ax.set_ylim(0, anual["fatales"].max() * 1.35)
    ax.set_xticks(anual.index[::2])
    ax2 = axes[1]
    ax2.bar(hora.index, hora["size"] / 1e3, color=AZUL, width=0.8)
    ax2.set_title("Incidentes por hora del día (2008-2025)")
    ax2.set_xlabel("Hora")
    ax2.set_ylabel("Miles de incidentes")
    ax2.set_xticks(range(0, 24, 3))
    fig.suptitle("Siniestralidad vial en Medellín", x=0.01, ha="left", fontsize=13, fontweight="bold", color=TINTA)
    _fuente(fig, "Fuente: Correa Álvarez et al. (2026), Mendeley Data doi:10.17632/r6g5dfnpgh.1 (Secretaría de "
                 "Movilidad); cierre 2025: Observatorio de Movilidad vía El Colombiano")
    return _guardar(fig, "05_siniestralidad_vial.png"), {
        "incidentes_2008_2025": int(len(s)), "incidentes_fatales": int(s["fatal"].sum()),
        "fatales_por_anio": {int(k): int(v) for k, v in anual["fatales"].items()},
        "hora_con_mas_incidentes": int(hora["size"].idxmax())}


def fig_colapsos(D) -> tuple[str, pd.Series]:
    s = _laborable_pre(D["sens"], D["festivos"])
    tasa = (s.groupby("tramo")["colapso"].mean() * 100).sort_values()
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.barh(tasa.index, tasa.values, color=AZUL, height=0.7)
    for y, v in enumerate(tasa.values):
        ax.text(v + 0.2, y, f"{v:.1f} %", va="center", fontsize=8.5, color=TINTA)
    ax.set_title(f"Horas en colapso por tramo (vel < {config.COLLAPSE_SPEED_RATIO:.0%} de la de referencia)")
    ax.set_xlabel("% de horas laborables 6-20 h, pre-pandemia")
    ax.tick_params(axis="y", labelsize=8.5)
    ax.grid(axis="y", visible=False)
    _fuente(fig, FUENTE_SENSORES)
    return _guardar(fig, "06_horas_en_colapso_por_tramo.png"), tasa


def fig_pandemia(D) -> str:
    s = D["sens"][(D["sens"]["timestamp"] >= "2019-01-01")]
    s = s[s["timestamp"].dt.hour.isin([7, 8, 17, 18])]
    dia = s.groupby(s["timestamp"].dt.normalize())["indice_congestion"].mean()
    sem = dia.rolling(7, min_periods=3).mean()
    fig, ax = plt.subplots(figsize=(11, 4.2))
    ax.plot(sem.index, sem.values, color=AZUL, lw=1.8)
    ax.axvline(pd.Timestamp(config.FIN_TEST), color=CRITICO, lw=1.2, ls="--")
    ax.annotate("20-mar-2020: inicia el aislamiento\n(comienza la ventana de drift)",
                (pd.Timestamp(config.FIN_TEST), sem.max() * 0.95), xytext=(8, 0), textcoords="offset points",
                fontsize=8.5, color=TINTA)
    ax.set_title("La pandemia cambió el régimen: la congestión en hora pico se desplomó")
    ax.set_ylabel("Índice de congestión en pico (media 7 días)")
    ax.set_ylim(0, None)
    _fuente(fig, FUENTE_SENSORES)
    return _guardar(fig, "07_drift_pandemia.png")


def fig_lluvia(D) -> tuple[str, dict]:
    s = _laborable_pre(D["sens"], D["festivos"]).merge(D["clima"][["timestamp", "lluvia_mm"]], on="timestamp",
                                                         how="left")
    s = s[s["timestamp"].dt.hour.isin([7, 8, 17, 18, 19])]
    cortes = pd.cut(s["lluvia_mm"].fillna(0), [-0.01, 0.0, 1, 3, 1000], labels=["Sin lluvia", "0-1 mm", "1-3 mm", "> 3 mm"])
    r = s.groupby(cortes, observed=True).agg(indice=("indice_congestion", "mean"), colapso=("colapso", "mean"),
                                             horas=("colapso", "size"))
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.bar(r.index.astype(str), r["colapso"] * 100, color=[NEUTRO, "#9cc3ee", AZUL, "#0d3f7a"], width=0.6)
    for i, (v, n) in enumerate(zip(r["colapso"] * 100, r["horas"])):
        ax.text(i, v + 0.3, f"{v:.1f} %\n(n={_miles(n)})", ha="center", fontsize=8.5, color=TINTA)
    ax.set_title("Con lluvia fuerte, el colapso en hora pico es más frecuente")
    ax.set_ylabel("% de horas pico en colapso")
    ax.set_ylim(0, r["colapso"].max() * 130)
    ax.grid(axis="x", visible=False)
    _fuente(fig, FUENTE_SENSORES + "; lluvia: Open-Meteo (ERA5)")
    return _guardar(fig, "08_lluvia_vs_colapso.png"), (r["colapso"] * 100).round(2).to_dict()


def fig_aforos(D) -> tuple[str | None, dict]:
    a = D["aforos"]
    if a is None:
        return None, {}
    g = a.groupby("hora")[["autos", "buses", "camiones", "motos", "bicicletas"]].sum()
    pct = g.div(g.sum(axis=1), axis=0) * 100
    pct = pct[(pct.index >= 5) & (pct.index <= 21)]
    fig, ax = plt.subplots(figsize=(10, 4.2))
    ax.plot(pct.index, pct["motos"], color=NARANJA, lw=2.2, marker="o", ms=4, label="Motos")
    ax.plot(pct.index, pct["autos"], color=AZUL, lw=2.2, marker="o", ms=4, label="Autos")
    ax.plot(pct.index, pct["buses"] + pct["camiones"], color=NEUTRO, lw=2, label="Buses y camiones")
    tot = a[["autos", "buses", "camiones", "motos", "bicicletas"]].sum()
    ax.set_title(f"Composición vehicular aforada: {tot['motos'] / tot.sum() * 100:.0f} % son motos")
    ax.set_xlabel("Hora")
    ax.set_ylabel("% de vehículos aforados")
    ax.set_ylim(0, 65)
    ax.legend(loc="upper right")
    _fuente(fig, f"Fuente: MEData — Aforos vehiculares {int(a['anio'].min())}-{int(a['anio'].max())}, "
                 f"{int(a['nodos'].max())} intersecciones (Secretaría de Movilidad)")
    return _guardar(fig, "09_composicion_vehicular_aforos.png"), {
        "pct_motos": round(float(tot["motos"] / tot.sum() * 100), 1),
        "pct_autos": round(float(tot["autos"] / tot.sum() * 100), 1)}


# --------------------------------------------------------------------------- #
# Reporte
# --------------------------------------------------------------------------- #
def ejecutar_eda() -> dict:
    config.ensure_dirs()
    _estilo()
    D = _datos()
    s, ind = D["sens"], D["eod"]["indicadores"]

    figuras = {}
    figuras["congestion"], piv = fig_congestion(D)
    figuras["eod_horas"] = fig_eod_horas(D)
    figuras["eod_modos"], modos = fig_eod_modos(D)
    figuras["densidad_od"], zonas = fig_densidad_od(D)
    f_sin, sin = fig_siniestralidad(D)
    if f_sin:
        figuras["siniestralidad"] = f_sin
    figuras["colapsos"], tasa = fig_colapsos(D)
    figuras["pandemia"] = fig_pandemia(D)
    figuras["lluvia"], lluvia = fig_lluvia(D)
    f_af, afo = fig_aforos(D)
    if f_af:
        figuras["aforos"] = f_af

    # Calidad de la serie de sensores
    dias = s["timestamp"].dt.normalize().nunique()
    cobertura = (s.groupby("tramo_id").size() / (dias * 15) * 100).round(1)
    calidad = pd.DataFrame({"lecturas": s.groupby("tramo_id").size(), "cobertura_pct": cobertura,
                            "vel_media_kmh": s.groupby("tramo_id")["velocidad_kmh"].mean().round(1),
                            "vel_referencia_p95": s.groupby("tramo_id")["vref"].first().round(1)})
    calidad.to_csv(config.EDA_REPORTS / "calidad_sensores_por_tramo.csv")
    zonas[["id_macrozona", "municipio", "macrozona", "atraidos", "generados"]].round(0).to_csv(
        config.EDA_REPORTS / "eod2025_viajes_por_macrozona.csv", index=False)

    resumen = {
        "sensores": {"lecturas": int(len(s)), "tramos": int(s["tramo_id"].nunique()), "dias": int(dias),
                     "periodo": [str(s["timestamp"].min()), str(s["timestamp"].max())],
                     "cobertura_media_pct": float(cobertura.mean().round(1)),
                     "tramo_mas_congestionado": str(piv.mean(axis=1).idxmax()),
                     "hora_pico": int(piv.mean(axis=0).idxmax()),
                     "pct_horas_colapso_pre_pandemia": round(float(tasa.mean()), 2),
                     "tramo_mas_colapsos": str(tasa.idxmax())},
        "eod2025": {"viajes_dia_habil": round(ind["1"]["valor"]), "pct_no_viaja": round(ind["2"]["valor"] * 100, 2),
                    "tiempo_promedio_min": round(ind["3"]["valor"], 1),
                    "motos_por_1000_hab": round(ind["13"]["valor"], 1),
                    "autos_por_1000_hab": round(ind["12"]["valor"], 1), "reparto_modal_pct": modos,
                    "macrozona_mas_atractora": str(zonas.iloc[0]["municipio"] + " - " + zonas.iloc[0]["macrozona"]),
                    "viajes_por_municipio_residencia": D["eod"].get("viajes_por_municipio_residencia", {})},
        "siniestralidad": sin, "aforos": afo, "lluvia_pct_colapso_pico": lluvia,
        "referencias_publicadas": config.REFERENCIAS, "figuras": figuras,
    }
    (config.EDA_REPORTS / "eda_summary.json").write_text(json.dumps(resumen, indent=2, ensure_ascii=False, default=str),
                                                         encoding="utf-8")
    _reporte(resumen, tasa, calidad)
    log.info("EDA completo: %d figuras", len(figuras))
    return resumen


def _reporte(r: dict, tasa: pd.Series, calidad: pd.DataFrame) -> None:
    se, eo, si = r["sensores"], r["eod2025"], r["siniestralidad"]
    ref = config.REFERENCIAS
    L = [
        "# Reporte EDA — MobiPredict (datos reales)",
        "",
        "Generado por `src/mobipredict/eda.py`. Figuras en `eda/figures/`. Fuentes en `docs/fuentes_de_datos.md`.",
        "",
        "## Cifras clave",
        "",
        "| Indicador | Valor | Fuente |",
        "| --- | --- | --- |",
        f"| Viajes en un día hábil | {_miles(eo['viajes_dia_habil'])} | EOD 2025 (API AMVA) |",
        f"| Personas que no viajan | {eo['pct_no_viaja']} % | EOD 2025 |",
        f"| Tiempo promedio de viaje | {eo['tiempo_promedio_min']} min | EOD 2025 |",
        f"| Motos / autos por 1.000 habitantes | {eo['motos_por_1000_hab']} / {eo['autos_por_1000_hab']} | EOD 2025 |",
        f"| Variación tenencia de motos 2017-2025 | +{ref['variacion_tenencia_motos_2017_2025_pct']['valor']:.0f} % | "
        "EOD 2025, publicado por El Colombiano |",
        f"| Variación participación TP 2017-2025 | {ref['variacion_participacion_tp_2017_2025_pct']['valor']:.0f} % | "
        "EOD 2025, publicado por El Colombiano |",
        f"| Muertes viales Medellín 2025 | {ref['muertes_viales_medellin_2025']['valor']} | Observatorio de Movilidad |",
        f"| Lecturas de velocidad (tramo-hora) | {_miles(se['lecturas'])} en {se['tramos']} tramos, {se['dias']} días | "
        "MEData / SIMM |",
        f"| Horas en colapso (laborable, pre-pandemia) | {se['pct_horas_colapso_pre_pandemia']} % | MEData / SIMM |",
    ]
    if si:
        L.append(f"| Incidentes viales 2008-ago 2025 | {_miles(si['incidentes_2008_2025'])} "
                 f"({_miles(si['incidentes_fatales'])} fatales) | Mendeley Data |")
    if r.get("aforos"):
        L.append(f"| Motos en los aforos | {r['aforos']['pct_motos']} % de los vehículos | MEData aforos |")
    L += [
        "",
        "## Hallazgos",
        "",
        f"1. **Congestión crónica.** Se repite a diario en las horas pico; {se['tramo_mas_congestionado']} es el tramo "
        "más congestionado. El perfil histórico por tramo y hora es una línea base fuerte y una feature clave.",
        f"2. **Demanda.** La EOD 2025 registra {_miles(eo['viajes_dia_habil'])} viajes en un día hábil, con pico de "
        "inicio a las 6 h. La distribución horaria oficial entra al modelo como prior de demanda.",
        f"3. **Densidad.** {eo['macrozona_mas_atractora']} es la macrozona que más viajes atrae. Cada tramo hereda los "
        "viajes generados y atraídos de su macrozona como feature espacial.",
        f"4. **Motos.** Hay {eo['motos_por_1000_hab']} motos por cada 1.000 habitantes, casi el doble que autos. "
        + (f"En los aforos son el {r['aforos']['pct_motos']} % de los vehículos." if r.get("aforos") else ""),
        "5. **Lluvia.** Con más de 3 mm/h el colapso en hora pico es más frecuente; la lluvia actual y la de las "
        "últimas 3 horas entran como features.",
        "6. **Pandemia.** Desde el 20-mar-2020 la congestión se desplomó. Ese periodo se aparta del entrenamiento y "
        "se usa como prueba de estrés de drift.",
    ]
    if si:
        L.append(f"7. **Siniestralidad.** {_miles(si['incidentes_2008_2025'])} incidentes reportados entre 2008 y agosto "
                 f"de 2025; la hora con más incidentes es las {si['hora_con_mas_incidentes']} h. El cierre oficial de "
                 f"2025 fue de {ref['muertes_viales_medellin_2025']['valor']} muertes "
                 f"({ref['muertes_viales_medellin_2025']['detalle']}).")
    L += [
        "",
        "## Horas en colapso por tramo",
        "",
        "| Tramo | % horas |",
        "| --- | --- |",
        *[f"| {k} | {v:.2f} |" for k, v in tasa.sort_values(ascending=False).items()],
        "",
        "## Calidad de la serie de sensores",
        "",
        "Cobertura = lecturas / (días con datos × 15 horas). Detalle en `calidad_sensores_por_tramo.csv`.",
        "",
        f"Cobertura media: {se['cobertura_media_pct']} %. Las lecturas solo existen de 6:00 a 20:00; los rezagos "
        "no cruzan la noche y las horas faltantes no se imputan.",
        "",
        "## Limitaciones",
        "",
        "- La serie de velocidad abierta llega a diciembre de 2020; la EOD es de 2025. Se asume que la forma horaria "
        "de la demanda y la jerarquía de macrozonas son estables; antes de operar hay que reentrenar con el feed "
        "actual del SIMM.",
        "- Las velocidades provienen de tiempos de viaje de Google Traffic agregados por la Alcaldía, no de espiras.",
        "- La base de siniestros cuenta incidentes (un incidente puede tener varias víctimas); el cierre oficial cuenta "
        "personas fallecidas.",
    ]
    (config.EDA_REPORTS / "eda_report.md").write_text("\n".join(L), encoding="utf-8")

### 4. `training.py` — candidatos, RandomizedSearchCV + Optuna con `TimeSeriesSplit`, registro en MLflow

In [ ]:
%%writefile src/mobipredict/training.py
"""Entrenamiento, optimización de hiperparámetros y registro en MLflow.

Dos tareas sobre la serie real de velocidad horaria por tramo (MEData/SIMM 2017-2020):
  * Regresión: velocidad media del tramo dentro de 1 hora (km/h).
  * Clasificación: colapso dentro de 1 hora (velocidad < 50 % de la de referencia del tramo).

Cada tarea compara una línea base sin entrenamiento contra modelos lineales,
Random Forest, LightGBM (Gradient Boosting) y una red neuronal (MLP). La
búsqueda de hiperparámetros usa validación cruzada temporal (TimeSeriesSplit):
RandomizedSearchCV para scikit-learn y Optuna para LightGBM.

Protocolo: se entrena con `train`, se elige umbral y modelo con `valid` y se
reporta el desempeño final en `test` (nunca visto durante el ajuste). Además se mide el
desempeño en `drift` (marzo-diciembre de 2020, pandemia) como prueba de estrés de drift.
"""
from __future__ import annotations

import json
import logging
import time
import warnings

import joblib
import lightgbm as lgb
import mlflow
import numpy as np
import optuna
import pandas as pd
from scipy.stats import loguniform
from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (average_precision_score, brier_score_loss, f1_score, mean_absolute_error,
                             mean_squared_error, precision_recall_curve, precision_score, r2_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit, cross_val_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from . import config
from .features import FEATURES_NUM, TARGET_CLF, TARGET_REG

log = logging.getLogger(__name__)
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=UserWarning)
CAND_DIR = config.MODELS_DIR / "candidatos"


# --------------------------------------------------------------------------- #
# Líneas base (sin entrenamiento) con interfaz scikit-learn
# --------------------------------------------------------------------------- #
class PerfilHistoricoRegressor(BaseEstimator, RegressorMixin):
    """Predice la velocidad promedio histórica del tramo para el tipo de día y la hora objetivo."""

    def fit(self, X, y=None):
        self.fallback_ = float(np.nanmean(y)) if y is not None else 0.0
        return self

    def predict(self, X):
        p = X["perfil_velocidad"].to_numpy(dtype=float)
        return np.where(np.isnan(p), X["velocidad_kmh"].fillna(self.fallback_).to_numpy(), p)


class ReglaVelocidadClassifier(BaseEstimator, ClassifierMixin):
    """Regla operativa: riesgo de colapso según qué tan cerca está la velocidad actual del umbral."""

    def fit(self, X, y=None):
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        r = X["ratio_vel"].fillna(1).to_numpy()
        tendencia = X["ratio_vel_delta1"].fillna(0).to_numpy()
        umbral = config.COLLAPSE_SPEED_RATIO
        p = np.clip((umbral + 0.15 - (r + tendencia)) / 0.30, 0, 1)
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


# --------------------------------------------------------------------------- #
# Métricas
# --------------------------------------------------------------------------- #
def metricas_regresion(y, p, es_pico) -> dict:
    y, p = np.asarray(y), np.asarray(p)
    es_pico = np.asarray(es_pico).astype(bool)
    return {
        "mae": mean_absolute_error(y, p),
        "rmse": float(np.sqrt(mean_squared_error(y, p))),
        "wape": float(np.abs(y - p).sum() / np.abs(y).sum()),
        "r2": r2_score(y, p),
        "mae_pico": mean_absolute_error(y[es_pico], p[es_pico]) if es_pico.any() else np.nan,
    }


def umbral_optimo(y, prob) -> float:
    """Umbral que maximiza F1 en validación (se fija ANTES de mirar test)."""
    prec, rec, thr = precision_recall_curve(y, prob)
    f1 = 2 * prec * rec / np.clip(prec + rec, 1e-9, None)
    return float(thr[np.nanargmax(f1[:-1])]) if len(thr) else 0.5


def metricas_clasificacion(y, prob, umbral: float, n_dias: float) -> dict:
    y, prob = np.asarray(y), np.asarray(prob)
    pred = (prob >= umbral).astype(int)
    falsas = int(((pred == 1) & (y == 0)).sum())
    if y.sum() == 0:   # sin colapsos reales (p. ej. pandemia): solo tiene sentido medir falsas alarmas
        return {"precision_colapso": np.nan, "recall_colapso": np.nan, "f1_colapso": np.nan, "pr_auc": np.nan,
                "roc_auc": np.nan, "brier": brier_score_loss(y, prob),
                "falsas_alarmas_por_dia": falsas / max(n_dias, 1e-9), "tasa_positivos": 0.0}
    return {
        "precision_colapso": precision_score(y, pred, zero_division=0),
        "recall_colapso": recall_score(y, pred, zero_division=0),
        "f1_colapso": f1_score(y, pred, zero_division=0),
        "pr_auc": average_precision_score(y, prob),
        "roc_auc": roc_auc_score(y, prob) if len(np.unique(y)) > 1 else np.nan,
        "brier": brier_score_loss(y, prob),
        "falsas_alarmas_por_dia": falsas / max(n_dias, 1e-9),
        "tasa_positivos": float(np.mean(y)),
    }


# --------------------------------------------------------------------------- #
# Utilidades
# --------------------------------------------------------------------------- #
def _recientes(df: pd.DataFrame) -> pd.DataFrame:
    """Las filas más recientes hasta FIT_MAX_ROWS (para modelos costosos: RF y MLP)."""
    return df.iloc[-config.FIT_MAX_ROWS:] if len(df) > config.FIT_MAX_ROWS else df


def _muestra_busqueda(train: pd.DataFrame) -> pd.DataFrame:
    """Submuestra ordenada en el tiempo para la búsqueda de hiperparámetros."""
    if len(train) > config.SEARCH_SAMPLE:
        train = train.sample(config.SEARCH_SAMPLE, random_state=config.SEED)
    return train.sort_values("timestamp")


def _latencia_ms(model, X: pd.DataFrame, clasif: bool) -> float:
    Xs = X.iloc[: min(1000, len(X))]
    t0 = time.perf_counter()
    for _ in range(3):
        model.predict_proba(Xs) if clasif else model.predict(Xs)
    return (time.perf_counter() - t0) / 3 * 1000 / len(Xs) * 1000   # ms por 1000 filas


def _importancias(model, nombre: str) -> pd.DataFrame | None:
    est = model
    if isinstance(est, TransformedTargetRegressor):
        est = est.regressor_
    if isinstance(est, Pipeline):
        est = est.steps[-1][1]
    if hasattr(est, "feature_importances_"):
        imp = est.feature_importances_
    elif hasattr(est, "coef_"):
        imp = np.abs(np.ravel(est.coef_))
    else:
        return None
    imp = imp / imp.sum() if imp.sum() else imp
    return (pd.DataFrame({"feature": FEATURES_NUM, "importancia": imp})
            .sort_values("importancia", ascending=False).reset_index(drop=True))


def _tscv() -> TimeSeriesSplit:
    return TimeSeriesSplit(n_splits=3)


# --------------------------------------------------------------------------- #
# Candidatos y búsqueda de hiperparámetros
# --------------------------------------------------------------------------- #
def _imp() -> SimpleImputer:
    return SimpleImputer(strategy="median")


def _random_search(est, dist, X, y, scoring, n_iter=None):
    rs = RandomizedSearchCV(est, dist, n_iter=n_iter or config.N_ITER_RANDOM, cv=_tscv(), scoring=scoring,
                            random_state=config.SEED, n_jobs=-1, refit=False)
    rs.fit(X, y)
    return rs.best_params_, float(rs.best_score_)


def _optuna_lgbm(X, y, clasif: bool):
    def objetivo(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 150, 600, step=50),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 15, 127, log=True),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 120, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "subsample_freq": 1,
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
        }
        if clasif:
            params["scale_pos_weight"] = trial.suggest_float("scale_pos_weight", 1, 20, log=True)
            m = lgb.LGBMClassifier(**params, random_state=config.SEED, verbose=-1, n_jobs=-1)
            return cross_val_score(m, X, y, cv=_tscv(), scoring="average_precision").mean()
        m = lgb.LGBMRegressor(**params, random_state=config.SEED, verbose=-1, n_jobs=-1)
        return cross_val_score(m, X, y, cv=_tscv(), scoring="neg_root_mean_squared_error").mean()

    estudio = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=config.SEED))
    estudio.optimize(objetivo, n_trials=config.N_TRIALS_OPTUNA, show_progress_bar=False)
    best = dict(estudio.best_params, subsample_freq=1)
    return best, float(estudio.best_value)


def ajustar_regresores(train: pd.DataFrame) -> dict:
    """Devuelve {nombre: (modelo_entrenado, params, cv_score)} para la tarea de velocidad."""
    s = _muestra_busqueda(train)
    Xs, ys = s[FEATURES_NUM], s[TARGET_REG]
    X, y = train[FEATURES_NUM], train[TARGET_REG]
    out = {}

    out["perfil_historico"] = (PerfilHistoricoRegressor().fit(X, y), {"tipo": "linea_base"}, np.nan)

    ridge = Pipeline([("imp", _imp()), ("sc", StandardScaler()), ("m", Ridge())])
    bp, sc = _random_search(ridge, {"m__alpha": loguniform(1e-2, 1e3)}, Xs, ys, "neg_root_mean_squared_error")
    out["ridge"] = (ridge.set_params(**bp).fit(X, y), bp, sc)

    rf = Pipeline([("imp", _imp()), ("m", RandomForestRegressor(random_state=config.SEED, n_jobs=-1))])
    dist = {"m__n_estimators": [100, 150, 200], "m__max_depth": [8, 12, 16, 22],
            "m__min_samples_leaf": [1, 3, 5, 10], "m__max_features": ["sqrt", 0.3, 0.5]}
    bp, sc = _random_search(rf, dist, Xs, ys, "neg_root_mean_squared_error")
    r = _recientes(train)
    out["random_forest"] = (rf.set_params(**bp).fit(r[FEATURES_NUM], r[TARGET_REG]), bp, sc)

    bp, sc = _optuna_lgbm(Xs, ys, clasif=False)
    out["lightgbm"] = (lgb.LGBMRegressor(**bp, random_state=config.SEED, verbose=-1, n_jobs=-1).fit(X, y), bp, sc)

    mlp = TransformedTargetRegressor(
        regressor=Pipeline([("imp", _imp()), ("sc", StandardScaler()),
                            ("m", MLPRegressor(early_stopping=True, max_iter=300, random_state=config.SEED))]),
        transformer=StandardScaler())
    dist = {"regressor__m__hidden_layer_sizes": [(64,), (128, 64), (64, 32), (128, 64, 32)],
            "regressor__m__alpha": loguniform(1e-5, 1e-2),
            "regressor__m__learning_rate_init": loguniform(5e-4, 5e-3)}
    bp, sc = _random_search(mlp, dist, Xs, ys, "neg_root_mean_squared_error", n_iter=max(3, config.N_ITER_RANDOM // 2))
    out["mlp"] = (mlp.set_params(**bp).fit(r[FEATURES_NUM], r[TARGET_REG]), bp, sc)
    return out


def ajustar_clasificadores(train: pd.DataFrame) -> dict:
    s = _muestra_busqueda(train)
    Xs, ys = s[FEATURES_NUM], s[TARGET_CLF]
    X, y = train[FEATURES_NUM], train[TARGET_CLF]
    out = {}

    out["regla_umbral"] = (ReglaVelocidadClassifier().fit(X, y), {"tipo": "linea_base"}, np.nan)

    lr = Pipeline([("imp", _imp()), ("sc", StandardScaler()),
                   ("m", LogisticRegression(class_weight="balanced", max_iter=3000))])
    bp, sc = _random_search(lr, {"m__C": loguniform(1e-3, 1e2)}, Xs, ys, "average_precision")
    out["logistica"] = (lr.set_params(**bp).fit(X, y), bp, sc)

    rf = Pipeline([("imp", _imp()), ("m", RandomForestClassifier(class_weight="balanced_subsample",
                                                                  random_state=config.SEED, n_jobs=-1))])
    dist = {"m__n_estimators": [100, 150, 200], "m__max_depth": [6, 10, 14, 20],
            "m__min_samples_leaf": [1, 3, 5, 10], "m__max_features": ["sqrt", 0.3, 0.5]}
    bp, sc = _random_search(rf, dist, Xs, ys, "average_precision")
    r = _recientes(train)
    out["random_forest"] = (rf.set_params(**bp).fit(r[FEATURES_NUM], r[TARGET_CLF]), bp, sc)

    bp, sc = _optuna_lgbm(Xs, ys, clasif=True)
    out["lightgbm"] = (lgb.LGBMClassifier(**bp, random_state=config.SEED, verbose=-1, n_jobs=-1).fit(X, y), bp, sc)
    return out


# --------------------------------------------------------------------------- #
# Evaluación + MLflow
# --------------------------------------------------------------------------- #
def _setup_mlflow() -> None:
    mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)
    mlflow.set_experiment(config.MLFLOW_EXPERIMENT)


def _registrar_run(tarea: str, nombre: str, modelo, params: dict, cv_score: float, met: dict,
                   imp: pd.DataFrame | None, extra_params: dict | None = None) -> str:
    with mlflow.start_run(run_name=f"{tarea}-{nombre}", nested=True) as run:
        mlflow.set_tags({"tarea": tarea, "modelo": nombre, "proyecto": "MobiPredict"})
        mlflow.log_params({k: (str(v) if isinstance(v, (tuple, list)) else v) for k, v in params.items()})
        if extra_params:
            mlflow.log_params(extra_params)
        if not np.isnan(cv_score):
            mlflow.log_metric("cv_score", cv_score)
        mlflow.log_metrics({k: float(v) for k, v in met.items() if v is not None and not np.isnan(v)})
        if imp is not None:
            ruta = config.MODELS_DIR / f"_imp_{tarea}_{nombre}.csv"
            imp.to_csv(ruta, index=False)
            mlflow.log_artifact(str(ruta), artifact_path="importancias")
            ruta.unlink()
        ruta_modelo = CAND_DIR / f"{tarea}_{nombre}.joblib"
        joblib.dump(modelo, ruta_modelo)
        mlflow.log_artifact(str(ruta_modelo), artifact_path="modelo")
        return run.info.run_id


def entrenar_y_registrar(particiones: dict[str, pd.DataFrame], spec: dict) -> pd.DataFrame:
    """Entrena todos los candidatos, los evalúa y registra cada uno como un run de MLflow.

    Devuelve la tabla comparativa (una fila por tarea x modelo).
    """
    config.ensure_dirs()
    CAND_DIR.mkdir(exist_ok=True)
    _setup_mlflow()
    train = particiones["train"].sort_values("timestamp")
    valid, test, drift = particiones["valid"], particiones["test"], particiones["drift"]
    dias = {p: particiones[p]["timestamp"].dt.normalize().nunique() for p in ["valid", "test", "drift"]}
    filas = []

    with mlflow.start_run(run_name=f"pipeline-{time.strftime('%Y%m%d-%H%M%S')}") as padre:
        mlflow.set_tags({"tipo": "pipeline", "modo_rapido": str(config.FAST)})
        mlflow.log_params({"horizonte_min": spec["horizonte_min"], "n_features": len(FEATURES_NUM),
                           "filas_train": len(train), "filas_valid": len(valid), "filas_test": len(test),
                           "trials_optuna": config.N_TRIALS_OPTUNA, "iter_random_search": config.N_ITER_RANDOM})
        mlflow.log_dict(spec, "feature_spec.json")

        # ----- Regresión: velocidad a 1 hora
        t0 = time.time()
        regs = ajustar_regresores(train)
        log.info("Regresores ajustados en %.0f s", time.time() - t0)
        for nombre, (m, params, cv) in regs.items():
            met = {}
            for part, df in [("valid", valid), ("test", test), ("drift", drift)]:
                if df.empty:
                    continue
                for k, v in metricas_regresion(df[TARGET_REG], m.predict(df[FEATURES_NUM]), df["es_pico"]).items():
                    met[f"{part}_{k}"] = v
            met["latencia_ms_1000"] = _latencia_ms(m, test[FEATURES_NUM], clasif=False)
            run_id = _registrar_run("velocidad", nombre, m, params, cv, met, _importancias(m, nombre))
            filas.append({"tarea": "velocidad", "modelo": nombre, "run_id": run_id, "cv_score": cv,
                          "params": json.dumps(params, default=str), **met})

        # ----- Clasificación: colapso a 1 hora
        t0 = time.time()
        clfs = ajustar_clasificadores(train)
        log.info("Clasificadores ajustados en %.0f s", time.time() - t0)
        for nombre, (m, params, cv) in clfs.items():
            pv = m.predict_proba(valid[FEATURES_NUM])[:, 1]
            umbral = umbral_optimo(valid[TARGET_CLF], pv)
            met = {f"valid_{k}": v for k, v in
                   metricas_clasificacion(valid[TARGET_CLF], pv, umbral, dias["valid"]).items()}
            pt = m.predict_proba(test[FEATURES_NUM])[:, 1]
            met.update({f"test_{k}": v for k, v in
                        metricas_clasificacion(test[TARGET_CLF], pt, umbral, dias["test"]).items()})
            if not drift.empty:
                pdft = m.predict_proba(drift[FEATURES_NUM])[:, 1]
                met.update({f"drift_{k}": v for k, v in
                            metricas_clasificacion(drift[TARGET_CLF], pdft, umbral, dias["drift"]).items()})
            met["umbral"] = umbral
            met["latencia_ms_1000"] = _latencia_ms(m, test[FEATURES_NUM], clasif=True)
            run_id = _registrar_run("colapso", nombre, m, params, cv, met, _importancias(m, nombre),
                                    {"umbral_decision": round(umbral, 4)})
            filas.append({"tarea": "colapso", "modelo": nombre, "run_id": run_id, "cv_score": cv,
                          "params": json.dumps(params, default=str), **met})

        tabla = pd.DataFrame(filas)
        ruta = config.MODELS_DIR / "_resultados_entrenamiento.csv"
        tabla.to_csv(ruta, index=False)
        mlflow.log_artifact(str(ruta))
        tabla.attrs["parent_run_id"] = padre.info.run_id
    return tabla

### 5. `selection.py` — selección razonada, tabla comparativa, decisión y Model Registry

In [ ]:
%%writefile src/mobipredict/selection.py
"""Selección razonada del modelo ganador por tarea.

Reglas (definidas antes de ver resultados y aplicadas sobre VALIDACIÓN; test solo se reporta):

Velocidad a 1 hora
  1. Debe mejorar el WAPE de la línea base (perfil histórico) en al menos 10 %.
  2. Gana el menor RMSE de validación.
  3. Desempate: si otro candidato está a menos de 1 % de RMSE, se prefiere el de menor latencia.

Colapso a 1 hora
  1. Elegibles: recall de validación >= 0,70 con el umbral F1-óptimo.
     Si ninguno cumple, se consideran todos y se advierte.
  2. Gana la mayor PR-AUC de validación (robusta al desbalance).
  3. Desempate: diferencia de PR-AUC < 0,005 -> menor Brier y luego menor latencia.

El periodo de pandemia (drift) se reporta para todos los modelos, pero NO se usa para elegir.
"""
from __future__ import annotations

import json
import logging
import shutil
from datetime import datetime, timezone

import joblib
import mlflow
import mlflow.sklearn
import pandas as pd

from . import config
from .training import CAND_DIR

log = logging.getLogger(__name__)

BASE_VEL, BASE_COLAPSO = "perfil_historico", "regla_umbral"
MEJORA_MIN_WAPE = 0.10
TOL_RMSE = 0.01
RECALL_MIN = 0.70
TOL_PRAUC = 0.005


def _elegir_velocidad(t: pd.DataFrame) -> tuple[str, list[str]]:
    razones = []
    base = t.loc[t["modelo"] == BASE_VEL].iloc[0]
    cand = t[t["modelo"] != BASE_VEL].copy()
    cand["mejora_wape_vs_base"] = 1 - cand["valid_wape"] / base["valid_wape"]
    aptos = cand[cand["mejora_wape_vs_base"] >= MEJORA_MIN_WAPE]
    if aptos.empty:
        razones.append("Ningún modelo superó a la línea base en 10 % de WAPE; se elige el mejor disponible y se "
                       "recomienda revisar features.")
        aptos = cand
    else:
        fuera = sorted(set(cand["modelo"]) - set(aptos["modelo"]))
        if fuera:
            razones.append(f"Descartados por no mejorar 10 % el WAPE de la línea base: {', '.join(fuera)}.")
    aptos = aptos.sort_values("valid_rmse")
    mejor = aptos.iloc[0]
    cercanos = aptos[aptos["valid_rmse"] <= mejor["valid_rmse"] * (1 + TOL_RMSE)]
    ganador = cercanos.sort_values("latencia_ms_1000").iloc[0] if len(cercanos) > 1 else mejor
    if ganador["modelo"] != mejor["modelo"]:
        razones.append(f"{ganador['modelo']} queda a menos de 1 % del RMSE de {mejor['modelo']} y es más rápido; "
                       "se prefiere por latencia.")
    razones.append(f"{ganador['modelo']} obtiene el menor RMSE de validación ({ganador['valid_rmse']:.2f} km/h) y "
                   f"reduce el WAPE {ganador['mejora_wape_vs_base'] * 100:.1f} % frente al perfil histórico.")
    return ganador["modelo"], razones


def _elegir_colapso(t: pd.DataFrame) -> tuple[str, list[str]]:
    razones = []
    cand = t[t["modelo"] != BASE_COLAPSO]
    aptos = cand[cand["valid_recall_colapso"] >= RECALL_MIN]
    if aptos.empty:
        razones.append(f"Ningún modelo alcanza recall >= {RECALL_MIN} en validación; se elige por PR-AUC entre todos.")
        aptos = cand
    else:
        fuera = sorted(set(cand["modelo"]) - set(aptos["modelo"]))
        if fuera:
            razones.append(f"Descartados por recall < {RECALL_MIN}: {', '.join(fuera)}.")
    aptos = aptos.sort_values("valid_pr_auc", ascending=False)
    mejor = aptos.iloc[0]
    cercanos = aptos[aptos["valid_pr_auc"] >= mejor["valid_pr_auc"] - TOL_PRAUC]
    ganador = cercanos.sort_values(["valid_brier", "latencia_ms_1000"]).iloc[0] if len(cercanos) > 1 else mejor
    if ganador["modelo"] != mejor["modelo"]:
        razones.append(f"{ganador['modelo']} empata en PR-AUC con {mejor['modelo']} (< 0,005) y está mejor calibrado.")
    razones.append(f"{ganador['modelo']} obtiene PR-AUC de validación {ganador['valid_pr_auc']:.3f} con recall "
                   f"{ganador['valid_recall_colapso']:.2f} y precisión {ganador['valid_precision_colapso']:.2f} "
                   f"(umbral {ganador['umbral']:.3f}).")
    return ganador["modelo"], razones


def _registrar_en_mlflow(tarea: str, modelo: str, run_id: str) -> str | None:
    """Marca el run ganador y registra el modelo en el Model Registry local con alias champion."""
    mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)
    client = mlflow.MlflowClient()
    client.set_tag(run_id, "seleccionado", "true")
    nombre = f"mobipredict-{tarea}"
    try:
        modelo_obj = joblib.load(CAND_DIR / f"{tarea}_{modelo}.joblib")
        with mlflow.start_run(run_id=run_id):
            info = mlflow.sklearn.log_model(modelo_obj, name="modelo_mlflow", serialization_format="cloudpickle")
        mv = mlflow.register_model(info.model_uri, nombre)
        client.set_registered_model_alias(nombre, "champion", mv.version)
        return f"{nombre} v{mv.version} (alias champion)"
    except Exception as exc:
        log.warning("No se pudo registrar %s en el Model Registry: %s", nombre, exc)
        return None


ORDEN = ["tarea", "modelo", "seleccionado",
         "valid_mae", "valid_rmse", "valid_wape", "valid_r2", "valid_mae_pico",
         "test_mae", "test_rmse", "test_wape", "test_r2", "test_mae_pico",
         "drift_mae", "drift_rmse", "drift_wape",
         "valid_precision_colapso", "valid_recall_colapso", "valid_f1_colapso", "valid_pr_auc", "valid_roc_auc",
         "valid_brier", "test_precision_colapso", "test_recall_colapso", "test_f1_colapso", "test_pr_auc",
         "test_roc_auc", "test_brier", "test_falsas_alarmas_por_dia", "test_tasa_positivos",
         "drift_precision_colapso", "drift_recall_colapso", "drift_pr_auc", "drift_falsas_alarmas_por_dia",
         "umbral", "latencia_ms_1000", "cv_score", "run_id", "params"]


def seleccionar(tabla: pd.DataFrame, spec: dict) -> dict:
    config.ensure_dirs()
    vel = tabla[tabla["tarea"] == "velocidad"].copy()
    col = tabla[tabla["tarea"] == "colapso"].copy()
    g_vel, r_vel = _elegir_velocidad(vel)
    g_col, r_col = _elegir_colapso(col)

    tabla = tabla.copy()
    tabla["seleccionado"] = (((tabla["tarea"] == "velocidad") & (tabla["modelo"] == g_vel))
                             | ((tabla["tarea"] == "colapso") & (tabla["modelo"] == g_col)))
    tabla[[c for c in ORDEN if c in tabla.columns]].round(4).to_csv(config.COMPARISON_FILE, index=False)

    fv, fc = vel.set_index("modelo").loc[g_vel], col.set_index("modelo").loc[g_col]
    shutil.copy(CAND_DIR / f"velocidad_{g_vel}.joblib", config.MODEL_SPEED_FILE)
    shutil.copy(CAND_DIR / f"colapso_{g_col}.joblib", config.MODEL_COLLAPSE_FILE)
    reg_v = _registrar_en_mlflow("velocidad", g_vel, fv["run_id"])
    reg_c = _registrar_en_mlflow("colapso", g_col, fc["run_id"])

    manifest = json.loads(config.MANIFEST_FILE.read_text(encoding="utf-8")) if config.MANIFEST_FILE.exists() else {}
    metadata = {
        "generado_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "horizonte_min": spec["horizonte_min"],
        "features": spec["features"],
        "umbral_colapso_ratio_vel": spec["umbral_colapso_ratio_vel"],
        "min_observaciones": config.MIN_HISTORY,
        "modelo_velocidad": {"nombre": g_vel, "run_id": fv["run_id"], "registro": reg_v,
                             "test_mae_kmh": round(float(fv["test_mae"]), 3),
                             "test_rmse_kmh": round(float(fv["test_rmse"]), 3),
                             "test_wape": round(float(fv["test_wape"]), 4)},
        "modelo_colapso": {"nombre": g_col, "run_id": fc["run_id"], "registro": reg_c,
                           "umbral_decision": round(float(fc["umbral"]), 4),
                           "test_pr_auc": round(float(fc["test_pr_auc"]), 4),
                           "test_recall": round(float(fc["test_recall_colapso"]), 4),
                           "test_precision": round(float(fc["test_precision_colapso"]), 4)},
        "fuentes": {k: {"titulo": v.get("titulo"), "origen": v.get("origen"), "ficha": v.get("ficha")}
                    for k, v in manifest.get("fuentes", {}).items()},
    }
    config.METADATA_FILE.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
    _escribir_decision(vel, col, g_vel, g_col, r_vel, r_col, metadata, spec)
    log.info("Seleccionados: velocidad=%s, colapso=%s", g_vel, g_col)
    return metadata


def _tabla_md(df: pd.DataFrame, cols: dict[str, str], ganador: str, fmt: dict[str, str]) -> list[str]:
    head = "| Modelo | " + " | ".join(cols.values()) + " |"
    sep = "| --- | " + " | ".join("---" for _ in cols) + " |"
    filas = []
    for _, r in df.iterrows():
        nombre = f"**{r['modelo']}** ✔" if r["modelo"] == ganador else r["modelo"]
        celdas = [("—" if pd.isna(r.get(c)) else format(r[c], fmt.get(c, ".3f"))) for c in cols]
        filas.append(f"| {nombre} | " + " | ".join(celdas) + " |")
    return [head, sep, *filas]


def _nota_test(df, ganador, col, menor_es_mejor, nombre_met) -> list[str]:
    mejor_test = df.sort_values(col, ascending=menor_es_mejor).iloc[0]
    if mejor_test["modelo"] == ganador:
        return [f"- En prueba el ganador también es el mejor en {nombre_met}: la decisión se sostiene fuera de muestra."]
    g = df.set_index("modelo").loc[ganador]
    return [f"- En prueba, {mejor_test['modelo']} queda ligeramente mejor en {nombre_met} "
            f"({mejor_test[col]:.3f} frente a {g[col]:.3f}). No se cambia la elección: hacerlo usaría el conjunto de "
            "prueba para seleccionar. La diferencia se vigila en producción con el esquema champion/challenger."]


def _hz(spec) -> str:
    m = spec["horizonte_min"]
    return "1 hora" if m == 60 else f"{m} min"


def _escribir_decision(vel, col, g_vel, g_col, r_vel, r_col, meta, spec) -> None:
    v = vel.sort_values("valid_rmse")
    c = col.sort_values("valid_pr_auc", ascending=False)
    gv, gc = v.set_index("modelo").loc[g_vel], c.set_index("modelo").loc[g_col]
    bv, bc = v.set_index("modelo").loc[BASE_VEL], c.set_index("modelo").loc[BASE_COLAPSO]
    tasa = spec.get("tasa_colapso", {})
    L = [
        "# Decisión de modelo — MobiPredict",
        "",
        f"Generado automáticamente el {meta['generado_utc']} por `src/mobipredict/selection.py`.",
        "",
        "## Resumen",
        "",
        f"- **Velocidad a {_hz(spec)}:** `{g_vel}` — RMSE de prueba {gv['test_rmse']:.2f} km/h, "
        f"MAE {gv['test_mae']:.2f} km/h, WAPE {gv['test_wape'] * 100:.1f} % "
        f"(perfil histórico: WAPE {bv['test_wape'] * 100:.1f} %).",
        f"- **Colapso a {_hz(spec)}:** `{g_col}` — PR-AUC de prueba {gc['test_pr_auc']:.3f}, recall "
        f"{gc['test_recall_colapso']:.2f}, precisión {gc['test_precision_colapso']:.2f} con umbral {gc['umbral']:.3f} "
        f"(regla base: PR-AUC {bc['test_pr_auc']:.3f}).",
        "",
        "## Datos y protocolo",
        "",
        f"- Sensores: {spec['fuente_sensores']}, {spec['tramos']} tramos, lecturas horarias 6:00-20:00.",
        f"- Split temporal: entrenamiento hasta {spec['corte_validacion']}, validación hasta {spec['corte_test']}, "
        f"prueba hasta {spec['fin_test']} (antes de las restricciones por COVID-19). La hora frontera de cada "
        "partición se descarta para que el objetivo no cruce de una a otra.",
        f"- Drift: {spec['ventana_drift'][0]} a {spec['ventana_drift'][1]} (pandemia), solo para diagnóstico.",
        f"- Colapso = velocidad < {spec['umbral_colapso_ratio_vel']:.0%} de la velocidad de referencia del tramo "
        f"(percentil {spec['percentil_vel_referencia'] * 100:.0f} del entrenamiento). Tasa de colapso: "
        + ", ".join(f"{k} {v:.1%}" for k, v in tasa.items()) + ".",
        "- Hiperparámetros con validación cruzada temporal (TimeSeriesSplit, 3 cortes): RandomizedSearchCV y Optuna.",
        "- Selección y umbral de decisión se fijan en **validación**; **prueba** solo se reporta.",
        "",
        f"## Velocidad a {_hz(spec)}",
        "",
        *_tabla_md(v, {"valid_rmse": "RMSE val", "valid_wape": "WAPE val", "valid_mae_pico": "MAE pico val",
                       "test_mae": "MAE test", "test_rmse": "RMSE test", "test_r2": "R² test",
                       "latencia_ms_1000": "ms/1000 filas"}, g_vel,
                   {"valid_rmse": ".2f", "valid_mae_pico": ".2f", "test_mae": ".2f", "test_rmse": ".2f",
                    "latencia_ms_1000": ".1f"}),
        "",
        "Unidades: km/h. WAPE = error absoluto total / velocidad total.",
        "",
        "**Razonamiento**",
        "",
        *[f"- {r}" for r in r_vel],
        *_nota_test(v, g_vel, "test_rmse", True, "RMSE"),
        "",
        f"## Colapso a {_hz(spec)}",
        "",
        *_tabla_md(c, {"valid_pr_auc": "PR-AUC val", "valid_recall_colapso": "Recall val",
                       "valid_precision_colapso": "Precisión val", "valid_brier": "Brier val",
                       "test_pr_auc": "PR-AUC test", "test_recall_colapso": "Recall test",
                       "test_precision_colapso": "Precisión test",
                       "test_falsas_alarmas_por_dia": "Falsas alarmas/día (red)",
                       "latencia_ms_1000": "ms/1000 filas"}, g_col,
                   {"test_falsas_alarmas_por_dia": ".1f", "latencia_ms_1000": ".1f"}),
        "",
        "**Razonamiento**",
        "",
        *[f"- {r}" for r in r_col],
        *_nota_test(c, g_col, "test_pr_auc", False, "PR-AUC"),
        "- Se prioriza recall: una alerta perdida cuesta más que una falsa alarma que un operador descarta.",
        "",
        "## Prueba de estrés: pandemia (drift)",
        "",
        *_tabla_md(v, {"test_mae": "MAE test (pre-pandemia)", "drift_mae": "MAE drift (pandemia)",
                       "drift_wape": "WAPE drift"}, g_vel, {"test_mae": ".2f", "drift_mae": ".2f"}),
        "",
        *_tabla_md(c, {"test_pr_auc": "PR-AUC test", "drift_pr_auc": "PR-AUC drift",
                       "drift_recall_colapso": "Recall drift", "drift_precision_colapso": "Precisión drift",
                       "drift_falsas_alarmas_por_dia": "Falsas alarmas/día drift"},
                   g_col, {"drift_falsas_alarmas_por_dia": ".1f"}),
        "",
        "— = no aplica: sin colapsos reales en el periodo no se pueden medir recall ni PR-AUC.",
        "",
        "Con las vías casi vacías la congestión casi desaparece: la tasa de colapso del periodo de drift es "
        f"{tasa.get('drift', float('nan')):.1%} frente a {tasa.get('test', float('nan')):.1%} en prueba. Es un cambio de "
        "régimen que el monitoreo debe detectar (drift de entrada en velocidades y de salida en la tasa de alertas) "
        "y que justifica el reentrenamiento por disparador, no solo por calendario.",
        "",
        "## Riesgos y siguientes pasos",
        "",
        "- Los sensores abiertos más recientes publicados por la Alcaldía llegan a dic-2020; antes de operar en 2026 "
        "hay que reentrenar con el feed actual del SIMM.",
        "- Revisar el umbral de colapso y el de alerta con la Secretaría de Movilidad.",
        "- Evaluar un modelo de grafos espacio-temporal como challenger con más tramos.",
    ]
    config.DECISION_FILE.write_text("\n".join(L), encoding="utf-8")

### 6. `pipeline.py` — `@flow` y `@task` de Prefect

In [ ]:
%%writefile src/mobipredict/pipeline.py
"""Orquestación con Prefect.

Adquisición -> Estandarización -> EDA -> Features -> Entrenamiento/Optimización (+ MLflow) -> Selección.
"""
from __future__ import annotations

import time

import pandas as pd
from prefect import flow, get_run_logger, task

from . import config, data_acquisition, eda, features, fuentes, selection, training


@task(name="adquisicion-datos", retries=1, retry_delay_seconds=10)
def tarea_adquisicion(forzar_descarga: bool = False) -> dict:
    manifest = data_acquisition.adquirir_datos(forzar=forzar_descarga)
    get_run_logger().info("Fuentes: %s", {k: v["origen"] for k, v in manifest["fuentes"].items()})
    return manifest


@task(name="estandarizacion-fuentes")
def tarea_estandarizacion() -> dict:
    resumen = fuentes.estandarizar_todo()
    get_run_logger().info("Tablas estandarizadas: %s", resumen)
    return resumen


@task(name="eda")
def tarea_eda() -> dict:
    resumen = eda.ejecutar_eda()
    get_run_logger().info("EDA: %d figuras en eda/figures/", len(resumen["figuras"]))
    return resumen


@task(name="feature-engineering")
def tarea_features() -> dict:
    spec = features.preparar_dataset()
    get_run_logger().info("Particiones: %s", spec["filas"])
    return spec


@task(name="entrenamiento-optimizacion-mlflow")
def tarea_entrenamiento(spec: dict) -> pd.DataFrame:
    tabla = training.entrenar_y_registrar(features.cargar_particiones(), spec)
    get_run_logger().info("Candidatos entrenados: %d (MLflow: %s)", len(tabla), config.MLFLOW_TRACKING_URI)
    return tabla


@task(name="seleccion-modelo")
def tarea_seleccion(tabla: pd.DataFrame, spec: dict) -> dict:
    meta = selection.seleccionar(tabla, spec)
    get_run_logger().info("Ganadores: velocidad=%s, colapso=%s",
                          meta["modelo_velocidad"]["nombre"], meta["modelo_colapso"]["nombre"])
    return meta


@flow(name="mobipredict-pipeline", log_prints=True)
def mobipredict_pipeline(ejecutar_eda: bool = True, forzar_descarga: bool = False) -> dict:
    t0 = time.time()
    tarea_adquisicion(forzar_descarga)
    tarea_estandarizacion()
    resumen_eda = tarea_eda() if ejecutar_eda else None
    spec = tarea_features()
    tabla = tarea_entrenamiento(spec)
    meta = tarea_seleccion(tabla, spec)
    print(f"Pipeline completo en {time.time() - t0:.0f} s")
    return {"eda": resumen_eda, "spec": spec, "metadata": meta}

### `run_pipeline.py` — un solo comando

In [ ]:
%%writefile run_pipeline.py
"""Punto de entrada único del pipeline MobiPredict.

Uso:
    python run_pipeline.py              # corrida completa
    python run_pipeline.py --fast       # corrida rápida (menos trials de búsqueda)
    python run_pipeline.py --skip-eda   # omite el EDA
    python run_pipeline.py --descargar  # vuelve a descargar las fuentes aunque existan en data/raw/
"""
from __future__ import annotations

import argparse
import json
import logging
import os
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT / "src"))


def main() -> None:
    p = argparse.ArgumentParser(description="Pipeline MLOps de MobiPredict (Prefect + MLflow)")
    p.add_argument("--fast", action="store_true", help="modo rápido para pruebas")
    p.add_argument("--skip-eda", action="store_true", help="no ejecutar el EDA")
    p.add_argument("--descargar", action="store_true", help="forzar nueva descarga de las fuentes")
    a = p.parse_args()
    if a.fast:
        os.environ["MOBIPREDICT_FAST"] = "1"   # debe fijarse antes de importar config

    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
    from mobipredict.pipeline import mobipredict_pipeline

    out = mobipredict_pipeline(ejecutar_eda=not a.skip_eda, forzar_descarga=a.descargar)
    print("\n=== Modelos seleccionados ===")
    print(json.dumps({k: out["metadata"][k] for k in ["modelo_velocidad", "modelo_colapso"]}, indent=2,
                     ensure_ascii=False))
    print("\nArtefactos: models/model_comparison.csv · docs/model_decision.md · mlruns/")


if __name__ == "__main__":
    main()

### `scripts/descargar_datos.py` — descarga de fuentes sin entrenar

In [ ]:
%%writefile scripts/descargar_datos.py
"""Descarga todas las fuentes reales a data/raw/ sin entrenar (multiplataforma).

Uso:  python scripts/descargar_datos.py [--forzar]
"""
import argparse
import json
import logging
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from mobipredict.data_acquisition import FuenteNoDisponible, adquirir_datos  # noqa: E402

if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--forzar", action="store_true", help="volver a descargar aunque existan")
    a = p.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    try:
        m = adquirir_datos(forzar=a.forzar)
    except FuenteNoDisponible as exc:
        print(f"\nERROR: {exc}")
        sys.exit(1)
    print(json.dumps({k: {"origen": v["origen"], "MB": round(v["bytes"] / 1e6, 1)} for k, v in m["fuentes"].items()},
                     indent=2, ensure_ascii=False))
    if m["errores"]:
        print("Fuentes con error (opcionales):", m["errores"])

### `scripts/descargar_datos.ps1` — descarga en Windows (PowerShell)

In [ ]:
%%writefile scripts/descargar_datos.ps1
# Descarga las fuentes reales de MobiPredict a data\raw\ (Windows PowerShell 5+ o PowerShell 7).
# Uso, desde la raíz del proyecto:
#   powershell -ExecutionPolicy Bypass -File scripts\descargar_datos.ps1
# Luego: python run_pipeline.py
$ErrorActionPreference = "Stop"
$ProgressPreference = "SilentlyContinue"   # acelera Invoke-WebRequest
$raw = Join-Path $PSScriptRoot "..\data\raw"
New-Item -ItemType Directory -Force -Path $raw | Out-Null

$fuentes = @(
  @{ nombre = "medata_velocidad_tiempo_viaje_gt.csv";
     url = "https://medata.app.medellin.gov.co/sites/default/files/distribution/1-023-25-000287/Velocidad_y_tiempo_de_viaje_GT.csv" },
  @{ nombre = "medata_aforos_vehiculares.csv";
     url = "https://medata.app.medellin.gov.co/sites/default/files/distribution/1-023-25-000301/Aforos_Vehiculares.csv" },
  @{ nombre = "mendeley_siniestros_medellin_2008_2025.zip";
     url = "https://data.mendeley.com/public-files/datasets/r6g5dfnpgh/files/e992c888-5819-40c6-9538-41980ea1d429/file_downloaded" },
  @{ nombre = "open_meteo_medellin_horario.json";
     url = "https://archive-api.open-meteo.com/v1/archive?latitude=6.2442&longitude=-75.5812&start_date=2017-01-01&end_date=2020-12-31&hourly=precipitation,temperature_2m&timezone=America%2FBogota" },
  @{ nombre = "eod2025_macrozonas.geo.json";
     url = "https://eod2025.metropol.gov.co/macrozonas_actualizado.geo.json" }
)

foreach ($f in $fuentes) {
  $destino = Join-Path $raw $f.nombre
  if (Test-Path $destino) { Write-Host "Ya existe: $($f.nombre)"; continue }
  Write-Host "Descargando $($f.nombre) ..."
  try {
    Invoke-WebRequest -Uri $f.url -OutFile $destino -UseBasicParsing -TimeoutSec 600
    $mb = [math]::Round((Get-Item $destino).Length / 1MB, 1)
    Write-Host "  OK ($mb MB)"
  } catch {
    Write-Warning "  No se pudo descargar $($f.url): $($_.Exception.Message)"
  }
}
Write-Host "`nLa EOD 2025 se descarga por API al correr el pipeline; si la API no responde se usa data\reference\eod2025."
Write-Host "Siguiente paso: python run_pipeline.py"

### 7. API FastAPI

In [ ]:
%%writefile api/__init__.py
"""API de predicción de MobiPredict."""

#### `api/schemas.py`

In [ ]:
%%writefile api/schemas.py
"""Esquemas de entrada y salida de la API (Pydantic)."""
from __future__ import annotations

from datetime import datetime
from typing import Literal

from pydantic import BaseModel, Field

# Lecturas horarias reales del tramo AVENIDA_80_NS (MEData/SIMM). Se reemplazan al correr el pipeline
# por un ejemplo tomado del conjunto de prueba (ver models/ejemplo_api.json).
EJEMPLO_TRAMO = "AVENIDA_80_NS"
EJEMPLO_OBS = [
    {"timestamp": "2020-02-11T14:00:00", "velocidad_kmh": 24.0, "lluvia_mm": 0.0},
    {"timestamp": "2020-02-11T15:00:00", "velocidad_kmh": 22.0, "lluvia_mm": 0.0},
    {"timestamp": "2020-02-11T16:00:00", "velocidad_kmh": 19.0, "lluvia_mm": 0.4},
    {"timestamp": "2020-02-11T17:00:00", "velocidad_kmh": 15.0, "lluvia_mm": 1.2},
]


class Observacion(BaseModel):
    """Lectura horaria de un tramo vial."""
    timestamp: datetime = Field(..., description="Inicio de la hora (hora local de Medellín)")
    velocidad_kmh: float = Field(..., gt=0, le=130, description="Velocidad media del tramo en esa hora (km/h)")
    lluvia_mm: float = Field(0.0, ge=0, description="Precipitación en esa hora (mm)")


class PredictRequest(BaseModel):
    tramo_id: str = Field(..., examples=[EJEMPLO_TRAMO], description="Ver GET /tramos")
    observaciones: list[Observacion] = Field(
        ..., min_length=4, description="Últimas lecturas horarias consecutivas (>= 4, la más reciente al final)",
        examples=[EJEMPLO_OBS])


class PredictResponse(BaseModel):
    tramo_id: str
    corredor: str
    sentido: str
    timestamp_prediccion: datetime
    horizonte_min: int
    velocidad_predicha_kmh: float
    tiempo_viaje_predicho_min: float
    velocidad_referencia_kmh: float
    probabilidad_colapso: float
    alerta_colapso: bool
    nivel: Literal["normal", "precaucion", "colapso"]
    umbral_decision: float
    modelo_velocidad: str
    modelo_colapso: str


class PredictBatchRequest(BaseModel):
    peticiones: list[PredictRequest] = Field(..., min_length=1, max_length=500)


class PredictBatchResponse(BaseModel):
    predicciones: list[PredictResponse]
    alertas: int


class HealthResponse(BaseModel):
    status: Literal["ok", "degradado"]
    modelos_cargados: bool
    modelo_velocidad: str | None = None
    modelo_colapso: str | None = None
    horizonte_min: int | None = None
    entrenado_utc: str | None = None
    detalle: str | None = None

#### `api/main.py`

In [ ]:
%%writefile api/main.py
"""API de predicción de MobiPredict (FastAPI).

Endpoints:
  GET  /health        estado del servicio y de los modelos cargados
  GET  /tramos        catálogo de corredores disponibles
  POST /predict       velocidad a 1 hora + probabilidad y alerta de colapso para un tramo
  POST /predict/batch varias peticiones en una sola llamada

Arranque local:  uvicorn api.main:app --reload --port 8000
Documentación interactiva: http://localhost:8000/docs
"""
from __future__ import annotations

import json
import sys
import time
from contextlib import asynccontextmanager
from pathlib import Path

import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from mobipredict import config  # noqa: E402
from mobipredict.features import features_inferencia  # noqa: E402

from .schemas import (HealthResponse, PredictBatchRequest, PredictBatchResponse,  # noqa: E402
                      PredictRequest, PredictResponse)

ESTADO: dict = {"cargado": False}


def cargar_modelos() -> None:
    """Carga el modelo ganador de cada tarea, el feature store y la metadata."""
    faltan = [p for p in [config.MODEL_SPEED_FILE, config.MODEL_COLLAPSE_FILE,
                          config.FEATURE_STORE_FILE, config.METADATA_FILE] if not p.exists()]
    if faltan:
        ESTADO.update(cargado=False, error=f"Faltan artefactos: {[p.name for p in faltan]}. "
                                           "Ejecute `python run_pipeline.py` primero.")
        return
    ESTADO.update(
        cargado=True, error=None,
        modelo_velocidad=joblib.load(config.MODEL_SPEED_FILE),
        modelo_colapso=joblib.load(config.MODEL_COLLAPSE_FILE),
        store=joblib.load(config.FEATURE_STORE_FILE),
        meta=json.loads(config.METADATA_FILE.read_text(encoding="utf-8")),
        cargado_en=time.strftime("%Y-%m-%dT%H:%M:%S"),
    )


@asynccontextmanager
async def lifespan(_: FastAPI):
    cargar_modelos()
    yield


app = FastAPI(
    title="MobiPredict API",
    description="Pronóstico de velocidad por corredor y alertas tempranas de colapso en Medellín "
                "(modelos entrenados con datos abiertos del SIMM, la EOD 2025 y Open-Meteo).",
    version="1.0.0",
    lifespan=lifespan,
)


def _requiere_modelos() -> None:
    if not ESTADO.get("cargado"):
        raise HTTPException(status_code=503, detail=ESTADO.get("error", "Modelos no cargados"))


def _nivel(prob: float, umbral: float) -> str:
    if prob >= umbral:
        return "colapso"
    if prob >= umbral * 0.6:
        return "precaucion"
    return "normal"


def _predecir(req: PredictRequest) -> PredictResponse:
    store, meta = ESTADO["store"], ESTADO["meta"]
    obs = [o.model_dump() for o in sorted(req.observaciones, key=lambda o: o.timestamp)]
    try:
        X = features_inferencia(req.tramo_id, obs, store)
    except ValueError as exc:
        raise HTTPException(status_code=422, detail=str(exc)) from exc

    vel = max(float(ESTADO["modelo_velocidad"].predict(X)[0]), 1.0)
    prob = float(ESTADO["modelo_colapso"].predict_proba(X)[:, 1][0])
    umbral = float(meta["modelo_colapso"]["umbral_decision"])
    tramo = store["tramos"].set_index("tramo_id").loc[req.tramo_id]
    t_pred = pd.Timestamp(obs[-1]["timestamp"]).floor("h") + pd.Timedelta(minutes=meta["horizonte_min"])
    return PredictResponse(
        tramo_id=req.tramo_id,
        corredor=str(tramo["corredor"]),
        sentido=str(tramo["sentido"]),
        timestamp_prediccion=t_pred.to_pydatetime(),
        horizonte_min=meta["horizonte_min"],
        velocidad_predicha_kmh=round(vel, 2),
        tiempo_viaje_predicho_min=round(float(tramo["longitud_km"]) / vel * 60, 2),
        velocidad_referencia_kmh=round(float(tramo["vel_referencia_kmh"]), 1),
        probabilidad_colapso=round(prob, 4),
        alerta_colapso=prob >= umbral,
        nivel=_nivel(prob, umbral),
        umbral_decision=umbral,
        modelo_velocidad=meta["modelo_velocidad"]["nombre"],
        modelo_colapso=meta["modelo_colapso"]["nombre"],
    )


@app.get("/health", response_model=HealthResponse, tags=["estado"])
def health() -> HealthResponse:
    meta = ESTADO.get("meta", {})
    return HealthResponse(
        status="ok" if ESTADO.get("cargado") else "degradado",
        modelos_cargados=bool(ESTADO.get("cargado")),
        modelo_velocidad=meta.get("modelo_velocidad", {}).get("nombre"),
        modelo_colapso=meta.get("modelo_colapso", {}).get("nombre"),
        horizonte_min=meta.get("horizonte_min"),
        entrenado_utc=meta.get("generado_utc"),
        detalle=ESTADO.get("error"),
    )


@app.get("/tramos", tags=["catalogo"])
def tramos() -> list[dict]:
    _requiere_modelos()
    cols = ["tramo_id", "corredor", "sentido", "inicio", "fin", "longitud_km", "flujo_tipo", "macrozona",
            "vel_referencia_kmh"]
    t = ESTADO["store"]["tramos"][cols].copy()
    return t.astype(object).where(t.notna(), None).to_dict(orient="records")


@app.post("/predict", response_model=PredictResponse, tags=["prediccion"])
def predict(req: PredictRequest) -> PredictResponse:
    _requiere_modelos()
    return _predecir(req)


@app.post("/predict/batch", response_model=PredictBatchResponse, tags=["prediccion"])
def predict_batch(req: PredictBatchRequest) -> PredictBatchResponse:
    _requiere_modelos()
    preds = [_predecir(r) for r in req.peticiones]
    return PredictBatchResponse(predicciones=preds, alertas=sum(p.alerta_colapso for p in preds))


@app.post("/admin/reload", tags=["estado"])
def reload() -> dict:
    """Recarga los modelos desde disco (tras un reentrenamiento)."""
    cargar_modelos()
    return {"recargado": ESTADO.get("cargado"), "detalle": ESTADO.get("error")}

### 8. Pruebas

In [ ]:
%%writefile tests/conftest.py
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

#### Extracto real de MEData (Av. 80 y Autopista Sur, 10-11 feb 2020)

In [ ]:
%%writefile tests/fixtures/velocidad_gt_extracto.csv
ID,FECHA_HORA,TV_[S],CORREDOR,DÍA,MES,AÑO,HORA,FECHA,NOMBRE_CORREDOR,Flujo_tipo,SENTIDO,INICIO,FIN,LONGITUD_KM,TV_MINUTOS,VELOCIDAD_KM/H,TV_MIN/KM,DIASEM,NOMBRE_DÍA
1578647,10/02/2020 6:00,613,4,2,2,2020,6,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,10.2,38.2,1.6,1,Lunes
1578662,10/02/2020 6:00,832,7,2,2,2020,6,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,13.9,22.1,2.7,1,Lunes
1578746,10/02/2020 6:30,1135,4,2,2,2020,6,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,18.9,20.6,2.9,1,Lunes
1578760,10/02/2020 6:30,1197,7,2,2,2020,6,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,20,15.3,3.9,1,Lunes
1578847,10/02/2020 7:00,993,4,2,2,2020,7,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,16.6,23.6,2.5,1,Lunes
1578861,10/02/2020 7:00,1006,7,2,2,2020,7,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,16.8,18.3,3.3,1,Lunes
1578943,10/02/2020 7:30,1218,4,2,2,2020,7,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,20.3,19.2,3.1,1,Lunes
1578960,10/02/2020 7:30,901,7,2,2,2020,7,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,15,20.4,2.9,1,Lunes
1579043,10/02/2020 8:00,862,4,2,2,2020,8,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,14.4,27.1,2.2,1,Lunes
1579058,10/02/2020 8:00,806,7,2,2,2020,8,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,13.4,22.8,2.6,1,Lunes
1579147,10/02/2020 8:30,492,4,2,2,2020,8,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,8.2,47.6,1.3,1,Lunes
1579161,10/02/2020 8:30,759,7,2,2,2020,8,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,12.7,24.2,2.5,1,Lunes
1579246,10/02/2020 9:00,575,4,2,2,2020,9,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,9.6,40.7,1.5,1,Lunes
1579261,10/02/2020 9:00,961,7,2,2,2020,9,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,16,19.1,3.1,1,Lunes
1579343,10/02/2020 9:30,531,4,2,2,2020,9,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,8.9,44.1,1.4,1,Lunes
1579359,10/02/2020 9:30,819,7,2,2,2020,9,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,13.7,22.4,2.7,1,Lunes
1579445,10/02/2020 10:00,622,4,2,2,2020,10,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,10.4,37.6,1.6,1,Lunes
1579458,10/02/2020 10:00,837,7,2,2,2020,10,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,14,21.9,2.7,1,Lunes
1579544,10/02/2020 10:30,548,4,2,2,2020,10,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,9.1,42.7,1.4,1,Lunes
1579558,10/02/2020 10:30,798,7,2,2,2020,10,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,13.3,23,2.6,1,Lunes
1579647,10/02/2020 11:00,720,4,2,2,2020,11,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,12,32.5,1.8,1,Lunes
1579662,10/02/2020 11:00,838,7,2,2,2020,11,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,14,21.9,2.7,1,Lunes
1579747,10/02/2020 11:30,554,4,2,2,2020,11,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,9.2,42.2,1.4,1,Lunes
1579758,10/02/2020 11:30,804,7,2,2,2020,11,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,13.4,22.8,2.6,1,Lunes
1579843,10/02/2020 12:00,457,4,2,2,2020,12,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,7.6,51.2,1.2,1,Lunes
1579862,10/02/2020 12:00,1011,7,2,2,2020,12,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,16.9,18.2,3.3,1,Lunes
1579943,10/02/2020 12:30,477,4,2,2,2020,12,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,8,49.1,1.2,1,Lunes
1579958,10/02/2020 12:30,1017,7,2,2,2020,12,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,17,18.1,3.3,1,Lunes
1580047,10/02/2020 13:00,558,4,2,2,2020,13,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,9.3,41.9,1.4,1,Lunes
1580062,10/02/2020 13:00,906,7,2,2,2020,13,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,15.1,20.3,3,1,Lunes
1580146,10/02/2020 13:30,637,4,2,2,2020,13,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,10.6,36.7,1.6,1,Lunes
1580162,10/02/2020 13:30,856,7,2,2,2020,13,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,14.3,21.4,2.8,1,Lunes
1580244,10/02/2020 14:00,681,4,2,2,2020,14,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,11.4,34.4,1.7,1,Lunes
1580260,10/02/2020 14:00,912,7,2,2,2020,14,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,15.2,20.1,3,1,Lunes
1580347,10/02/2020 14:30,559,4,2,2,2020,14,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,9.3,41.9,1.4,1,Lunes
1580360,10/02/2020 14:30,939,7,2,2,2020,14,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,15.7,19.6,3.1,1,Lunes
1580446,10/02/2020 15:00,497,4,2,2,2020,15,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,8.3,47.1,1.3,1,Lunes
1580461,10/02/2020 15:00,1014,7,2,2,2020,15,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,16.9,18.1,3.3,1,Lunes
1580547,10/02/2020 15:30,461,4,2,2,2020,15,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,7.7,50.8,1.2,1,Lunes
1580559,10/02/2020 15:30,1002,7,2,2,2020,15,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,16.7,18.3,3.3,1,Lunes
1580647,10/02/2020 16:00,488,4,2,2,2020,16,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,8.1,48,1.3,1,Lunes
1580662,10/02/2020 16:00,929,7,2,2,2020,16,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,15.5,19.8,3,1,Lunes
1580743,10/02/2020 16:30,806,4,2,2,2020,16,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,13.4,29,2.1,1,Lunes
1580758,10/02/2020 16:30,1017,7,2,2,2020,16,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,17,18.1,3.3,1,Lunes
1580847,10/02/2020 17:00,709,4,2,2,2020,17,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,11.8,33,1.8,1,Lunes
1580861,10/02/2020 17:00,1091,7,2,2,2020,17,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,18.2,16.8,3.6,1,Lunes
1580943,10/02/2020 17:30,466,4,2,2,2020,17,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,7.8,50.2,1.2,1,Lunes
1580962,10/02/2020 17:30,1077,7,2,2,2020,17,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,18,17,3.5,1,Lunes
1581044,10/02/2020 18:00,402,4,2,2,2020,18,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,6.7,58.2,1,1,Lunes
1581059,10/02/2020 18:00,1025,7,2,2,2020,18,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,17.1,17.9,3.3,1,Lunes
1581147,10/02/2020 18:30,385,4,2,2,2020,18,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,6.4,60.8,1,1,Lunes
1581160,10/02/2020 18:30,827,7,2,2,2020,18,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,13.8,22.2,2.7,1,Lunes
1581246,10/02/2020 19:00,397,4,2,2,2020,19,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,6.6,58.9,1,1,Lunes
1581261,10/02/2020 19:00,830,7,2,2,2020,19,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,13.8,22.1,2.7,1,Lunes
1581347,10/02/2020 19:30,541,4,2,2,2020,19,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,9,43.3,1.4,1,Lunes
1581362,10/02/2020 19:30,992,7,2,2,2020,19,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,16.5,18.5,3.2,1,Lunes
1581443,10/02/2020 20:00,429,4,2,2,2020,20,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,7.2,54.5,1.1,1,Lunes
1581460,10/02/2020 20:00,863,7,2,2,2020,20,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,14.4,21.3,2.8,1,Lunes
1581546,10/02/2020 20:30,379,4,2,2,2020,20,10/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,6.3,61.7,1,1,Lunes
1581560,10/02/2020 20:30,709,7,2,2,2020,20,10/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,11.8,25.9,2.3,1,Lunes
1581647,11/02/2020 6:00,678,4,2,2,2020,6,11/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,11.3,34.5,1.7,2,Martes
1581662,11/02/2020 6:00,903,7,2,2,2020,6,11/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,15.1,20.3,3,2,Martes
1581746,11/02/2020 6:30,1267,4,2,2,2020,6,11/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,21.1,18.5,3.2,2,Martes
1581760,11/02/2020 6:30,1182,7,2,2,2020,6,11/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,19.7,15.5,3.9,2,Martes
1581847,11/02/2020 7:00,1063,4,2,2,2020,7,11/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,17.7,22,2.7,2,Martes
1581861,11/02/2020 7:00,1061,7,2,2,2020,7,11/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,17.7,17.3,3.5,2,Martes
1581944,11/02/2020 7:30,895,4,2,2,2020,7,11/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,14.9,26.1,2.3,2,Martes
1581959,11/02/2020 7:30,1013,7,2,2,2020,7,11/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,16.9,18.1,3.3,2,Martes
1582047,11/02/2020 8:00,587,4,2,2,2020,8,11/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,9.8,39.9,1.5,2,Martes
1582062,11/02/2020 8:00,865,7,2,2,2020,8,11/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,14.4,21.2,2.8,2,Martes
1582147,11/02/2020 8:30,481,4,2,2,2020,8,11/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,8,48.6,1.2,2,Martes
1582161,11/02/2020 8:30,827,7,2,2,2020,8,11/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,13.8,22.2,2.7,2,Martes
1582244,11/02/2020 9:00,1195,4,2,2,2020,9,11/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,19.9,19.6,3.1,2,Martes
1582261,11/02/2020 9:00,1106,7,2,2,2020,9,11/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,18.4,16.6,3.6,2,Martes
1582345,11/02/2020 9:30,855,4,2,2,2020,9,11/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,14.3,27.4,2.2,2,Martes
1582360,11/02/2020 9:30,917,7,2,2,2020,9,11/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,15.3,20,3,2,Martes
1582445,11/02/2020 10:00,672,4,2,2,2020,10,11/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,11.2,34.8,1.7,2,Martes
1582462,11/02/2020 10:00,882,7,2,2,2020,10,11/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,14.7,20.8,2.9,2,Martes
1582543,11/02/2020 10:30,574,4,2,2,2020,10,11/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,9.6,40.8,1.5,2,Martes
1582561,11/02/2020 10:30,932,7,2,2,2020,10,11/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,15.5,19.7,3,2,Martes
1582649,11/02/2020 11:00,520,4,2,2,2020,11,11/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,8.7,45,1.3,2,Martes
1582663,11/02/2020 11:00,828,7,2,2,2020,11,11/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,13.8,22.2,2.7,2,Martes
1582747,11/02/2020 11:30,490,4,2,2,2020,11,11/02/2020,Autopista Sur,Continuo,NS,CTI,Calle 10,6.5,8.2,47.8,1.3,2,Martes
1582760,11/02/2020 11:30,857,7,2,2,2020,11,11/02/2020,Avenida 80,Interrumpido,NS,Calle 65,Calle 30,5.1,14.3,21.4,2.8,2,Martes

#### `tests/test_features.py`

In [ ]:
%%writefile tests/test_features.py
"""Pruebas con un extracto REAL de MEData (Av. 80 y Autopista Sur, 10-11 feb 2020)."""
from pathlib import Path

import numpy as np
import pandas as pd
import pytest

from mobipredict import config, features, fuentes

FIX = Path(__file__).parent / "fixtures" / "velocidad_gt_extracto.csv"


@pytest.fixture(scope="module")
def datos():
    sens, tramos = fuentes.estandarizar_sensores(FIX, guardar=False)
    horas = pd.DataFrame({"hora": range(24)})
    for c in ["share_viajes_privado", "share_viajes_publico", "share_viajes_total"]:
        horas[c] = 1 / 24
    store = {
        "tramos": tramos.assign(vel_referencia_kmh=sens.groupby("tramo_id")["velocidad_kmh"].max()
                                .reindex(tramos["tramo_id"]).to_numpy()),
        "perfil": pd.DataFrame(columns=["tramo_id", "tipo_dia", "hora", "perfil_velocidad"]),
        "eod_hora": horas,
        "eod_mz": pd.DataFrame({"id_macrozona": [16, 21, 14], "eod_viajes_generados_mz": [1.0, 2.0, 3.0],
                                "eod_viajes_atraidos_mz": [1.0, 2.0, 3.0]}),
        "clima": pd.DataFrame({"timestamp": sens["timestamp"].unique(), "lluvia_mm": 0.0}),
        "festivos": set(),
    }
    return sens, tramos, store


def test_parser_fecha_y_agregacion_horaria(datos):
    sens, tramos, _ = datos
    assert set(tramos["tramo_id"]) == {"AVENIDA_80_NS", "AUTOPISTA_SUR_NS"}
    # Lecturas cada 30 min -> 1 fila por hora; 10-feb tiene 6:00..20:00 = 15 horas
    dia = sens[(sens["tramo_id"] == "AVENIDA_80_NS") & (sens["timestamp"].dt.day == 10)]
    assert len(dia) == 15
    # 6:00 (22.1) y 6:30 (15.3) se promedian
    assert dia.iloc[0]["velocidad_kmh"] == pytest.approx((22.1 + 15.3) / 2)
    assert (sens["timestamp"].dt.dayofweek.isin([0, 1])).all()   # lunes y martes


def test_rezagos_no_cruzan_la_noche(datos):
    sens, _, store = datos
    f = features.construir_features(sens, store)
    primera_hora = f[(f["tramo_id"] == "AVENIDA_80_NS") & (f["timestamp"] == "2020-02-11 06:00")]
    assert primera_hora["velocidad_kmh_lag1"].isna().all()
    otra = f[(f["tramo_id"] == "AVENIDA_80_NS") & (f["timestamp"] == "2020-02-10 09:00")].iloc[0]
    previa = sens[(sens["tramo_id"] == "AVENIDA_80_NS") & (sens["timestamp"] == "2020-02-10 08:00")].iloc[0]
    assert otra["velocidad_kmh_lag1"] == pytest.approx(previa["velocidad_kmh"])


def test_inferencia_igual_a_lote(datos):
    sens, _, store = datos
    lote = features.construir_features(sens, store)
    t = sens[sens["tramo_id"] == "AUTOPISTA_SUR_NS"].iloc[5:9]
    obs = [{"timestamp": r.timestamp, "velocidad_kmh": r.velocidad_kmh, "lluvia_mm": 0.0} for r in t.itertuples()]
    fila_api = features.features_inferencia("AUTOPISTA_SUR_NS", obs, store)
    fila_lote = lote[(lote["tramo_id"] == "AUTOPISTA_SUR_NS") & (lote["timestamp"] == t["timestamp"].iloc[-1])]
    a, b = fila_api[features.FEATURES_NUM].to_numpy(float), fila_lote[features.FEATURES_NUM].to_numpy(float)
    np.testing.assert_allclose(a, b, rtol=1e-6, equal_nan=True)


def test_historia_insuficiente(datos):
    sens, _, store = datos
    obs = [{"timestamp": "2020-02-10 06:00", "velocidad_kmh": 20.0}]
    with pytest.raises(ValueError):
        features.features_inferencia("AVENIDA_80_NS", obs, store)


def test_targets_sin_fuga(datos):
    sens, _, store = datos
    f = features.agregar_targets(features.construir_features(sens, store))
    ultima = f[(f["tramo_id"] == "AVENIDA_80_NS") & (f["timestamp"] == "2020-02-10 20:00")]
    assert ultima[features.TARGET_REG].isna().all()          # el objetivo sería al día siguiente
    r = f[(f["tramo_id"] == "AVENIDA_80_NS") & (f["timestamp"] == "2020-02-10 07:00")].iloc[0]
    s8 = sens[(sens["tramo_id"] == "AVENIDA_80_NS") & (sens["timestamp"] == "2020-02-10 08:00")].iloc[0]
    assert r[features.TARGET_REG] == pytest.approx(s8["velocidad_kmh"])
    assert config.HORIZON_STEPS == 1

#### `tests/test_api.py`

In [ ]:
%%writefile tests/test_api.py
"""Pruebas de la API. Requieren haber corrido el pipeline (models/ con artefactos)."""
import pytest
from fastapi.testclient import TestClient

from mobipredict import config

pytestmark = pytest.mark.skipif(not config.MODEL_SPEED_FILE.exists(), reason="ejecute run_pipeline.py primero")

from api.main import app  # noqa: E402
from api.schemas import EJEMPLO_OBS, EJEMPLO_TRAMO  # noqa: E402


@pytest.fixture(scope="module")
def client():
    with TestClient(app) as c:
        yield c


def test_health(client):
    r = client.get("/health")
    assert r.status_code == 200
    assert r.json()["modelos_cargados"] is True


def test_tramos(client):
    r = client.get("/tramos")
    assert r.status_code == 200
    assert any(t["tramo_id"] == EJEMPLO_TRAMO for t in r.json())


def test_predict(client):
    r = client.post("/predict", json={"tramo_id": EJEMPLO_TRAMO, "observaciones": EJEMPLO_OBS})
    assert r.status_code == 200, r.text
    b = r.json()
    assert 1 <= b["velocidad_predicha_kmh"] <= 130
    assert b["tiempo_viaje_predicho_min"] > 0
    assert 0 <= b["probabilidad_colapso"] <= 1


def test_predict_tramo_invalido(client):
    r = client.post("/predict", json={"tramo_id": "XX", "observaciones": EJEMPLO_OBS})
    assert r.status_code == 422


def test_predict_batch(client):
    ids = [t["tramo_id"] for t in client.get("/tramos").json()][:3]
    r = client.post("/predict/batch", json={"peticiones": [{"tramo_id": t, "observaciones": EJEMPLO_OBS} for t in ids]})
    assert r.status_code == 200
    assert len(r.json()["predicciones"]) == len(ids)

### 9. Contenedores
#### `Dockerfile`

In [ ]:
%%writefile Dockerfile
# Imagen única para el pipeline (Prefect + MLflow) y la API (FastAPI)
FROM python:3.12-slim

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1 \
    PIP_NO_CACHE_DIR=1 \
    PYTHONPATH=/app/src:/app \
    MOBIPREDICT_HOME=/app \
    MLFLOW_ALLOW_FILE_STORE=true

# libgomp1: requerido por LightGBM
RUN apt-get update && apt-get install -y --no-install-recommends libgomp1 curl \
    && rm -rf /var/lib/apt/lists/*

WORKDIR /app
COPY requirements.txt .
RUN pip install --upgrade pip && pip install -r requirements.txt

COPY src/ src/
COPY api/ api/
COPY tests/ tests/
COPY run_pipeline.py .
COPY data/reference/ data/reference/
RUN mkdir -p data/raw data/interim data/processed eda/figures eda/reports models docs mlruns \
    && useradd --create-home --uid 1000 mobi && chown -R mobi:mobi /app
USER mobi

EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=5s --start-period=20s --retries=3 \
    CMD curl -fsS http://localhost:8000/health || exit 1

CMD ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]

#### `docker-compose.yml`

In [ ]:
%%writefile docker-compose.yml
# Uso:
#   docker compose up --build            -> entrena (si no hay modelo) + API + MLflow UI
#   docker compose --profile orquestacion up   -> además, Prefect UI en :4200
#   RETRAIN=1 docker compose up trainer  -> fuerza un reentrenamiento
x-app: &app
  build: .
  image: mobipredict:latest
  volumes:
    - ./data:/app/data
    - ./models:/app/models
    - ./mlruns:/app/mlruns
    - ./eda:/app/eda
    - ./docs:/app/docs

services:
  trainer:
    <<: *app
    container_name: mobipredict-trainer
    environment:
      - RETRAIN=${RETRAIN:-0}
      - MOBIPREDICT_FAST=${MOBIPREDICT_FAST:-0}
      - MOBIPREDICT_EOD_URL=${MOBIPREDICT_EOD_URL:-}
      - MOBIPREDICT_SENSORS_URL=${MOBIPREDICT_SENSORS_URL:-}
      - PREFECT_API_URL=${PREFECT_API_URL:-}
    command: >
      sh -c "if [ -f models/metadata.json ] && [ \"$$RETRAIN\" != \"1\" ];
             then echo 'Modelo existente: se omite el entrenamiento (RETRAIN=1 para forzar)';
             else python run_pipeline.py; fi"
    restart: "no"

  api:
    <<: *app
    container_name: mobipredict-api
    ports:
      - "8000:8000"
    depends_on:
      trainer:
        condition: service_completed_successfully
    restart: unless-stopped

  mlflow:
    <<: *app
    container_name: mobipredict-mlflow
    ports:
      - "5000:5000"
    command: mlflow ui --backend-store-uri file:///app/mlruns --host 0.0.0.0 --port 5000
    healthcheck:
      test: ["CMD", "curl", "-fsS", "http://localhost:5000/health"]
      interval: 30s
      retries: 3
    restart: unless-stopped

  prefect:
    <<: *app
    container_name: mobipredict-prefect
    profiles: ["orquestacion"]
    ports:
      - "4200:4200"
    environment:
      - PREFECT_SERVER_API_HOST=0.0.0.0
      - PREFECT_UI_API_URL=http://localhost:4200/api
    command: prefect server start --host 0.0.0.0 --port 4200
    restart: unless-stopped

#### `.dockerignore`

In [ ]:
%%writefile .dockerignore
.git
.venv
__pycache__
*.pyc
.ipynb_checkpoints
.pytest_cache
mlruns
data/raw
data/interim
data/processed
models/*.joblib
models/candidatos
eda/figures
notebooks/*.ipynb

#### `requirements-dev.txt`

In [ ]:
%%writefile requirements-dev.txt
-r requirements.txt
pytest>=8
jupyter>=1.0
nbformat>=5.9
nbconvert>=7.16
ipykernel>=6.29

#### `Makefile`

In [ ]:
%%writefile Makefile
.PHONY: install datos pipeline fast api test notebook docker-up docker-down mlflow-ui
install:      ; pip install -r requirements-dev.txt
datos:        ; python scripts/descargar_datos.py
pipeline:     ; python run_pipeline.py
fast:         ; python run_pipeline.py --fast
api:          ; uvicorn api.main:app --reload --port 8000
test:         ; pytest -q tests
mlflow-ui:    ; MLFLOW_ALLOW_FILE_STORE=true mlflow ui --backend-store-uri ./mlruns --port 5000
notebook:     ; jupyter lab notebooks/MobiPredict_MLOps_end_to_end.ipynb
docker-up:    ; docker compose up --build -d
docker-down:  ; docker compose down

#### `.gitignore`

In [ ]:
%%writefile .gitignore
__pycache__/
*.pyc
.venv/
.ipynb_checkpoints/
.pytest_cache/
# Artefactos generados por el pipeline (se versionan con DVC o MLflow, no con Git)
data/raw/*
!data/raw/.gitkeep
data/interim/
data/processed/
mlruns/
models/*.joblib
models/candidatos/

### Documentación de fuentes

In [ ]:
%%writefile docs/fuentes_de_datos.md
# Fuentes de datos de MobiPredict

Todas las fuentes son abiertas y reales. No hay datos simulados en el pipeline. El inventario y la calidad se verificaron el 23 de octubre de 2026 leyendo los archivos completos.

## Resumen

| # | Fuente | Publica | Qué aporta al proyecto | Cobertura verificada | Formato y tamaño | Licencia |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | [Velocidad y tiempo de viaje GT](https://www.datos.gov.co/d/7t5n-3b3w) | Secretaría de Movilidad de Medellín (SIMM), vía MEData | **Serie del modelo:** velocidad y tiempo de viaje por tramo | 22 tramos (11 corredores × 2 sentidos), 6:00-20:00, 1-feb-2017 a 15-dic-2020, 682.502 filas, 1.045 días | CSV, 89,8 MB | Datos abiertos Alcaldía de Medellín |
| 2 | [EOD 2025 — visor oficial](https://eod2025.metropol.gov.co/) | Área Metropolitana del Valle de Aburrá | Demanda: 6.490.299 viajes en día hábil, viajes por hora y modo, reparto modal, motorización, **matriz OD de 67 macrozonas** | Valle de Aburrá, 10 municipios, día hábil 2025 | API JSON pública (`/api/api/indicadores/{id}/agregado`) + GeoJSON de macrozonas | Datos públicos del AMVA |
| 3 | [Road-traffic incidents and fatalities in Medellín 2008-2025](https://doi.org/10.17632/r6g5dfnpgh.1) | Correa Álvarez et al. (2026), Mendeley Data, a partir de registros de la Secretaría de Movilidad | Siniestralidad georreferenciada | 702.540 incidentes, 1-ene-2008 a 31-ago-2025 (2.762 fatales) | ZIP con XLSX, 63 MB | CC BY 4.0 |
| 4 | [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api) | Open-Meteo (reanálisis ERA5 / ERA5-Land de Copernicus) | Lluvia y temperatura horaria en Medellín | 2017-2020, horario | API JSON | CC BY 4.0 |
| 5 | [Aforos vehiculares](https://www.datos.gov.co/d/b9s9-jw7c) | Secretaría de Movilidad de Medellín, vía MEData | Composición vehicular (≈ 40 % motos) por hora | 74.900 registros de 15 min, 2017-2019, hasta 113 intersecciones | CSV, 19,7 MB | Datos abiertos Alcaldía de Medellín |
| 6 | Calendario de festivos de Colombia | Paquete [`holidays`](https://pypi.org/project/holidays/) (Ley 51 de 1983) | Festivos para el tipo de día | 2017-2026 | Calculado en código | MIT |
| 7 | Cifras publicadas de referencia | Observatorio de Movilidad / AMVA, vía [El Colombiano](https://www.elcolombiano.com/medellin/balance-de-accidentalidad-en-medellin-el-ano-pasado-PF32374139) | 276 muertes viales en Medellín en 2025 (162 motociclistas, 100 peatones); +48 % tenencia de motos y −14 % participación del TP entre 2017 y 2025 ([nota](https://www.elcolombiano.com/medellin/movilidad-habitantes-medellin-aburra-disminucion-teletrabajo-DD36063145)) | 2025 | Texto | — |

URLs de descarga directa usadas por `src/mobipredict/data_acquisition.py`:

- Velocidad GT: `https://medata.app.medellin.gov.co/sites/default/files/distribution/1-023-25-000287/Velocidad_y_tiempo_de_viaje_GT.csv`
- Aforos: `https://medata.app.medellin.gov.co/sites/default/files/distribution/1-023-25-000301/Aforos_Vehiculares.csv`
- Siniestros: `https://data.mendeley.com/public-files/datasets/r6g5dfnpgh/files/e992c888-5819-40c6-9538-41980ea1d429/file_downloaded`
- EOD 2025: `https://eod2025.metropol.gov.co/api/api/metadata/indicadores`, `.../indicadores/{1..46}/agregado`, `https://eod2025.metropol.gov.co/macrozonas_actualizado.geo.json`
- Clima: `https://archive-api.open-meteo.com/v1/archive?latitude=6.2442&longitude=-75.5812&hourly=precipitation,temperature_2m&timezone=America/Bogota`

## Por qué estas y no otras

| Fuente revisada | Decisión | Motivo |
| --- | --- | --- |
| Velocidad y tiempo de viaje GT | **Elegida** como serie del modelo | Es la serie abierta más larga por corredor (casi 4 años), con resolución horaria y los corredores principales. Permite un split temporal con validación y prueba reales, y un periodo de drift (pandemia). |
| Velocidad e intensidad vehicular (SIMM, CCTV/ARS) | Descartada | Solo cubre julio y agosto de 2020 (431.490 filas), en plena pandemia. No representa la operación normal. |
| Conteo vehicular Simmtrafficdata | Usada solo para ubicar corredores | 855 MB, pero solo enero de 2021 (cada 5 min). Sirvió para obtener la ubicación real de las cámaras por corredor (`data/reference/corredores_simm.csv`). |
| Incidentes viales MEData | Reemplazada | Llega a 2020. La base de Mendeley (misma fuente administrativa) llega a agosto de 2025. |
| Microdatos EOD (2017 o 2025) | No disponibles | El portal de datos abiertos del AMVA respondió 403 a las consultas. Se usan los agregados oficiales de la API del visor EOD 2025, que incluyen la matriz OD por macrozona. |
| SIATA (lluvia por estación) | Alternativa futura | No ofrece una API pública documentada de series históricas. Open-Meteo da lluvia horaria reproducible. Si se obtiene la serie del SIATA, reemplaza a Open-Meteo sin cambiar el modelo. |

## Problemas de calidad encontrados y cómo se tratan

| Fuente | Problema | Tratamiento |
| --- | --- | --- |
| Velocidad GT | Las columnas `DÍA` y `MES` están desalineadas en parte de 2020 (p. ej. `FECHA_HORA` = 1/02/2020 con `DÍA` = 2). | La fecha se toma de `FECHA_HORA`, que coincide con `DIASEM` en el 100 % de las filas. Las filas incoherentes se descartarían (hoy: 0). |
| Velocidad GT | La resolución cambia de horaria a cada 30 minutos en 2020. | Todo se agrega al promedio horario. |
| Velocidad GT | 9.699 claves (tramo, fecha-hora) duplicadas. | Se promedian. |
| Velocidad GT | El ID de corredor 13/14 se usa a veces para Guayabal y para El Poblado; `INICIO`/`FIN` y la longitud varían levemente. | El tramo se identifica por nombre del corredor + sentido; la longitud es la mediana. |
| Velocidad GT | Encabezados y nombres de día con caracteres dañados (`Mi�rcoles`). | Lectura UTF-8 con reemplazo y columnas por posición. |
| Velocidad GT | No hay lecturas entre 21:00 y 5:59. | Los rezagos no cruzan la noche; no se imputa. |
| Velocidad GT | Marzo-diciembre de 2020 es un régimen distinto (pandemia). | Fuera de entrenamiento, validación y prueba; se usa como prueba de estrés de drift. |
| Siniestros | La base cuenta incidentes; el cierre oficial de 2025 cuenta personas fallecidas, y la base termina en agosto de 2025. | Se grafican incidentes fatales por año y se cita aparte el cierre oficial (276). |
| EOD 2025 | La API identifica las macrozonas solo con números. | El diccionario id → municipio/macrozona se extrajo del visor oficial (`data/reference/eod2025_macrozonas.csv`). |

## Limitación principal

La serie de velocidad abierta más reciente termina en diciembre de 2020 y la EOD es de 2025. El modelo aprende la dinámica de la congestión con datos 2017-2020 y usa la EOD 2025 como prior de demanda por hora y por macrozona. Para operar en 2026 hay que reentrenar con el feed actual del SIMM: basta con poner un CSV con las mismas columnas en `data/raw/` y correr `python run_pipeline.py`.

### `README.md`

In [ ]:
%%writefile README.md
# MobiPredict — Movilidad urbana del Valle de Aburrá con datos reales

Flujo MLOps de extremo a extremo que pronostica la **velocidad de los corredores viales de Medellín a 1 hora**, emite **alertas tempranas de colapso** y caracteriza la **demanda y la densidad de viajes** con la **Encuesta Origen-Destino 2025 (AMVA)**. Todo con datos abiertos y reales; no hay datos simulados.

```
Fuentes oficiales ─► Estandarización ─► EDA ─► Features + split temporal ─► Entrenamiento + Optuna ─► MLflow ─► Selección ─► API
(data/raw)          (data/interim)     (eda/)   (data/processed)             (RandomizedSearchCV)       (mlruns/)  (models/, docs/) (FastAPI)
```

Orquestado con **Prefect** y ejecutable con un comando:

```bash
python run_pipeline.py
```

## Fuentes de datos

| Uso | Fuente | Cobertura |
| --- | --- | --- |
| Serie del modelo | [Velocidad y tiempo de viaje GT](https://www.datos.gov.co/d/7t5n-3b3w) — SIMM, Secretaría de Movilidad de Medellín (MEData) | 22 tramos de 11 corredores, cada hora 6:00-20:00, feb-2017 a dic-2020 (682.502 filas) |
| Demanda y densidad | [EOD 2025 — API del visor oficial](https://eod2025.metropol.gov.co/) (AMVA) | 6.490.299 viajes en día hábil, viajes por hora y modo, reparto modal, motorización, matriz OD de 67 macrozonas |
| Siniestralidad | [Mendeley Data doi:10.17632/r6g5dfnpgh.1](https://doi.org/10.17632/r6g5dfnpgh.1) (CC BY 4.0) | 702.540 incidentes, 2008 a ago-2025 |
| Lluvia | [Open-Meteo Historical Weather](https://open-meteo.com/en/docs/historical-weather-api) (ERA5) | Horaria 2017-2020 |
| Composición vehicular | [Aforos vehiculares](https://www.datos.gov.co/d/b9s9-jw7c) (MEData) | 2017-2019, ~40 % motos |
| Festivos | Paquete `holidays` (Colombia) | 2017-2026 |
| Cifras de referencia | [276 muertes viales en 2025](https://www.elcolombiano.com/medellin/balance-de-accidentalidad-en-medellin-el-ano-pasado-PF32374139); [+48 % motos, −14 % TP (2017-2025)](https://www.elcolombiano.com/medellin/movilidad-habitantes-medellin-aburra-disminucion-teletrabajo-DD36063145) | 2025 |

- Catálogo completo con URLs de descarga, licencias y tamaños: [`data/fuentes_datos.csv`](data/fuentes_datos.csv).
- Problemas de calidad encontrados y su tratamiento: [`docs/fuentes_de_datos.md`](docs/fuentes_de_datos.md).
- Ya incluidos en el repositorio: los agregados oficiales de la EOD 2025 (`data/reference/eod2025/`, incluida la matriz OD de 3.126 pares que suma 6.490.299 viajes), el diccionario y los centroides de las macrozonas, y la ubicación de las cámaras del SIMM.

## Inicio rápido

Requisitos: Python 3.11-3.13 y conexión a internet para la primera descarga (~175 MB).

```bash
python -m venv .venv
.venv\Scripts\activate             # Windows  (Linux/Mac: source .venv/bin/activate)
pip install -r requirements-dev.txt

# 1) Descargar las fuentes reales a data/raw/ (cualquiera de las dos)
python scripts/descargar_datos.py
powershell -ExecutionPolicy Bypass -File scripts\descargar_datos.ps1

# 2) Pipeline completo: estandarización, EDA, features, entrenamiento, MLflow, selección
python run_pipeline.py             # completo (~15-25 min)
python run_pipeline.py --fast      # menos trials de búsqueda (~5 min)

# 3) Servicio y seguimiento
uvicorn api.main:app --port 8000                                                    # http://localhost:8000/docs
set MLFLOW_ALLOW_FILE_STORE=true && mlflow ui --backend-store-uri ./mlruns --port 5000   # http://localhost:5000
```

`run_pipeline.py` descarga solo lo que falte en `data/raw/`. Si la API de la EOD no responde, usa la copia oficial de `data/reference/eod2025/`. Si falta una fuente obligatoria (velocidad, clima), se detiene con un mensaje claro: nunca inventa datos.

## Estructura

```
mobipredict/
├── run_pipeline.py                 # Punto de entrada (flujo Prefect)
├── scripts/
│   ├── descargar_datos.py          # Descarga todas las fuentes (multiplataforma)
│   ├── descargar_datos.ps1         # Igual, en PowerShell (Windows)
│   └── build_notebook.py           # Regenera el notebook desde el código
├── src/mobipredict/
│   ├── config.py                   # URLs oficiales, rutas, ventanas temporales, parámetros
│   ├── data_acquisition.py         # Descarga con reintentos + manifest.json (origen, tamaño, hash)
│   ├── fuentes.py                  # Lectura, validación y estandarización de cada fuente
│   ├── eda.py                      # 9 figuras y reportes con datos reales
│   ├── features.py                 # Features espaciotemporales + EOD, split temporal
│   ├── training.py                 # Candidatos, RandomizedSearchCV/Optuna, MLflow
│   ├── selection.py                # Reglas de selección, comparación, decisión
│   └── pipeline.py                 # @flow y @task de Prefect
├── api/                            # FastAPI: /health, /tramos, /predict, /predict/batch
├── tests/                          # Pruebas con un extracto real (fixtures/)
├── notebooks/MobiPredict_MLOps_end_to_end.ipynb
├── data/
│   ├── fuentes_datos.csv           # Catálogo de fuentes y enlaces
│   ├── reference/                  # Tablas oficiales pequeñas (versionadas)
│   ├── raw/                        # Descargas (no versionadas)
│   ├── interim/                    # Tablas estandarizadas
│   └── processed/                  # train / valid / test / drift .parquet
├── eda/figures/  eda/reports/
├── models/                         # Ganadores, feature store, model_comparison.csv
├── docs/                           # model_decision.md, fuentes_de_datos.md
├── mlruns/                         # Tracking local de MLflow
└── Dockerfile  docker-compose.yml  Makefile  requirements*.txt
```

## Componentes

### 1. Adquisición (`data_acquisition.py`, `fuentes.py`)

Cada fuente se toma de `data/raw/` si ya existe; si no, se descarga de la URL oficial con reintentos. El resultado queda en `data/raw/manifest.json` con origen, URL, tamaño y hash. `fuentes.py` valida y estandariza:

- **Velocidad:** la fecha sale de `FECHA_HORA`, porque las columnas `DÍA` y `MES` están corridas en parte de 2020. Las lecturas cada 30 minutos de 2020 y los 9.699 duplicados se promedian por hora. Los valores fuera de 2-110 km/h se descartan.
- **Tramos:** corredor + sentido. Cada tramo tiene longitud, tipo de flujo, ubicación (cámaras del SIMM) y macrozona EOD más cercana.
- **EOD 2025:** distribución horaria por grupo modal y matriz OD por macrozona.
- **Siniestros:** se lee el XLSX (hoja `BD`) y se guarda en parquet.

### 2. EDA (`eda.py`)

| Figura | Hallazgo |
| --- | --- |
| `01_congestion_cronica_heatmap` | Congestión por tramo y hora en días laborables, pre-pandemia |
| `02_eod2025_viajes_por_hora` | 6,49 M de viajes; pico de salida a las 6 h |
| `03_eod2025_reparto_modal` | A pie 32,7 %; la moto (14,6 %) supera al auto (11,7 %); 120,5 motos por 1.000 hab. |
| `04_eod2025_densidad_destinos` | La Candelaria y El Poblado atraen más viajes |
| `05_siniestralidad_vial` | Incidentes fatales por año 2008-2025 y por hora; cierre 2025: 276 muertes |
| `06_horas_en_colapso_por_tramo` | % de horas con velocidad < 50 % de la referencia |
| `07_drift_pandemia` | Caída de la congestión desde el 20-mar-2020 |
| `08_lluvia_vs_colapso` | Más colapso en hora pico con lluvia fuerte |
| `09_composicion_vehicular_aforos` | ~40 % de los vehículos aforados son motos |

### 3. Features y split (`features.py`)

- **Tiempo:** hora, día y mes cíclicos; fin de semana, pico, festivo.
- **Rezagos:** velocidad y razón vel/vel-referencia en t-1, t-2, t-3 h, media de 3 h y tendencia. Los rezagos no cruzan la noche.
- **Clima:** lluvia actual y de las últimas 3 h.
- **Espacio:** longitud, tipo de flujo, ubicación y velocidad de referencia (p95 del entrenamiento) del tramo.
- **Perfil histórico:** velocidad media del tramo por tipo de día y hora objetivo, calculada solo con entrenamiento.
- **EOD 2025:** participación horaria de viajes privados, públicos y totales en la hora objetivo, y viajes generados y atraídos por la macrozona del tramo.
- **Targets:** velocidad en t+1 h (regresión) y colapso en t+1 h = velocidad < 50 % de la referencia (clasificación).
- **Split temporal:** entrenamiento < 2019-09-01 ≤ validación < 2019-12-01 ≤ prueba < 2020-03-20 ≤ drift (pandemia) < 2021.

### 4-6. Entrenamiento, optimización y MLflow (`training.py`)

| Tarea | Línea base | Candidatos | Búsqueda |
| --- | --- | --- | --- |
| Velocidad a 1 h | Perfil histórico | Ridge, Random Forest, LightGBM, red neuronal MLP | RandomizedSearchCV / Optuna con `TimeSeriesSplit` |
| Colapso a 1 h | Regla de velocidad | Regresión logística, Random Forest, LightGBM | RandomizedSearchCV / Optuna (`average_precision`) |

Cada modelo es un run anidado en MLflow con parámetros, métricas `valid_*`, `test_*` y `drift_*` (MAE, RMSE, WAPE, R², precisión, recall, F1, PR-AUC, Brier, falsas alarmas por día), importancias y el modelo. Los ganadores se registran como `mobipredict-velocidad` y `mobipredict-colapso` con alias `champion`.

### 7. Selección (`selection.py`)

Reglas fijadas antes de ver resultados y aplicadas en validación:

- **Velocidad:** debe mejorar ≥ 10 % el WAPE del perfil histórico. Gana el menor RMSE; si otro queda a menos de 1 %, gana el más rápido.
- **Colapso:** solo modelos con recall ≥ 0,70. Gana la mayor PR-AUC; si empatan (< 0,005), el de menor Brier.

Salidas: `models/model_comparison.csv` y `docs/model_decision.md`. Este último incluye la prueba de estrés de la pandemia.

## API

| Método | Ruta | Descripción |
| --- | --- | --- |
| GET | `/health` | Estado y modelos cargados |
| GET | `/tramos` | Los 22 tramos con su `tramo_id` |
| POST | `/predict` | Velocidad y tiempo de viaje a 1 h, probabilidad y alerta de colapso |
| POST | `/predict/batch` | Hasta 500 peticiones |
| POST | `/admin/reload` | Recarga modelos tras reentrenar |

```bash
curl -X POST http://localhost:8000/predict -H "Content-Type: application/json" -d '{
  "tramo_id": "AVENIDA_80_NS",
  "observaciones": [
    {"timestamp": "2020-02-11T14:00:00", "velocidad_kmh": 24.0, "lluvia_mm": 0.0},
    {"timestamp": "2020-02-11T15:00:00", "velocidad_kmh": 22.0, "lluvia_mm": 0.0},
    {"timestamp": "2020-02-11T16:00:00", "velocidad_kmh": 19.0, "lluvia_mm": 0.4},
    {"timestamp": "2020-02-11T17:00:00", "velocidad_kmh": 15.0, "lluvia_mm": 1.2}
  ]}'
```

La respuesta trae `velocidad_predicha_kmh`, `tiempo_viaje_predicho_min`, `probabilidad_colapso`, `alerta_colapso` y `nivel` (`normal`, `precaucion` o `colapso`).

## Docker

```bash
python scripts/descargar_datos.py        # o el .ps1; los datos se montan como volumen
docker compose up --build                # trainer -> API (:8000) + MLflow UI (:5000)
docker compose --profile orquestacion up # + Prefect UI (:4200)
RETRAIN=1 docker compose up trainer      # reentrenar
```

## Pruebas

```bash
pytest -q tests
```

- `test_features.py` corre sobre un extracto real (Av. 80 y Autopista Sur, 10-11 feb 2020). Verifica el parser de fechas, la agregación horaria, que los rezagos no crucen la noche, que no haya fuga en los targets y que las features de la API sean idénticas a las de entrenamiento.
- `test_api.py` prueba los endpoints una vez entrenados los modelos.

## Limitaciones

- La serie abierta de velocidad termina en dic-2020; la EOD es de 2025. El modelo aprende la dinámica de congestión 2017-2020 y usa la EOD como prior de demanda. Para operar hoy hay que reentrenar con el feed actual del SIMM: basta un CSV con las mismas columnas en `data/raw/`.
- Las velocidades vienen de tiempos de viaje de Google Traffic agregados por la Alcaldía, no de espiras.
- No hay conteos de flujo vehicular abiertos y continuos; por eso el modelo predice velocidad y tiempo de viaje en lugar de flujo.

---
# Parte B — Datos y pipeline

## B.1 Descarga de las fuentes oficiales
Descarga a `data/raw/` lo que falte (~175 MB la primera vez) y escribe `manifest.json`.

In [ ]:
os.environ.setdefault("MOBIPREDICT_FAST", "1")   # "0" para la búsqueda completa de hiperparámetros
sys.path[:0] = [str(Path.cwd()), str(Path.cwd() / "src")]
import logging, warnings, json
import pandas as pd
logging.basicConfig(level=logging.WARNING)
warnings.filterwarnings("ignore")
from mobipredict import config
from mobipredict.data_acquisition import adquirir_datos

manifest = adquirir_datos()
pd.DataFrame(manifest["fuentes"]).T[["titulo", "origen", "bytes", "ficha"]]

## B.2 Pipeline orquestado con Prefect
Equivale a `python run_pipeline.py`.

In [ ]:
from mobipredict.pipeline import mobipredict_pipeline
resultado = mobipredict_pipeline(ejecutar_eda=True)

## B.3 Tablas estandarizadas

In [ ]:
sens = pd.read_parquet(config.SENSORES_STD)
print(f"{len(sens):,} lecturas horarias, {sens.tramo_id.nunique()} tramos, {sens.timestamp.min()} a {sens.timestamp.max()}")
pd.read_csv(config.TRAMOS_STD)[["tramo_id", "corredor", "sentido", "inicio", "fin", "longitud_km", "flujo_tipo", "macrozona"]]

In [ ]:
pd.read_csv(config.EOD_HORA_STD)[["hora", "viajes_publico", "viajes_privado", "viajes_no_motorizado", "viajes_informal", "viajes_total"]].round(0).head(24)

## B.4 EDA

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown((config.EDA_REPORTS / "eda_report.md").read_text(encoding="utf-8")))

In [ ]:
for ruta in sorted(config.EDA_FIGURES.glob("*.png")):
    display(Markdown(f"**{ruta.name}**"))
    display(Image(filename=str(ruta), width=900))

## B.5 Features y split temporal

In [ ]:
spec = json.loads((config.DATA_PROCESSED / "feature_spec.json").read_text(encoding="utf-8"))
print("Features:", len(spec["features"]))
print("Filas por partición:", spec["filas"])
print("Tasa de colapso:", spec["tasa_colapso"])
pd.read_parquet(config.DATA_PROCESSED / "train.parquet").head()

## B.6 Corridas en MLflow

In [ ]:
import mlflow
mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)
runs = mlflow.search_runs(experiment_names=[config.MLFLOW_EXPERIMENT], order_by=["start_time DESC"])
cols = ["tags.tarea", "tags.modelo", "metrics.valid_rmse", "metrics.valid_wape", "metrics.valid_pr_auc",
        "metrics.valid_recall_colapso", "metrics.test_rmse", "metrics.test_pr_auc", "tags.seleccionado"]
runs[runs["tags.tarea"].notna()][[c for c in cols if c in runs.columns]].head(12)

In [ ]:
for m in mlflow.MlflowClient().search_registered_models():
    print(m.name, dict(m.aliases or {}))

## B.7 Comparación y decisión

In [ ]:
comp = pd.read_csv(config.COMPARISON_FILE)
display(comp[comp.tarea == "velocidad"][["modelo", "seleccionado", "valid_rmse", "valid_wape", "test_mae", "test_rmse", "drift_mae", "latencia_ms_1000"]])
display(comp[comp.tarea == "colapso"][["modelo", "seleccionado", "valid_pr_auc", "valid_recall_colapso", "test_pr_auc", "test_recall_colapso", "test_precision_colapso", "umbral"]])

In [ ]:
display(Markdown(config.DECISION_FILE.read_text(encoding="utf-8")))

In [ ]:
import matplotlib.pyplot as plt, joblib
from mobipredict.features import FEATURES_NUM
modelo = joblib.load(config.MODEL_SPEED_FILE)
test = pd.read_parquet(config.DATA_PROCESSED / "test.parquet")
t = test[test.tramo_id == "AVENIDA_80_NS"].sort_values("timestamp")
t = t[t.timestamp.dt.normalize().isin(t.timestamp.dt.normalize().drop_duplicates().iloc[5:10])]
x = t["timestamp"] + pd.Timedelta(hours=1)
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(x, t["y_velocidad_1h"], "o-", color="#52514e", ms=3, lw=1.2, label="Observada")
ax.plot(x, modelo.predict(t[FEATURES_NUM]), "o-", color="#2a78d6", ms=3, lw=1.6, label="Predicha a 1 h")
ax.set_title("Avenida 80 (NS): velocidad observada vs. predicha, 5 días de prueba", loc="left")
ax.set_ylabel("km/h"); ax.legend(frameon=False); ax.grid(alpha=.3)
plt.show()

---
# Parte C — Servicio y despliegue

In [ ]:
from fastapi.testclient import TestClient
from api.main import app
from api.schemas import EJEMPLO_OBS, EJEMPLO_TRAMO
with TestClient(app) as cliente:
    print(cliente.get("/health").json())
    r = cliente.post("/predict", json={"tramo_id": EJEMPLO_TRAMO, "observaciones": EJEMPLO_OBS})
    display(pd.Series(r.json()).to_frame("respuesta"))

In [ ]:
!{sys.executable} -m pytest -q tests

## Despliegue

```bash
python scripts/descargar_datos.py && python run_pipeline.py
uvicorn api.main:app --port 8000
docker compose up --build                  # trainer -> API (:8000) + MLflow UI (:5000)
```

Fuentes, licencias y tratamiento de calidad: `docs/fuentes_de_datos.md` y `data/fuentes_datos.csv`.